# Column-wise L1 normalisation with coalesced threads

Normalise every **column** of a matrix to unit L1 norm:

$$y_{ij} = \frac{x_{ij}}{\max\!\left(\sum_{r} |x_{rj}|,\ \varepsilon\right)}, \qquad \varepsilon = 10^{-12}$$

The clamp keeps an all-zero column at zero instead of producing NaN.

## The mapping, and why it is the fast one
Use **one thread per column**: blocks of 64 threads, grid $\lceil \text{cols}/64 \rceil$, `j = cuda.grid(1)`. Each thread walks down its column twice (once to sum, once to write).

That sounds like a strided access pattern, but look at what a **warp** does at each loop step: 32 threads with consecutive `j` read `x[r, j]` for the same `r`, which are 32 consecutive floats in a row-major array. One 128-byte transaction serves the whole warp. The opposite mapping (one thread per row, looping over columns) would have the 32 threads of a warp read 32 addresses `cols` floats apart.

No shared memory and no synchronisation are needed: columns are independent.

## Inputs and output
- `x`: `float32`, shape `(rows, cols)`, both at least 1, row-major (C-contiguous).
- Return `y` as `float32` with the same shape. Accumulate the column sum in `float32`.
- Compared with relative tolerance $10^{-5}$, absolute $10^{-7}$.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import numpy as np
from numba import cuda

TPB = 64
EPS = 1e-12

def l1_normalize_columns(x):
    """x: float32 (rows, cols). Returns float32 (rows, cols) with unit-L1 columns."""
    rows, cols = x.shape
    d_x = cuda.to_device(np.ascontiguousarray(x))
    d_y = cuda.device_array((rows, cols), dtype=np.float32)
    blocks = (rows + TPB - 1) // TPB
    l1_kernel[blocks, TPB](d_x, d_y)
    return d_y.copy_to_host()

@cuda.jit
def l1_kernel(x, y):
    r = cuda.grid(1)                   # TODO: one thread per ROW is the slow mapping, and it
    if r < x.shape[0]:                 # normalises rows, not columns
        s = 0.0
        for j in range(x.shape[1]):
            s += x[r, j]               # TODO: absolute value, epsilon clamp
        for j in range(x.shape[1]):
            y[r, j] = x[r, j] / s


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, -2.0], [3.0, 2.0]], "dtype": "float32"}], "e": {"__nd__": [[0.25, -0.5], [0.75, 0.5]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, 2.0, 3.0]], "dtype": "float32"}], "e": {"__nd__": [[1.0, 1.0, 1.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.231665015220642, 0.2671189606189728, 0.0, 0.5015352964401245], [-1.326728343963623, 1.107768177986145, 0.0, -1.1708180904388428], [-1.3582403659820557, -1.3065639734268188, 0.0, 1.1856213808059692], [0.8945337533950806, -0.5440478324890137, 0.0, 1.929587721824646], [0.9027885794639587, -0.8033905625343323, 0.0, -0.4302915930747986]], "dtype": "float32"}], "e": {"__nd__": [[-0.21555380523204803, 0.0663008913397789, 0.0, 0.09611907601356506], [-0.23219084739685059, 0.27495619654655457, 0.0, -0.22438690066337585], [-0.2377057671546936, -0.32429876923561096, 0.0, 0.2272239476442337], [0.15655243396759033, -0.1350366622209549, 0.0, 0.36980485916137695], [0.15799710154533386, -0.1994074434041977, 0.0, -0.08246524631977081]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.5], [-0.5], [-0.5], [-0.5], [-0.5], [-0.5], [-0.5]], "dtype": "float32"}], "e": {"__nd__": [[-0.1428571492433548], [-0.1428571492433548], [-0.1428571492433548], [-0.1428571492433548], [-0.1428571492433548], [-0.1428571492433548], [-0.1428571492433548]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.15154145658016205, 0.879149854183197, -0.2748914659023285, 0.36928102374076843, 0.568265438079834, 0.7835076451301575, -0.6304340958595276, -0.221796452999115, 1.972353458404541, -0.1859680563211441, 0.029913093894720078, 0.4409482479095459, -0.5772605538368225, -0.38974758982658386, 1.6782280206680298, 1.296455979347229, -1.1034374237060547, 0.06516579538583755, -1.2894296646118164, -1.531747579574585, 0.002736472524702549, 1.5245542526245117, -0.08182613551616669, 1.470201015472412, -1.0568504333496094, 0.13626828789710999, 0.0809459462761879, 0.08619596809148788, 0.9934000968933105, 0.5733787417411804, -0.5490205883979797, -1.3512771129608154, -0.5538102984428406, -0.02058545872569084, -0.5506706833839417, -1.5250804424285889, -1.7636319398880005, 0.8112818002700806, 0.2564311623573303, -0.5855581164360046, 0.44081568717956543, 0.49441832304000854, -0.21350893378257751, -0.42714518308639526, 1.1488161087036133, 0.8532398343086243, 1.4325295686721802, 1.0122079849243164, 0.2584077715873718, -0.4491339921951294, -0.342641144990921, -0.5945685505867004, 0.06525478512048721, 0.2091742902994156, 1.1665877103805542, 1.3475943803787231, -0.018009493127465248, 0.38995879888534546, -1.1755001544952393, -0.6776876449584961, -1.0862568616867065, 0.12670397758483887, -0.5381069183349609, -0.7740989327430725, -0.02922406792640686, -1.8748778104782104, -0.09780754894018173, 0.4532788097858429, 1.23257577419281, -1.1779193878173828, -0.436354398727417, -2.0592474937438965, 1.3180829286575317, -0.21181365847587585, 0.21356536448001862, -1.539515495300293, -0.27373242378234863, -0.6360392570495605, -1.2428117990493774, -1.2089320421218872, -0.9871983528137207, -0.9083639979362488, -1.7987728118896484, 1.272772192955017, -0.0793139785528183, 0.060225922614336014, 0.044567592442035675, -0.3296782970428467, -0.18658225238323212, 0.4264450967311859, -0.3936433792114258, 1.1390490531921387, -0.3440118730068207, 1.020664930343628, -1.0355762243270874, 0.15507908165454865, 0.5550908446311951, 1.180607557296753, 1.2072468996047974, -1.0895689725875854, -0.7675547003746033, 1.4228184223175049, -0.4935930371284485, -1.621298909187317, 0.9784850478172302, 0.4722200632095337, -1.30815589427948, -0.7023630142211914, 1.1429699659347534, -0.5613574385643005, -0.6684896349906921, -0.526546835899353, 1.1640942096710205, -1.7585563659667969, 0.19440534710884094, -0.07624801248311996, 0.25920358300209045, 0.5132573246955872, -0.1721293181180954, 0.36764371395111084, 0.2298281192779541, 1.189131259918213, -2.4045605659484863, -0.3134860396385193, -0.07221069931983948, -1.4599545001983643, 1.2167854309082031, 0.3932887017726898, -1.7590194940567017, -0.35882997512817383], [0.5960685014724731, 0.025773322209715843, -0.7931777238845825, 0.5250595808029175, 0.2500934302806854, -0.9445063471794128, -0.26594141125679016, 1.2275904417037964, 1.119468092918396, -0.9631155729293823, -0.6444364190101624, 1.3431423902511597, 0.5524418354034424, 0.029327161610126495, -0.32119131088256836, 0.17749160528182983, 0.8866062164306641, -2.288200855255127, 0.829703152179718, -0.8995903730392456, -1.2850415706634521, -0.6508762240409851, -1.0716497898101807, -0.9911635518074036, 0.038889165967702866, -0.42375022172927856, 0.957984447479248, -1.5076820850372314, 0.7718095779418945, -1.1784183979034424, 0.1596258580684662, -0.32315003871917725, 0.9225702881813049, -1.2341771125793457, 0.08295951783657074, 1.5358880758285522, 1.2605599164962769, -0.2439141571521759, 0.770711362361908, 1.0872801542282104, 0.5351113677024841, 1.2129619121551514, 1.0185121297836304, -2.1618597507476807, 0.6773707866668701, 3.5597283840179443, -0.31077268719673157, -0.24353623390197754, 0.10438201576471329, -0.7267517447471619, -0.26457422971725464, -0.04359615966677666, 2.0314807891845703, -0.18332357704639435, 0.6167254447937012, -0.1390594244003296, -0.8664671778678894, -0.6007385849952698, -1.056006669998169, 1.4052516222000122, -0.19162526726722717, -1.38845694065094, 0.03138507530093193, 2.082955837249756, 0.3078670799732208, 0.008673423901200294, 0.32595303654670715, -0.34332075715065, -0.9822558760643005, 1.3391870260238647, -3.2547059059143066, -0.4892156422138214, -0.8556835055351257, -0.17551381886005402, -0.7015539407730103, 1.157409906387329, 1.5896323919296265, 0.18425005674362183, -1.2253994941711426, -0.7015854716300964, -0.32697615027427673, -0.8646931052207947, -1.2076977491378784, 1.7627272605895996, 0.9617324471473694, -0.9412595629692078, -0.851597249507904, 0.08525010943412781, -0.6428089737892151, 0.593061625957489, 0.8079827427864075, 0.06297189742326736, -0.9002277255058289, -0.29209840297698975, -1.6145751476287842, -0.3478091061115265, 1.8098320960998535, -0.5386336445808411, 0.6886866092681885, 1.2130156755447388, 1.6145192384719849, -1.3120112419128418, -0.1924862414598465, -0.5493267774581909, 1.3663653135299683, 1.0959259271621704, -1.0438508987426758, -0.5422149300575256, 0.366072416305542, 2.058976173400879, -0.35550379753112793, 1.2292213439941406, -0.8260408639907837, -1.2130968570709229, 0.8253031969070435, -0.7692468166351318, -1.5506521463394165, 1.2537946701049805, 0.6117683053016663, -1.2745925188064575, 0.857964277267456, 0.4337024390697479, 0.5588881373405457, -1.060407042503357, -0.41429799795150757, -2.1277596950531006, -2.129279851913452, 0.8037746548652649, -0.46875032782554626, 0.20520815253257751], [-0.2271338701248169, -0.530676007270813, -0.5164893269538879, 0.4858003556728363, -0.24624468386173248, -1.5440070629119873, 1.1161552667617798, 1.3851897716522217, 0.7613213062286377, 1.476744532585144, 0.18322238326072693, 0.022499946877360344, 1.1285054683685303, 0.5375382304191589, -1.1878360509872437, -0.1755090206861496, 0.08813906461000443, 2.184619903564453, 0.04455455765128136, -0.7684325575828552, -1.0834417343139648, -0.1652008444070816, -0.6478673815727234, -0.12506245076656342, -1.014116883277893, -1.6235766410827637, -0.2932146489620209, 0.33261555433273315, 0.1041238084435463, -1.536759376525879, -0.318141907453537, 0.7739880084991455, 0.2690930962562561, -0.656875491142273, -0.4777217507362366, -0.47695234417915344, 0.18829451501369476, -0.39333564043045044, 0.42789047956466675, 0.14740756154060364, 0.1303260326385498, 0.760811984539032, 0.8184022903442383, -0.41750234365463257, -0.31030595302581787, 0.8960642218589783, -0.3192208707332611, -1.2501672506332397, -0.5973032116889954, -1.0774143934249878, -0.05341658741235733, 0.07965216785669327, 1.3070718050003052, 0.401521235704422, 0.5813412666320801, 1.4593138694763184, 0.7279235124588013, -0.8917006850242615, -0.09770470857620239, -0.671313464641571, -0.21694064140319824, 1.1539034843444824, 2.8165431022644043, 0.7882928252220154, 0.22549296915531158, -0.7381506562232971, 0.24933147430419922, 1.527621865272522, -0.7919186353683472, 0.4065745174884796, 0.8057714104652405, 0.9353095889091492, 2.3535754680633545, 0.0011273984564468265, 0.2394411414861679, -1.0382267236709595, -0.4736107289791107, -0.9314520955085754, 0.08250977843999863, 0.4278387129306793, 0.6690530180931091, 0.7707390785217285, -0.511815071105957, -1.0085352659225464, -1.0098390579223633, -0.2984476089477539, -0.45233121514320374, -1.1882069110870361, 0.3851269483566284, -1.104894995689392, -0.1368810385465622, -0.6051692962646484, 0.32264477014541626, 0.47580310702323914, -0.9136974215507507, 0.40290042757987976, 1.9751986265182495, 0.44679248332977295, -1.071823239326477, 2.171954870223999, -0.9415614604949951, 0.7091088891029358, -1.0639488697052002, 0.7279537916183472, -0.15171284973621368, 0.5139342546463013, 0.22257617115974426, -1.4824752807617188, -1.5991263389587402, -0.8412506580352783, 0.8653973340988159, 0.311890184879303, 1.1964235305786133, 1.208795189857483, 1.694093942642212, -0.9437826871871948, -1.1534873247146606, -1.1008813381195068, 1.3384267091751099, -0.4076651334762573, 0.3021187484264374, 1.9075424671173096, -0.38002482056617737, -0.6118348836898804, -0.9361985325813293, 0.9660681486129761, 1.0271005630493164, 0.10220704227685928, -0.3633004128932953, -0.10827392339706421]], "dtype": "float32"}], "e": {"__nd__": [[0.15546798706054688, 0.6123923063278198, -0.17348143458366394, 0.26756760478019714, 0.5337812900543213, 0.23945678770542145, -0.31325438618659973, -0.07824676483869553, 0.5118817090988159, -0.07082262635231018, 0.03488115221261978, 0.24407757818698883, -0.25562775135040283, -0.40742453932762146, 0.5265433192253113, 0.7859897613525391, -0.5309626460075378, 0.014360067434608936, -0.595940887928009, -0.47870543599128723, 0.0011540359118953347, 0.6513431072235107, -0.045425064861774445, 0.5684293508529663, -0.5009109973907471, 0.06240547075867653, 0.060763612389564514, 0.044742412865161896, 0.5314193964004517, 0.17435574531555176, -0.5346969366073608, -0.5518986582756042, -0.3172836899757385, -0.01076849177479744, -0.4954962134361267, -0.4310669004917145, -0.5489928126335144, 0.5600718855857849, 0.1762373447418213, -0.3216917812824249, 0.39847633242607117, 0.20031596720218658, -0.10412919521331787, -0.14207355678081512, 0.5377112030982971, 0.1607147455215454, 0.6945520043373108, 0.40392807126045227, 0.26914867758750916, -0.19932274520397186, -0.5186566114425659, -0.8283011317253113, 0.01917111687362194, 0.26343733072280884, 0.4933438003063202, 0.4574369490146637, -0.011169369332492352, 0.20716063678264618, -0.5046772956848145, -0.2460513710975647, -0.7266793847084045, 0.0474713072180748, -0.15891948342323303, -0.21235257387161255, -0.05194612964987755, -0.7151376605033875, -0.14531080424785614, 0.19502393901348114, 0.40993621945381165, -0.4028891623020172, -0.0970359593629837, -0.5910969376564026, 0.2911383807659149, -0.5452722311019897, 0.18497547507286072, -0.4121693968772888, -0.1171310544013977, -0.3630896806716919, -0.48723945021629333, -0.5170007944107056, -0.4977736473083496, -0.35708996653556824, -0.5112640261650085, 0.3147282898426056, -0.0386730432510376, 0.04633001610636711, 0.033049847930669785, -0.20564594864845276, -0.15362656116485596, 0.20073658227920532, -0.2940913736820221, 0.6302872896194458, -0.21955154836177826, 0.5706608891487122, -0.29057803750038147, 0.17120890319347382, 0.12789753079414368, 0.5450550317764282, 0.4067876636981964, -0.24350415170192719, -0.23093830049037933, 0.4131369888782501, -0.28204861283302307, -0.5593425631523132, 0.39193281531333923, 0.2268020361661911, -0.5081039667129517, -0.2575538456439972, 0.3677310049533844, -0.1621677726507187, -0.35381224751472473, -0.2546585202217102, 0.36531388759613037, -0.42066213488578796, 0.07163578271865845, -0.04261385649442673, 0.08746998757123947, 0.17896418273448944, -0.08110414445400238, 0.17934700846672058, 0.16535453498363495, 0.3368285000324249, -0.7191803455352783, -0.1578695923089981, -0.05075583979487419, -0.3206025958061218, 0.27823901176452637, 0.30269965529441833, -0.678877592086792, -0.5337253212928772], [0.6115130186080933, 0.01795300655066967, -0.5005670189857483, 0.3804391026496887, 0.23491697013378143, -0.2886614501476288, -0.13214276731014252, 0.4330771565437317, 0.29053375124931335, -0.3667854368686676, -0.7514663934707642, 0.7434680461883545, 0.24463729560375214, 0.030657289549708366, -0.1007736399769783, 0.10760611295700073, 0.42662572860717773, -0.5042325854301453, 0.3834672272205353, -0.28114214539527893, -0.5419327616691589, -0.2780771851539612, -0.5949169993400574, -0.3832172751426697, 0.0184321366250515, -0.1940607875585556, 0.719129204750061, -0.7826042771339417, 0.4128795564174652, -0.3583391010761261, 0.15546129643917084, -0.13198333978652954, 0.5285501480102539, -0.645612359046936, 0.07464738190174103, 0.434121698141098, 0.3923938572406769, -0.16838718950748444, 0.5296865105628967, 0.5973260402679443, 0.48371511697769165, 0.4914373755455017, 0.49673259258270264, -0.7190601825714111, 0.31704801321029663, 0.6705041527748108, -0.15067598223686218, -0.0971846953034401, 0.10872073471546173, -0.3225277066230774, -0.400486558675766, -0.06073437258601189, 0.5968260169029236, -0.23088055849075317, 0.2608099579811096, -0.04720330983400345, -0.5373772382736206, -0.3191347122192383, -0.4533751904964447, 0.5102115869522095, -0.12819263339042664, -0.5202035903930664, 0.00926897581666708, 0.5714011788368225, 0.5472373962402344, 0.0033083180896937847, 0.4842621684074402, -0.14771431684494019, -0.32668358087539673, 0.4580482840538025, -0.7237775325775146, -0.14042696356773376, -0.18900351226329803, -0.45182546973228455, -0.6076372265815735, 0.30986952781677246, 0.680209219455719, 0.10518108308315277, -0.480413019657135, -0.3000336289405823, -0.1648707240819931, -0.33992233872413635, -0.3432631194591522, 0.4358832836151123, 0.4689352512359619, -0.7240830659866333, -0.6315162777900696, 0.053177110850811005, -0.5292707681655884, 0.27916643023490906, 0.6036447286605835, 0.03484519198536873, -0.5745336413383484, -0.16331426799297333, -0.45304253697395325, -0.38398486375808716, 0.41700029373168945, -0.24867278337478638, 0.23205627501010895, 0.27109283208847046, 0.4857690632343292, -0.38096243143081665, -0.10999036580324173, -0.1895158588886261, 0.5472984910011292, 0.5263609886169434, -0.4054446518421173, -0.19882814586162567, 0.11777753382921219, 0.5948074460029602, -0.18815790116786957, 0.5944992303848267, -0.25922662019729614, -0.290183424949646, 0.3041132390499115, -0.42992034554481506, -0.5232779383659363, 0.43717706203460693, 0.28825390338897705, -0.6217823028564453, 0.6172799468040466, 0.12284878641366959, 0.16715793311595917, -0.5340142846107483, -0.29120397567749023, -0.4672510623931885, -0.4868966341018677, 0.6186354160308838, -0.1809099316596985, 0.30522751808166504], [-0.23301905393600464, -0.36965471506118774, -0.32595157623291016, 0.35199329257011414, -0.23130176961421967, -0.4718817472457886, 0.5546028017997742, 0.4886760711669922, 0.19758449494838715, 0.5623918771743774, 0.21365252137184143, 0.01245437003672123, 0.4997349977493286, 0.5619181990623474, -0.3726830780506134, -0.106404148042202, 0.04241160303354263, 0.4814072847366333, 0.02059195749461651, -0.24015238881111145, -0.4569132626056671, -0.07057960331439972, -0.3596579134464264, -0.04835336655378342, -0.48065677285194397, -0.743533730506897, -0.22010715305805206, 0.17265334725379944, 0.055701032280921936, -0.46730512380599976, -0.30984175205230713, 0.31611794233322144, 0.15416623651981354, -0.343619167804718, -0.42985638976097107, -0.13481149077415466, 0.058613330125808716, -0.27154093980789185, 0.294076144695282, 0.08098223060369492, 0.1178085058927536, 0.3082466423511505, 0.3991382122039795, -0.13886623084545135, -0.14524081349372864, 0.1687810719013214, -0.15477201342582703, -0.49888724088668823, -0.6221305727958679, -0.47814950346946716, -0.0808568000793457, 0.11096446216106415, 0.3840028941631317, 0.5056820511817932, 0.24584616720676422, 0.49535977840423584, 0.45145338773727417, -0.47370460629463196, -0.04194754734635353, -0.24373707175254822, -0.1451280117034912, 0.4323250651359558, 0.8318115472793579, 0.21624627709388733, 0.4008164405822754, -0.2815539836883545, 0.3704269826412201, 0.6572617888450623, -0.2633802592754364, 0.1390625387430191, 0.17918646335601807, 0.26847606897354126, 0.5198581218719482, 0.002902263542637229, 0.2073872685432434, -0.27796101570129395, -0.20265968143939972, -0.5317292213439941, 0.032347630709409714, 0.18296559154987335, 0.33735567331314087, 0.3029877841472626, -0.14547285437583923, -0.24938836693763733, -0.4923917353153229, -0.2295868992805481, -0.33543384075164795, -0.7411769032478333, 0.31710267066955566, -0.5200970768928528, -0.10226396471261978, -0.3348674774169922, 0.20591486990451813, 0.2660248577594757, -0.2563793957233429, 0.4448062479496002, 0.4551021158695221, 0.2062721699476242, -0.36115601658821106, 0.4854029715061188, -0.28329265117645264, 0.20590056478977203, -0.6079609990119934, 0.2511415481567383, -0.06076867878437042, 0.24683690071105957, 0.08645135164260864, -0.5436180233955383, -0.5144914984703064, -0.24302475154399872, 0.4580298066139221, 0.15084221959114075, 0.3754594326019287, 0.28915444016456604, 0.6242510676383972, -0.5274658203125, -0.38925203680992126, -0.3838587701320648, 0.6306418776512146, -0.19887059926986694, 0.21736550331115723, 0.5403227210044861, -0.11366167664527893, -0.3081161677837372, -0.6580402255058289, 0.2121463119983673, 0.2348642796278, 0.07866495102643967, -0.14021249115467072, -0.16104711592197418]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.00038544912240467966, 0.0010662951972335577, -0.0009776582010090351, -0.0007572444155812263, 0.0008888245793059468, 0.000839727115817368, 4.6126839151838794e-05, -0.00013515265891328454, 0.0009932818356901407, 0.00013732867955695838, -0.0017003935063257813, -0.001048831851221621, 0.0007121216622181237, -0.00016205855354201049, 0.00041954717016778886, -0.0017103498103097081, 0.0011321625206619501, 0.0002474047360010445, -0.00019701113342307508, 7.815790013410151e-05], [-0.00032034990726970136, -0.00022788112983107567, -0.0022631483152508736, -0.0010202842531725764, 0.0010695221135392785, -0.00021817955712322146, -0.0008919280371628702, 0.0004798694862984121, 0.001478193560615182, -2.7863672585226595e-05, 0.0008187226485460997, -0.0006407896289601922, -0.0017858518986031413, 0.0015478484565392137, -0.0007351582171395421, 0.0007994324550963938, 0.0010412834817543626, 0.0016675444785505533, -0.000520941917784512, 0.0011061100522056222], [-0.000925087311770767, -0.0014235102571547031, -0.0016001737676560879, 0.00022704410366714, -0.0014539506519213319, 0.0026615073438733816, 0.0010955294128507376, 0.0010720418067649007, 0.00038225160096772015, 0.0009474648395553231, -0.00041369322570972145, -0.001025973237119615, -0.0008081739069893956, 0.000628204143140465, 0.0001049558341037482, 0.00042119520367123187, -0.0013187862932682037, 0.0008374298922717571, 0.0012858342379331589, 0.00041362849879078567], [-0.0022528348490595818, -0.0005347747937776148, -2.2466729205916636e-06, 0.001099983579479158, 0.00021350252791307867, 0.0008156073745340109, -0.0009464479517191648, -0.0006721612880937755, -0.0006722682737745345, 0.0013775789411738515, -0.0017446648562327027, -0.0004026945389341563, -0.0018205154919996858, 0.0016473369905725121, 0.0007517916383221745, -0.0007952132145874202, 0.0007332628592848778, 0.0007240670965984464, -0.00023329947725869715, 0.0003333240165375173], [-0.00019824635819531977, -0.00019442947814241052, -0.0017231530509889126, -0.0009345338912680745, 0.0007066067191772163, 0.00016122468514367938, 0.0021721897646784782, 0.001298998948186636, 0.0004529370635282248, -0.0019742639269679785, -0.0006613357691094279, 0.00017515631043352187, 0.001938994275406003, 0.0004481575160752982, -0.0014917689841240644, 0.000859200197737664, 0.0010423330822959542, 0.0012574796564877033, 0.0014672811375930905, -0.00088169751688838], [-3.152451972709969e-05, 0.0005040087271481752, 0.0009133619605563581, -0.0006061650929041207, 0.0009991356637328863, 0.0008631970849819481, -0.002267527859658003, 0.0006602867506444454, 0.00027918582782149315, -0.0006040013395249844, -0.0004314268007874489, 0.0004883516812697053, 0.0017087360611185431, 0.0005461517139337957, -0.0015430269995704293, -0.0004299015272408724, 4.754157271236181e-05, -0.0024676595348864794, 7.749616997898556e-06, -0.00070957001298666], [-0.00048303857329301536, -0.0012283158721402287, 0.000324979773722589, -0.0011249394156038761, -0.0011646444909274578, 4.355262717581354e-05, 0.0007784701883792877, -0.0006167070241644979, -0.00040865119080990553, 0.0007196590886451304, 0.0004092980525456369, 4.180088944849558e-05, 0.0010539514478296041, 0.000795623695012182, -0.001085258787497878, -0.0011409654980525374, 0.0017550159245729446, -0.0008392480085603893, -0.0019810153171420097, -0.0004424105864018202], [2.6890396839007735e-05, 0.0007859211764298379, 0.001842115307226777, -0.00012441020226106048, -0.002109099179506302, -0.0007442079368047416, 0.0005186458583921194, 0.0013943295925855637, -0.0003075172135140747, -0.0009455418330617249, -0.0004960661171935499, -0.0015161741757765412, -0.0011952643981203437, 0.0022466250229626894, 0.0016047972021624446, -0.0006717226351611316, -0.00129137362819165, 0.0007927885744720697, 0.0006519691087305546, 0.00046399622806347907], [-0.0019764231983572245, -0.0001417907333234325, -0.000901215011253953, 0.0010648075258359313, -0.0027041088324040174, 0.0007642419077455997, -0.0006752407643944025, 0.000883875007275492, -0.0005501190316863358, 0.001154792495071888, -0.0003567085659597069, -0.0006903970497660339, -0.0005640931194648147, 0.0014003274263814092, -0.0008451505564153194, 0.00027421017875894904, 0.0008283809875138104, -0.0011285777436569333, 4.3362473661545664e-05, -0.0002855355851352215], [0.0014970165211707354, -0.0006372201023623347, 0.00046239892253652215, -0.0011925798607990146, -0.0014928200980648398, 0.0005697584128938615, 0.0007604403072036803, 0.0006776551599614322, -0.0028079510666429996, 0.001564246485941112, 0.0004216381348669529, 0.00036218157038092613, 0.002229534788057208, 0.0008864696137607098, 0.00016614567721262574, -0.001062693540006876, -0.00013737936387769878, 0.0009266942506656051, -0.0020932473707944155, 0.0003417368861846626], [-0.0005215632845647633, -0.0007021513883955777, 0.0010146580170840025, -0.00016986510308925062, -0.0015502894530072808, -0.000918517995160073, 0.0004736415867228061, 0.000657165888696909, -0.0018590530380606651, -0.0009021993610076606, 0.0002999701246153563, 0.0013554394245147705, 0.0005923871649429202, -0.0006913319230079651, 0.0007222729036584496, -0.0009278251673094928, 0.00027428948669694364, -0.0001297154522035271, 0.001134292921051383, 0.0013320456491783261], [0.001536979922093451, 0.00024899275740608573, -8.509858162142336e-06, 0.00039168313378468156, 0.0023567327298223972, 0.00017732226115185767, 0.0003716335922945291, 7.113307947292924e-05, -0.0010175438364967704, 0.0009121126495301723, 0.0006972098490223289, 0.0001950215664692223, -0.0005165874608792365, -0.0003456100239418447, -0.00041852600406855345, -0.001537616248242557, -0.00045952078653499484, 0.0011177555425092578, 0.0005642425967380404, -0.0003093875420745462], [0.0007988415309228003, -0.00023783823417034, -0.0005913987988606095, 7.525926048401743e-05, 0.0003659441426862031, 0.0018815770745277405, 0.0001379199093207717, 7.610979082528502e-05, 0.0006220738869160414, -0.00130581168923527, 0.0013727638870477676, 0.001823701080866158, -0.00011412079766159877, -0.000507777149323374, 0.00020543437858577818, 0.00039393090992234647, -0.0006881392910145223, -0.0015867570182308555, -5.116407191962935e-05, -0.0006426338222809136], [0.001467217574827373, -0.00010704654414439574, 3.189164635841735e-05, 0.00014574298984371126, -0.0010593477636575699, -0.0008966315654106438, 0.001956093590706587, 0.0013950798893347383, -0.001172973308712244, 0.0003632503794506192, -0.00028739191475324333, 0.0008077475358732045, 0.000995971611700952, -0.00043393392115831375, 0.00100858008954674, -0.0004977903445251286, 0.00039326283149421215, 0.0016609418671578169, 0.0006387892062775791, 0.00023541749396827072], [0.001490214723162353, -0.000581040047109127, 0.0006496942951343954, -0.00021095204283483326, -0.0015696838963776827, -9.217890328727663e-05, 0.0004242988652549684, -0.002031669719144702, -0.001972249709069729, -0.00015196179447229952, 3.98850352212321e-06, -0.00014049508899915963, 0.0016914027510210872, 0.0031140686478465796, -0.00027312905876897275, 0.00028361566364765167, -0.0006187318940646946, 0.0011269625974819064, -0.002018984640017152, 0.0011500651016831398], [-0.0018375787185505033, 0.0003469646326266229, 0.00044830451952293515, -0.0010315192630514503, 0.00018614009604789317, 0.0015143280616030097, -6.298906373558566e-05, 0.00025813764659687877, -0.0011969043407589197, -0.00036851956974714994, -0.0016031523700803518, 0.0014479721430689096, -0.00025879847817122936, 0.0010410621762275696, 0.0009751162142492831, -0.0003735150385182351, 0.0006021662265993655, 0.0010871231788769364, 0.0013625839492306113, -0.002876873593777418], [0.0015308241127058864, 0.0014033501502126455, 0.00026044444530270994, -0.00013656560622621328, 0.00016203653649426997, 0.0008698691381141543, -0.0019424466881901026, 0.0018131565302610397, -0.001218638033606112, 0.002367348875850439, 0.00045873160706833005, 0.0011413197498768568, 0.0003470524388831109, -0.00044462503865361214, -0.001332118990831077, -0.0006492195534519851, 0.0009521651663817465, -0.0003815426316577941, -3.368789475644007e-05, -0.00011622872261796147], [-0.000818655185867101, -0.0005016334471292794, 0.0014846855774521828, 0.0014823263045400381, 0.0028692660853266716, -0.0008747361716814339, -0.0010838700691238046, 0.00017127787577919662, -0.0014571906067430973, 0.0003768921596929431, 0.0017143134027719498, 0.00041170261101797223, -0.00031131436116993427, -0.0018833663780242205, -1.0162707440031227e-05, -0.0010399810271337628, 0.00040556464227847755, 0.00021539119188673794, -0.002205661265179515, -0.0013821454485878348], [-0.0009761359542608261, 0.0002517661196179688, 0.0008243150659836829, -0.0012861854629591107, -0.0005875123315490782, 0.0004553321923594922, -0.0016391514800488949, -0.0009526584763079882, -0.0007815382559783757, -0.0007780823507346213, 0.0013701653806492686, 0.00019634549971669912, -0.0016314551467075944, 0.0013751969672739506, -0.00024183825007639825, -0.00018272655142936856, 0.0014370628632605076, 2.3326965674641542e-05, -0.000822571397293359, -0.0008448610897175968], [-0.00010614799248287454, 0.0008461068500764668, 0.000819120672531426, -0.0002983658923767507, -0.0012220406206324697, 0.0008140957215800881, -0.0015846764435991645, -0.00026151654310524464, 0.0007909596315585077, -0.001035944209434092, -0.00014325708616524935, -0.0009048147476278245, -0.0007758226129226387, -0.0005011066677980125, 0.0006099676247686148, -0.000690164917614311, -0.0006691868184134364, 0.001451505464501679, 0.0013813531259074807, 0.0010382506297901273], [-0.00010116433259099722, -1.335261549684219e-05, 0.00036510449717752635, 3.344990182085894e-05, 8.931208139983937e-05, 0.0010774459224194288, 0.002079377183690667, 0.00034604239044710994, -0.00021899958665017039, 0.0003645706456154585, -0.00014058631495572627, -0.0007117235218174756, 0.0002844373811967671, -0.0003660193469841033, 0.0012488140491768718, 0.000397191644879058, 0.0002915888908319175, -0.0005406466079875827, 0.001092670951038599, 0.001315454370342195], [0.0005563987069763243, -0.0005526260356418788, 0.000174562432221137, -0.0029287701472640038, 0.00035114551428705454, 0.0008440578822046518, -0.0013610762543976307, 0.000383690552553162, -0.0012179487384855747, -0.0004258391272742301, -0.0006177007453516126, 0.0015502794412896037, 0.0005902612465433776, -6.969393871258944e-05, 2.4796630896162242e-05, 0.0025384335312992334, 1.4505284525512252e-05, 0.00037405017064884305, -0.0024583570193499327, 0.001265642000362277], [0.00018522192840464413, -0.0006567533710040152, -0.00029814455774612725, 0.0005474624340422451, 0.0003527853114064783, -0.0009837615070864558, 0.00018555372662376612, -9.011890506371856e-05, -0.0005635640700347722, 0.0020088953897356987, 0.0005055131623521447, -0.001368221826851368, 0.00011499743413878605, 2.5768409614101984e-05, 0.0004613600031007081, -0.0006422135629691184, -0.0017612881492823362, -0.0010086853289976716, 0.001972951926290989, 0.0026529538445174694], [-0.0002166722115362063, -0.002493016654625535, -0.0009339372045360506, 0.0003697263018693775, 0.0012212746078148484, 0.0007183241541497409, 0.0011246406938880682, 0.0006824524607509375, 8.746112143853679e-05, -0.00022691983031108975, -0.0009064077748917043, -0.0018422040157020092, -0.0008748421678319573, -0.0003764322609640658, 0.0007931452710181475, -0.0017874966142699122, 0.0012409461196511984, -0.0015370962209999561, 0.0014895661734044552, -0.0008329045958817005], [0.00039297391776926816, -0.00023791003332007676, -3.0058776246733032e-05, -0.0010036963503807783, 0.0002123408339684829, -0.00041769156814552844, 0.000570098462048918, -5.764512934547383e-06, -0.00032386346720159054, -0.001606092555448413, -3.2736181765358197e-06, -0.002342384774237871, 2.3723847334622405e-05, -8.052635530475527e-05, 0.0008879179949872196, -0.0017249759985134006, 0.0008584567694924772, -0.00018975100829266012, 0.0005540470010600984, -0.0003548113163560629], [6.472477980423719e-05, 0.0014265013160184026, -0.0002484304131940007, 0.0005560517311096191, -0.00031469250097870827, 0.00011045279825339094, 0.0011361251818016171, -0.0002596243575681001, -5.810681977891363e-05, 0.0011714918073266745, 0.0004565693670883775, -0.00047658022958785295, -7.112853927537799e-05, -0.001134650083258748, -0.0006090396200306714, -0.0013549169525504112, 0.00012496112321969122, 0.000429160165367648, -0.0018611447885632515, 0.000996037619188428], [0.000363340659532696, -0.0006784910219721496, -0.0008277582237496972, -0.0010151444002985954, -0.0003357745008543134, -0.00014448343426920474, 0.0008389134891331196, 0.0023272885009646416, 0.00017106093582697213, 0.0006289788288995624, -0.0013115565525367856, -0.0001408788375556469, 0.0006943893386051059, 0.00045681133633479476, -0.0013894244330003858, 0.0011544200824573636, 0.0004108863649889827, -0.0014426297275349498, 0.0007223774446174502, -0.0010388792725279927], [-0.0014259401941671968, -0.0012186922831460834, -0.00024928234051913023, 0.001386787393130362, 0.0005781665677204728, -0.0005582983721978962, 0.0013840572210028768, 0.0003033266693819314, 0.0008935929508879781, -0.0015348929446190596, 0.0005032838671468198, -0.00029168845503591, -0.0013828729279339314, 0.0007125111878849566, 0.0001104181501432322, 0.00016970276192296296, -0.0005066312151029706, -0.0011586996261030436, 0.0011568298796191812, -5.537090328289196e-05], [0.0007141334936022758, 2.4534547264920548e-06, 0.00024094323453027755, -0.0006855491083115339, 0.0002748389670159668, -8.422525570495054e-05, -0.0005909891333431005, -0.00026646238984540105, 0.00022328847262542695, 0.0003945419448427856, -4.791422907146625e-05, -0.0010235278168693185, 0.0005070705083198845, -0.0005152928060851991, -0.002339655766263604, -0.001355924759991467, 0.00029727540095336735, -0.0005813483148813248, 0.000386648578569293, 0.0004409785324241966], [0.0017510168254375458, 0.0007277442491613328, -0.003678626846522093, -0.0008808908751234412, -0.0023676669225096703, -0.0002131805376848206, -0.00026059721130877733, 0.0007620887481607497, 0.0005705285002477467, 0.000937596894800663, -0.0005738582112826407, -0.0008151793153956532, 0.00021321623353287578, -0.00046943954657763243, -0.0006587417447008193, 0.0011880395468324423, -0.0003696366911754012, -0.0004196997033432126, 0.00035567046143114567, -0.00028625171398743987], [0.0009176706080324948, 0.0011463034898042679, -0.0007760439766570926, 0.0006340619875118136, 0.0007528896676376462, 0.0004295220132917166, -0.0006653929012827575, 0.0025114670861512423, 0.0003132061392534524, 0.0005981564754620194, -0.0007083203527145088, 0.001969163538888097, -0.000525561801623553, -0.001075591309927404, -0.001185402157716453, -0.00028871683753095567, -0.0015297885984182358, -0.00034251748002134264, -6.779598334105685e-05, 0.000924181891605258], [0.0010254798689857125, 0.0009519346058368683, 0.0010064857779070735, 0.002243260620161891, 0.00022369131329469383, 0.0012677491176873446, 0.0008074643556028605, 0.00044584457646124065, -0.0016725356690585613, -7.869034016039222e-05, 0.0015871347859501839, -0.001003938727080822, -0.0006272814935073256, -6.012296580593102e-05, 0.00044917859486304224, 0.00018188939429819584, -0.00015769677702337503, -0.001512046786956489, 0.0008136450196616352, -0.0004390683025121689], [-0.0005076896632090211, -0.000607841822784394, 0.0004774081753566861, -0.002572263590991497, -0.0015501517336815596, 0.0011304134968668222, 0.00046077562728896737, -0.0010729325003921986, 0.0018051443621516228, -0.0008340243366546929, -3.925912096747197e-05, 0.0007114179898053408, -0.0008880069362930954, 0.0005593824898824096, 0.0004481413052417338, -0.0008587943157181144, -0.0013226723531261086, 0.0014041138347238302, -1.311457253905246e-05, 0.0019235124345868826], [0.0007473193109035492, 0.00037041472387500107, 3.130422192043625e-05, -0.0008120393613353372, 0.0002521759015507996, -0.0006040739826858044, 0.0014464823761954904, 0.0011904345592483878, -0.0015775386709719896, -0.0009320761892013252, -0.000943669700063765, 0.0007890371489338577, 0.0004783955810125917, 0.0005184858455322683, -0.00025255349464714527, 0.0007381478208117187, -0.0015738509828224778, 0.0007496842881664634, 0.00022762297885492444, 0.0001238002150785178], [-0.0003883903264068067, -0.0028835770208388567, 0.0009883746970444918, -0.0010854771826416254, 0.0003639458736870438, -0.000891378556843847, 0.0005878074443899095, -9.703034447738901e-05, 0.0007692485814914107, -0.00023046693240758032, 0.0005488778697326779, 0.00033338923822157085, -0.0022977814078330994, 0.0011029649758711457, 0.0003291147877462208, -0.0001799800811568275, -0.0003072434919886291, 0.0003998008614871651, -0.0011350457789376378, -0.0003858165582641959], [-0.00023865324328653514, 0.0018775630742311478, -0.0010345974005758762, -0.001771936658769846, -0.0008781561627984047, -0.0006831550854258239, -0.0005561212892644107, -0.00025172129971906543, 0.000535534811206162, 0.0002992520749103278, 1.2719665392069146e-05, 0.0025257361121475697, 0.001140953623689711, 0.0008126669563353062, -1.608800084795803e-05, -0.0008221309399232268, 0.000326705863699317, -0.0019557445775717497, -0.000720480689778924, -0.0006336483638733625], [-0.0012413746444508433, 0.0019359003053978086, 0.001261999597772956, -0.00036656009615398943, -0.0011666930513456464, 0.0007919752388261259, 0.0001028409824357368, -0.00016567537386436015, 0.0007726411568000913, -0.0009371601045131683, 0.0002853349724318832, 0.00233944458886981, 0.0007572650210931897, 0.000516560219693929, 0.0019918340258300304, 0.0012027646880596876, 0.0017471397295594215, -0.0005009842570871115, -0.000618870253674686, 0.00012199399498058483], [-0.000729148683603853, -0.0011341623030602932, 0.0016800329321995378, -0.00030204252107068896, -0.0015617427416145802, 0.0002733859873842448, -2.3028893338050693e-05, -8.322176290675998e-05, 9.065694030141458e-05, 0.00039336777990683913, -0.0010547488927841187, -0.00026759281172417104, 0.0001729899668134749, 0.000930936133954674, 7.10339518263936e-05, -0.0011010606540367007, 0.0006769946776330471, -0.0011653824476525187, 2.2657448425889015e-05, 0.00048517942195758224], [-0.00023994572984520346, 0.0007034086156636477, -0.00021884623856749386, -0.0006406539469026029, -0.0002652207622304559, 0.0002194671396864578, 0.0005551007925532758, -0.0016715554520487785, 0.0003199265629518777, -0.0004306514747440815, 0.0010183814447373152, -0.001957604195922613, 0.000339966471074149, -0.0006770851323381066, -0.0005231836694292724, -0.0011764687951654196, 0.0011995552340522408, -0.0014764189254492521, 0.000690524815581739, -0.0015703572425991297], [0.0005275385919958353, -0.001524154213257134, -0.00029014769825153053, 0.001556950737722218, 0.0006799969705753028, -0.0017852011369541287, 4.937941753269115e-07, -0.0030587196815758944, 0.000264556088950485, -0.002208952559158206, 0.0004292198282200843, -0.000380926561774686, 0.0006650213035754859, -0.0007237395620904863, 0.0008382973028346896, 0.00027002347633242607, 0.0016901444178074598, -0.0010312723461538553, -0.00041093863546848297, -0.000987790059298277]], "dtype": "float32"}], "e": {"__nd__": [[0.012230338528752327, 0.0322008915245533, -0.03059430792927742, -0.021776648238301277, 0.02307204343378544, 0.02855823002755642, 0.0013344488106667995, -0.004283157642930746, 0.030217675492167473, 0.0040087210945785046, -0.06274693459272385, -0.027851631864905357, 0.021129755303263664, -0.005091127008199692, 0.014381405897438526, -0.05049068108201027, 0.03512152284383774, 0.006529612932354212, -0.0055549293756484985, 0.00245635025203228], [-0.010164733976125717, -0.006881748791784048, -0.07082173973321915, -0.029341083019971848, 0.027762576937675476, -0.0074200560338795185, -0.025803467258810997, 0.015207666903734207, 0.04496968910098076, -0.0008133602677844465, 0.030212027952075005, -0.017016109079122543, -0.05298899859189987, 0.048626210540533066, -0.025200048461556435, 0.023599786683917046, 0.03230230510234833, 0.04401055723428726, -0.014688488095998764, 0.03476288169622421], [-0.02935311198234558, -0.0429883748292923, -0.050074975937604904, 0.0065292781218886375, -0.03774154558777809, 0.09051504731178284, 0.031693652272224426, 0.03397434949874878, 0.011628879234194756, 0.027657167986035347, -0.015265868045389652, -0.027244623750448227, -0.023979773744940758, 0.01973525620996952, 0.003597718197852373, 0.012433966621756554, -0.04091089591383934, 0.022101812064647675, 0.036255404353141785, 0.012999536469578743], [-0.07148267328739166, -0.01614958606660366, -7.030617416603491e-05, 0.031633056700229645, 0.00554208317771554, 0.027737943455576897, -0.027380727231502533, -0.02130163460969925, -0.020451784133911133, 0.04021250084042549, -0.06438060849905014, -0.01069351565092802, -0.05401752144098282, 0.05175167694687843, 0.025770215317606926, -0.02347523160278797, 0.02274700626730919, 0.01910989172756672, -0.006578116212040186, 0.010475724004209042], [-0.006290376652032137, -0.005871546920388937, -0.05392342433333397, -0.026875095441937447, 0.018342046067118645, 0.0054830810986459255, 0.06284142285585403, 0.04116690903902054, 0.013779276981949806, -0.057630158960819244, -0.02440422959625721, 0.004651259630918503, 0.05753296986222267, 0.014079027809202671, -0.05113545432686806, 0.025364169850945473, 0.03233486786484718, 0.033187948167324066, 0.04137148708105087, -0.02771003358066082], [-0.001000276068225503, 0.015220485627651215, 0.02858225628733635, -0.017431946471333504, 0.025935491546988487, 0.029356420040130615, -0.06559955328702927, 0.020925316959619522, 0.00849340669810772, -0.017631225287914276, -0.015920260921120644, 0.012968133203685284, 0.050700850784778595, 0.017157550901174545, -0.052892498672008514, -0.012690983712673187, 0.0014748168177902699, -0.06512753665447235, 0.00021850834309589118, -0.02230040170252323], [-0.015326861292123795, -0.03709372878074646, 0.010169741697609425, -0.032350730150938034, -0.03023175708949566, 0.00148117879871279, 0.022521132603287697, -0.019544219598174095, -0.012432009913027287, 0.0210073571652174, 0.015103679150342941, 0.0011100186966359615, 0.03127237409353256, 0.024994801729917526, -0.0372009351849556, -0.033682070672512054, 0.054443445056676865, -0.022149797528982162, -0.055856745690107346, -0.013904101215302944], [0.00085323496023193, 0.023733917623758316, 0.05764616280794144, -0.0035777578596025705, -0.054747842252254486, -0.025309724733233452, 0.015004417859017849, 0.04418805614113808, -0.009355305694043636, -0.027601033449172974, -0.018305543810129166, -0.04026186466217041, -0.03546535223722458, 0.07057852298021317, 0.055009882897138596, -0.019829705357551575, -0.040060508996248245, 0.020923620089888573, 0.018382932990789413, 0.014582496136426926], [-0.06271210312843323, -0.004281917586922646, -0.028202136978507042, 0.030621472746133804, -0.07019305974245071, 0.025991057977080345, -0.019534707069396973, 0.028011107817292213, -0.016735751181840897, 0.03370920941233635, -0.013163052499294281, -0.018333429470658302, -0.016737518832087517, 0.04399178549647331, -0.02897040918469429, 0.008094869554042816, 0.025697724893689156, -0.029785912483930588, 0.001222649123519659, -0.00897382665425539], [0.04750048369169235, -0.01924331672489643, 0.014470062218606472, -0.03429591655731201, -0.03875051438808441, 0.019376881420612335, 0.021999528631567955, 0.021475743502378464, -0.08542364835739136, 0.04566146060824394, 0.015559046529233456, 0.009617697447538376, 0.06615375727415085, 0.027848757803440094, 0.005695207975804806, -0.031371429562568665, -0.004261731170117855, 0.024457715451717377, -0.0590212419629097, 0.010740123689174652], [-0.016549253836274147, -0.021204166114330292, 0.03175216168165207, -0.004884938709437847, -0.04024230316281319, -0.031237825751304626, 0.013702444732189178, 0.020826412364840508, -0.05655621737241745, -0.026335837319493294, 0.011069323867559433, 0.03599356859922409, 0.017577046528458595, -0.02171843871474266, 0.024758359417319298, -0.027390025556087494, 0.00850890576839447, -0.0034235063940286636, 0.03198254480957985, 0.04186359420418739], [0.04876852408051491, 0.007519295439124107, -0.00026630290085449815, 0.011263926513493061, 0.06117589771747589, 0.0060305423103272915, 0.010751355439424515, 0.0022542965598404408, -0.030955776572227478, 0.026625212281942368, 0.02572803385555744, 0.005178779363632202, -0.01532795187085867, -0.01085746195167303, -0.014346402138471603, -0.045391470193862915, -0.014255081303417683, 0.029500288888812065, 0.01590939424932003, -0.009723447263240814], [0.02534732036292553, -0.0071824416518211365, -0.018506914377212524, 0.0021642872598022223, 0.00949915125966072, 0.06399044394493103, 0.0039900210686028, 0.0024120148736983538, 0.018924767151474953, -0.03811756521463394, 0.050656937062740326, 0.048428211361169815, -0.0033861412666738033, -0.01595200039446354, 0.007041961885988712, 0.011629106476902962, -0.021347198635339737, -0.04187837988138199, -0.0014426230918616056, -0.02019672654569149], [0.04655496031045914, -0.0032326828222721815, 0.0009980000322684646, 0.004191240761429071, -0.02749847248196602, -0.030493490397930145, 0.0565897636115551, 0.04421183466911316, -0.035684261471033096, 0.010603534989058971, -0.010605169460177422, 0.021449660882353783, 0.029552023857831955, -0.01363218855112791, 0.03457251191139221, -0.014695107005536556, 0.012199652381241322, 0.04383629560470581, 0.018011312931776047, 0.007398712448775768], [0.04728465899825096, -0.017546743154525757, 0.020331183448433876, -0.00606650672852993, -0.04074574261903763, -0.003134906990453601, 0.012274960987269878, -0.06438616663217545, -0.05999989062547684, -0.004435872193425894, 0.00014718144666403532, -0.0037308339960873127, 0.05018654465675354, 0.09782957285642624, -0.009362427517771721, 0.008372525684535503, -0.019194068387150764, 0.029743285849690437, -0.05692732706665993, 0.03614430129528046], [-0.05830655246973038, 0.010477934032678604, 0.014028999954462051, -0.02966417744755745, 0.00483181094750762, 0.05150069296360016, -0.0018222727812826633, 0.008180705830454826, -0.03641229122877121, -0.010757346637547016, -0.059158600866794586, 0.03845076635479927, -0.007678952533751726, 0.032705336809158325, 0.03342542424798012, -0.011026415973901749, 0.01868017576634884, 0.02869182638823986, 0.038419440388679504, -0.09041452407836914], [0.04857319965958595, 0.04237956553697586, 0.008150207810103893, -0.003927320241928101, 0.004206132609397173, 0.02958332933485508, -0.056194957345724106, 0.0574612058699131, -0.03707347437739372, 0.06910458207130432, 0.016927847638726234, 0.03030763939023018, 0.010297584347426891, -0.01396805327385664, -0.045662909746170044, -0.01916540041565895, 0.029537711292505264, -0.010069838725030422, -0.0009498645085841417, -0.003652842016890645], [-0.025976009666919708, -0.015148754231631756, 0.04646094888448715, 0.04262837395071983, 0.07448019832372665, -0.02974885143339634, -0.03135634586215019, 0.005428011063486338, -0.04433073475956917, 0.011001748032867908, 0.06326059997081757, 0.010932724922895432, -0.009237180463969707, -0.05916662514209747, -0.000348361354554072, -0.030700942501425743, 0.012581274844706059, 0.005684697534888983, -0.062190864235162735, -0.0434381365776062], [-0.030972890555858612, 0.007603047881275415, 0.025795672088861465, -0.036987803876399994, -0.01525060273706913, 0.015485365875065327, -0.04742063209414482, -0.03019094280898571, -0.02377600036561489, -0.022712772712111473, 0.050561048090457916, 0.005213936325162649, -0.04840780794620514, 0.04320230334997177, -0.008289827965199947, -0.005394211038947105, 0.04458002746105194, 0.0006156553863547742, -0.0231932383030653, -0.026552336290478706], [-0.0033680861815810204, 0.025551455095410347, 0.02563312090933323, -0.00858033262193203, -0.03172164037823677, 0.027686534449458122, -0.0458446703851223, -0.008287787437438965, 0.024062618613243103, -0.03023994341492653, -0.005286389961838722, -0.024027271196246147, -0.023019861429929733, -0.01574244350194931, 0.020908715203404427, -0.020374134182929993, -0.020759262144565582, 0.0383087620139122, 0.0389486588537693, 0.03263019025325775], [-0.003209954360499978, -0.00040323365828953683, 0.011425383388996124, 0.0009619440534152091, 0.002318356651812792, 0.03664279729127884, 0.060156356543302536, 0.010966517962515354, -0.00666241766884923, 0.01064207497984171, -0.0051878346130251884, -0.01889975182712078, 0.008439699187874794, -0.011498628184199333, 0.04280734807252884, 0.011725366115570068, 0.009045561775565147, -0.01426897943019867, 0.03080897033214569, 0.04134216532111168], [0.01765458658337593, -0.016688672825694084, 0.005462662782520056, -0.08422484993934631, 0.009115009568631649, 0.02870551496744156, -0.039375919848680496, 0.012159635312855244, -0.03705250471830368, -0.012430544942617416, -0.022794034332036972, 0.04116752743721008, 0.01751396805047989, -0.002189459977671504, 0.0008499888353981078, 0.07493627071380615, 0.0004499774950090796, 0.009872093796730042, -0.06931588053703308, 0.03977666050195694], [0.0058771101757884026, -0.019833197817206383, -0.00932997465133667, 0.015743790194392204, 0.00915757566690445, -0.03345668688416481, 0.0053680669516325, -0.002855981234461069, -0.01714477688074112, 0.05864107608795166, 0.018654154613614082, -0.036333002150058746, 0.0034121524076908827, 0.0008095237426459789, 0.015814682468771935, -0.018958577886223793, -0.05463802069425583, -0.026621660217642784, 0.055629387497901917, 0.0833771601319313], [-0.006875030696392059, -0.07528623938560486, -0.02922612801194191, 0.010632498189806938, 0.03170175850391388, 0.02442944422364235, 0.03253583982586861, 0.021627774462103844, 0.0026607471518218517, -0.006623950321227312, -0.033447735011577606, -0.04891955479979515, -0.025957925245165825, -0.01182575337588787, 0.027187751606106758, -0.05276810750365257, 0.038496166467666626, -0.04056771099567413, 0.04199983552098274, -0.026176566258072853], [0.012469100765883923, -0.007184610236436129, -0.0009406431345269084, -0.028864052146673203, 0.005511927884072065, -0.014205248095095158, 0.016492940485477448, -0.0001826846564654261, -0.009852592833340168, -0.04688297584652901, -0.00012080115993740037, -0.062201809138059616, 0.0007039234042167664, -0.0025297640822827816, 0.030436407774686813, -0.0509224571287632, 0.026630723848938942, -0.005007990635931492, 0.015621919184923172, -0.011151027865707874], [0.00205372367054224, 0.0430787019431591, -0.007774247322231531, 0.01599079929292202, -0.008168765343725681, 0.0037563827354460955, 0.03286808729171753, -0.008227821439504623, -0.0017677289433777332, 0.034196674823760986, 0.016848057508468628, -0.01265554316341877, -0.002110494300723076, -0.035645436495542526, -0.02087690308690071, -0.03999806568026543, 0.003876497270539403, 0.011326581239700317, -0.05247687175869942, 0.031303517520427704], [0.011528834700584412, -0.02048964984714985, -0.02590341866016388, -0.029193272814154625, -0.008716010488569736, -0.004913728684186935, 0.02426975592970848, 0.0737546980381012, 0.005204025190323591, 0.018360335379838943, -0.048398301005363464, -0.0037410242948681116, 0.020603610202670097, 0.014350890181958675, -0.047627247869968414, 0.034079261124134064, 0.012746362946927547, -0.03807450830936432, 0.020368168130517006, -0.03264994919300079], [-0.0452452227473259, -0.03680310770869255, -0.007800907362252474, 0.039880890399217606, 0.015008006244897842, -0.018987135961651802, 0.0400407575070858, 0.009612803347408772, 0.02718493528664112, -0.044804610311985016, 0.018571889027953148, -0.007745759561657906, -0.041031986474990845, 0.0223837923258543, 0.0037849575746804476, 0.005009740125387907, -0.015716522932052612, -0.030580902472138405, 0.032617997378110886, -0.001740199513733387], [0.022659527137875557, 7.409152021864429e-05, 0.007539947517216206, -0.019714852795004845, 0.007134249899536371, -0.002864411799237132, -0.01709730736911297, -0.008444528095424175, 0.006792894564568996, 0.011516958475112915, -0.001768103102222085, -0.027179686352610588, 0.015045569278299809, -0.016188107430934906, -0.08019965887069702, -0.040027815848588943, 0.009221966378390789, -0.01534319669008255, 0.010901950299739838, 0.013859095051884651], [0.05555993691086769, 0.02197704091668129, -0.11511696875095367, -0.025332443416118622, -0.061459727585315704, -0.007250044494867325, -0.007539073936641216, 0.024151548743247986, 0.01735665090382099, 0.027369115501642227, -0.021176183596253395, -0.021647009998559952, 0.006326456554234028, -0.014747610315680504, -0.02258061245083809, 0.035071730613708496, -0.011466731317341328, -0.011076896451413631, 0.010028490796685219, -0.008996332995593548], [0.029117779806256294, 0.03461705148220062, -0.024285102263092995, 0.018234198912978172, 0.01954345591366291, 0.01460758876055479, -0.0192498080432415, 0.0795915424823761, 0.00952837523072958, 0.01746061071753502, -0.026138028129935265, 0.05229095369577408, -0.015594234690070152, -0.03379008173942566, -0.04063368961215019, -0.008523115888237953, -0.0474565289914608, -0.009039869531989098, -0.0019115768373012543, 0.029045233502984047], [0.032538577914237976, 0.028747333213686943, 0.03149642422795296, 0.06451113522052765, 0.005806562956422567, 0.043114807456731796, 0.023359933868050575, 0.01412937417626381, -0.05088197439908981, -0.0022970265708863735, 0.05856752768158913, -0.02665949985384941, -0.018612416461110115, -0.001888784347102046, 0.015397123992443085, 0.005369497928768396, -0.0048920102417469025, -0.03990659490227699, 0.022941550239920616, -0.013799060136079788], [-0.0161090437322855, -0.01835612580180168, 0.014939754270017147, -0.07397252321243286, -0.04023872688412666, 0.038444165140390396, 0.013330233283340931, -0.034002579748630524, 0.054916203022003174, -0.0243457593023777, -0.0014487173175439239, 0.01889163814485073, -0.026348544284701347, 0.017573200166225433, 0.01536156702786684, -0.02535218931734562, -0.04103144630789757, 0.03705798089504242, -0.00036977871786803007, 0.06045224145054817], [0.023712515830993652, 0.011186099611222744, 0.0009796174708753824, -0.023352427408099174, 0.0065459636971354485, -0.020543916150927544, 0.04184671491384506, 0.03772636502981186, -0.04799196869134903, -0.027207963168621063, -0.034822750836610794, 0.020952807739377022, 0.014194739982485771, 0.01628841646015644, -0.008657129481434822, 0.021790621802210808, -0.048823416233062744, 0.019785992801189423, 0.006418061442673206, 0.0038907991256564856], [-0.01232366356998682, -0.08708071708679199, 0.03092966414988041, -0.031215885654091835, 0.009447280317544937, -0.030314842239022255, 0.017005261033773422, -0.003075013402849436, 0.023402122780680656, -0.0067274924367666245, 0.02025437355041504, 0.008853119798004627, -0.06817874312400818, 0.034650035202503204, 0.011281528510153294, -0.00531313382089138, -0.009531192481517792, 0.01055171713232994, -0.032003771513700485, -0.012125461362302303], [-0.007572491187602282, 0.05670025199651718, -0.032376132905483246, -0.05095691606402397, -0.022795114666223526, -0.023233382031321526, -0.01608858071267605, -0.007977363653481007, 0.016292070969939232, 0.008735379204154015, 0.0004693737137131393, 0.06707068532705307, 0.033853866159915924, 0.025530220940709114, -0.000551470962818712, -0.024269860237836838, 0.010134946554899216, -0.0516168586909771, -0.020314686000347137, -0.019914332777261734], [-0.03938893973827362, 0.05846196785569191, 0.03949233517050743, -0.010541444644331932, -0.030284933745861053, 0.026934240013360977, 0.002975188195705414, -0.005250460468232632, 0.023505331948399544, -0.02735636569559574, 0.010529265739023685, 0.06212373450398445, 0.022469228133559227, 0.016227923333644867, 0.06827688217163086, 0.03550642728805542, 0.05419911444187164, -0.013222193345427513, -0.017449676990509033, 0.003834033152088523], [-0.0231359601020813, -0.034250400960445404, 0.052574045956134796, -0.00868606474250555, -0.04053960368037224, 0.00929756835103035, -0.0006662255618721247, -0.002637402154505253, 0.0027579704765230417, 0.011482683010399342, -0.03892173245549202, -0.007105902303010225, 0.0051328809931874275, 0.029245689511299133, 0.0024349302984774113, -0.03250405564904213, 0.021001474931836128, -0.030757278203964233, 0.00063884980045259, 0.015248242765665054], [-0.007613501511514187, 0.021242132410407066, -0.006848456338047981, -0.018423767760396004, -0.006884581409394741, 0.007463844958692789, 0.016059057787060738, -0.05297369137406349, 0.009732823818922043, -0.012571020983159542, 0.037579722702503204, -0.051983997225761414, 0.010087332688272, -0.02127087116241455, -0.01793389953672886, -0.034730155020952225, 0.037212152034044266, -0.038966286927461624, 0.019470050930976868, -0.04935326427221298], [0.016738852486014366, -0.04602770879864693, -0.009079725481569767, 0.04477440565824509, 0.017651312053203583, -0.06071280315518379, 1.4285458746599033e-05, -0.09693466871976852, 0.008048339746892452, -0.06448088586330414, 0.01583882048726082, -0.010115469805896282, 0.019732214510440826, -0.02273653633892536, 0.028735490515828133, 0.007971275597810745, 0.052431028336286545, -0.027217786759138107, -0.01158683281391859, -0.031044313684105873]], "dtype": "float32"}}, {"a": [{"__nd__": [[-38.9168586730957, 86.3712387084961, -15.654115676879883, -60.58910369873047, -32.82209014892578, -27.912809371948242, 1.158125877380371, 155.09910583496094, -17.179521560668945, 207.97821044921875, 62.32550811767578, 53.78130340576172, -117.47901153564453, -71.64985656738281, -52.82799530029297, 90.0980224609375, -135.24484252929688, -66.59271240234375, -100.45321655273438, 106.74092864990234, -164.88455200195312, 55.277061462402344, 41.737022399902344, 24.083559036254883, 158.57632446289062, 56.48040008544922, -129.00363159179688, 74.8794937133789, -49.89250564575195, 69.43267822265625, -59.70335388183594, -33.15180587768555, -152.2754669189453, -43.73465347290039, -148.16152954101562, 13.066981315612793, 20.353158950805664, 185.8614959716797, -110.61947631835938, 62.368896484375, -2.228276252746582, -81.71308898925781, 215.8707275390625, -168.22744750976562, -58.748046875, 88.93799591064453, 20.227014541625977, 24.231403350830078, 73.1728515625, 110.22348022460938, -206.99984741210938, 34.96367263793945, 2.15710711479187, 83.11221313476562, 164.47718811035156, 34.32160186767578, -9.437556266784668, 55.49440002441406, -149.4210662841797, 95.9420166015625, 55.948551177978516, -93.63273620605469, -69.78177642822266, -29.958911895751953], [75.35620880126953, 226.9550018310547, 273.16748046875, -90.07960510253906, -57.34132766723633, -37.05240249633789, 117.51022338867188, 12.452229499816895, -169.9034881591797, 17.27903175354004, -150.482421875, 81.49488067626953, -0.9369256496429443, 118.55503845214844, 45.175025939941406, -75.59455871582031, -115.8366470336914, -139.4015350341797, -30.049720764160156, 197.4750213623047, 175.97584533691406, -95.07054901123047, -112.60318756103516, 178.24366760253906, -82.49141693115234, 8.079802513122559, 8.083513259887695, -51.629188537597656, 101.18002319335938, 88.34793090820312, 33.63484573364258, -151.4085693359375, -145.30618286132812, 32.67428970336914, -54.95915603637695, 78.31010437011719, -52.62419509887695, -103.29756164550781, -83.3614273071289, 306.40484619140625, -60.69792175292969, -53.81540298461914, -98.32893371582031, -21.547077178955078, 18.227163314819336, 165.45838928222656, 107.20416259765625, -50.180294036865234, 0.5312792062759399, 15.746057510375977, 17.753875732421875, 110.78082275390625, 133.63253784179688, 153.03005981445312, 89.16654205322266, 20.11812400817871, 18.037626266479492, -19.22521209716797, -1.286738395690918, -14.341354370117188, 18.4296932220459, -30.72365379333496, 67.97905731201172, -194.88682556152344], [81.09579467773438, 127.42786407470703, 37.42122268676758, -67.9011459350586, -171.4929656982422, -16.503215789794922, 119.2623519897461, 9.681819915771484, -62.475563049316406, 80.55387115478516, 31.163679122924805, -68.51647186279297, -79.8347396850586, 89.8359603881836, 127.69237518310547, 22.924718856811523, 141.6546630859375, 17.483325958251953, 72.6003189086914, -123.14764404296875, -57.418365478515625, -156.4994659423828, -56.689613342285156, -118.52913665771484, 167.70559692382812, 11.104440689086914, 121.78761291503906, -26.85211944580078, 89.93208312988281, -136.822265625, 75.24468231201172, -28.24155616760254, 78.67327880859375, 37.13933563232422, -251.16012573242188, 198.40219116210938, -114.16008758544922, -142.06134033203125, -185.1647491455078, -42.67893981933594, -152.30331420898438, -96.09088897705078, -0.17094917595386505, -66.35841369628906, -69.92472839355469, -37.98895263671875, -207.0479736328125, 22.563552856445312, 154.8973388671875, -158.77699279785156, 28.517566680908203, 28.59029769897461, 93.13081359863281, -162.26898193359375, 105.06396484375, 46.466617584228516, -196.88479614257812, -112.40737915039062, -24.102169036865234, -2.577868700027466, -47.314544677734375, -156.13192749023438, 43.8717041015625, 146.3681640625], [-137.59251403808594, -213.88177490234375, 50.85686492919922, 79.41079711914062, -49.964752197265625, 78.87288665771484, 38.08306121826172, 161.5874481201172, 106.94102478027344, -15.86140251159668, 40.907222747802734, -104.58338165283203, 71.27490997314453, -47.397926330566406, 118.43345642089844, -36.848453521728516, -144.74131774902344, 105.36917877197266, 34.6777458190918, 78.16737365722656, 144.35401916503906, -82.9012680053711, -67.80103302001953, 7.307565689086914, -98.41950988769531, -195.76959228515625, 28.172195434570312, 94.84178924560547, -61.40354919433594, 97.45142364501953, 35.7643928527832, 62.51272201538086, -32.755069732666016, -1.0845643281936646, 119.52748107910156, 159.0789337158203, 153.43719482421875, -46.790870666503906, 4.648959636688232, 64.80207824707031, 120.76044464111328, -9.142963409423828, 44.11823272705078, -172.4953155517578, -29.069686889648438, 79.49208068847656, 55.35718536376953, 190.9037322998047, -110.50284576416016, -87.93431091308594, -135.82638549804688, 119.13325500488281, -49.92973709106445, -74.8803482055664, -48.159725189208984, 4.796491622924805, 11.84085464477539, 63.61996841430664, -139.70396423339844, -2.3225698471069336, -4.68387508392334, -222.0477294921875, -93.40409851074219, 51.3109245300293], [43.12041473388672, -73.5838394165039, -40.79821014404297, 37.368324279785156, -215.4744873046875, 75.94385528564453, 170.9591827392578, -27.70882225036621, 27.606374740600586, 119.70556640625, -46.620872497558594, 68.24826049804688, 17.347925186157227, 66.49907684326172, -20.46087646484375, 79.4735107421875, 46.57390594482422, 17.04452133178711, -41.32969665527344, 122.42478942871094, 2.3272993564605713, 99.28132629394531, -137.44163513183594, -7.775852680206299, 24.366830825805664, 176.8176727294922, 89.9667739868164, 38.2752571105957, -47.19806671142578, -93.9751968383789, -34.168418884277344, 48.70973587036133, -54.69976043701172, -193.54115295410156, -118.31895446777344, 13.754476547241211, -67.7145004272461, 63.95711898803711, -116.6856918334961, -64.46461486816406, 158.55001831054688, -41.723026275634766, 176.13282775878906, -72.07855987548828, 211.62789916992188, 149.5972442626953, -75.64013671875, 41.344032287597656, -169.7187957763672, -84.71942901611328, 44.702117919921875, 48.27335739135742, -17.74726104736328, 106.84786224365234, 32.58805847167969, 150.72900390625, -43.705711364746094, 32.63295364379883, -68.73902130126953, 276.20086669921875, 82.15804290771484, 17.448448181152344, 91.07087707519531, 120.11138153076172], [65.74773406982422, -189.17483520507812, -92.06158447265625, 101.2085952758789, 12.47497844696045, -33.088497161865234, -58.7851676940918, -17.946346282958984, 76.24015045166016, -114.07244110107422, -138.50137329101562, -49.099361419677734, -13.95339584350586, -46.676971435546875, 98.64099884033203, -89.82951354980469, -37.70249557495117, 23.208993911743164, -80.4087142944336, 123.45292663574219, -118.60237884521484, -43.379356384277344, 15.67388916015625, 98.825927734375, -193.831298828125, -36.95610046386719, -84.52827453613281, -57.4227294921875, -30.876272201538086, 58.60258102416992, -47.66862487792969, 23.538803100585938, 174.24407958984375, 127.61363220214844, -35.687591552734375, 77.26261138916016, 12.02855396270752, 142.0314178466797, -3.767854690551758, 4.369932651519775, -71.75458526611328, -82.4109878540039, 124.90141296386719, 237.48753356933594, -1.652146339416504, 14.742059707641602, 81.96217346191406, 163.37301635742188, 16.273876190185547, 32.06562805175781, 35.05592727661133, -5.245560169219971, -143.1766815185547, -53.2210693359375, -8.093511581420898, 103.579345703125, -25.402631759643555, 127.97062683105469, 53.02005386352539, -61.820621490478516, 75.22777557373047, 13.557418823242188, -27.41570281982422, 59.0507926940918], [35.86407470703125, -65.13127899169922, 33.104129791259766, 49.4684944152832, -81.66182708740234, 0.25792717933654785, -71.99266052246094, -83.2012939453125, -54.989715576171875, -31.921693801879883, -91.14824676513672, -62.990909576416016, 8.132949829101562, 19.522693634033203, 14.924747467041016, -71.79086303710938, 39.81163024902344, 106.79589080810547, -152.9422607421875, 120.2457504272461, -22.987709045410156, 77.61753845214844, -90.4900131225586, 256.50616455078125, 14.041090965270996, 83.08059692382812, 82.65897369384766, -17.01309585571289, 205.5249786376953, 98.13133239746094, 29.1755313873291, 211.78907775878906, 29.046234130859375, -125.33969116210938, 132.2445831298828, 18.20343017578125, 19.887475967407227, 61.10673904418945, 87.7953872680664, 42.602203369140625, -4.710821151733398, -12.322888374328613, -110.29518127441406, 8.43509292602539, 47.73284149169922, -70.78083038330078, 80.9823989868164, 23.545902252197266, 108.57283020019531, 27.2867374420166, 38.91992950439453, -39.76348876953125, 223.05551147460938, 9.972091674804688, -67.21295928955078, -149.28297424316406, 63.83809280395508, 96.58546447753906, -136.80105590820312, -123.64391326904297, -58.573280334472656, -61.179054260253906, -70.55023193359375, 11.50895881652832], [-30.205638885498047, 131.66932678222656, -29.47488784790039, -116.0713882446289, 6.4649457931518555, -128.46554565429688, 105.530029296875, -8.113428115844727, 31.155790328979492, -117.13347625732422, -73.44979095458984, 140.57496643066406, 200.3059539794922, 79.7831039428711, -127.36872863769531, 95.76990509033203, -77.11796569824219, 116.04853057861328, -157.68490600585938, 94.72683715820312, -153.19573974609375, -3.9397356510162354, -52.12151336669922, 90.16277313232422, -52.633174896240234, -47.32078170776367, -161.3834686279297, 27.906206130981445, -54.8377799987793, -10.404250144958496, 134.17578125, 50.78028869628906, 158.96620178222656, -163.1525115966797, 51.11365509033203, 77.35824584960938, -89.30290222167969, -56.385780334472656, -15.35103988647461, 75.17466735839844, 171.74386596679688, -68.65884399414062, 130.25059509277344, 39.486968994140625, 174.2592010498047, -36.14385986328125, 63.682960510253906, 123.1693115234375, 67.65667724609375, 0.8799629807472229, -83.91253662109375, 158.05467224121094, 87.25740814208984, -89.74515533447266, 26.201000213623047, -167.31167602539062, 121.22607421875, 148.00083923339844, -198.91490173339844, 36.485809326171875, 22.434049606323242, 42.369232177734375, 216.54867553710938, -53.525634765625], [-51.239227294921875, -67.48578643798828, 64.60710906982422, -98.9085693359375, -217.29452514648438, 15.435853958129883, 219.32615661621094, -92.80947875976562, -43.7692985534668, -23.51113510131836, -79.46918487548828, -178.66473388671875, 121.40983581542969, 41.50579833984375, 39.44449996948242, -99.2991714477539, -45.75815963745117, 116.2302474975586, 136.1092529296875, 35.71139144897461, -202.40908813476562, -4.404481887817383, 28.860620498657227, -6.699171543121338, -55.71460723876953, -66.15786743164062, 82.82905578613281, -0.5729552507400513, 5.303577423095703, -10.55188274383545, -128.0835418701172, -190.25390625, 92.4653091430664, 152.70993041992188, 313.0527648925781, -45.81166076660156, -28.696685791015625, 120.2770004272461, -4.2058024406433105, -60.186187744140625, 63.10965347290039, 48.7585334777832, -101.02076721191406, -109.68708801269531, -136.4846954345703, -248.26498413085938, -104.9771499633789, 60.740474700927734, -58.35953903198242, -160.14317321777344, -76.52906799316406, 96.7634048461914, -28.611684799194336, -104.67070770263672, -7.833133697509766, -96.98466491699219, -36.82899475097656, 1.8690241575241089, -288.6034240722656, -44.56269454956055, -78.56209564208984, -135.70263671875, -158.24147033691406, -2.656175136566162], [86.82524871826172, -32.102752685546875, 41.155784606933594, 248.2369384765625, 60.648780822753906, 6.968297958374023, -190.44036865234375, -3.4612514972686768, 25.9212646484375, 62.8126220703125, -136.78501892089844, -60.185401916503906, 31.101255416870117, -9.22591495513916, -37.354896545410156, 1.5216346979141235, 102.9612045288086, -103.99103546142578, -70.29631805419922, -63.28488540649414, -46.474666595458984, -53.69998550415039, 73.53263092041016, -26.592918395996094, -8.872600555419922, -87.60994720458984, -118.40209197998047, -94.8884506225586, -51.09367752075195, 42.81615447998047, 10.928857803344727, -2.016467332839966, -107.60704803466797, 40.6671028137207, -135.3247833251953, -41.06804275512695, 99.4095687866211, 61.91777038574219, -34.19987487792969, -104.47753143310547, 73.89501953125, -148.94454956054688, -1.6183357238769531, 104.41155242919922, 61.17901611328125, 47.07041931152344, 42.56269836425781, 27.713390350341797, 127.21147918701172, -77.8212890625, -155.38717651367188, -115.53003692626953, -92.42015838623047, -2.3615522384643555, 72.16993713378906, -1.2427918910980225, -43.32234573364258, -40.362892150878906, -227.71060180664062, -134.10928344726562, -51.38042449951172, -37.170623779296875, -59.291481018066406, -57.25382995605469], [75.95464324951172, -53.7396354675293, -60.97281265258789, -145.65341186523438, 49.634422302246094, -60.83073806762695, 24.130016326904297, 86.09235382080078, -206.13328552246094, 30.49036979675293, -57.22832489013672, -165.00189208984375, 45.26545715332031, 100.5171890258789, -14.677085876464844, -65.0886459350586, -108.67491149902344, -73.98656463623047, -50.23251724243164, 35.727149963378906, -70.64923095703125, -26.8751163482666, -16.658018112182617, -71.78174591064453, -27.067140579223633, 83.90399169921875, -199.29954528808594, -9.571158409118652, 122.19905853271484, 188.27859497070312, 9.194642066955566, 63.90318298339844, -128.87875366210938, 50.12468338012695, -7.749587535858154, -62.3036003112793, 156.84506225585938, -84.84800720214844, 49.88609313964844, 12.636453628540039, -138.95233154296875, -164.5327911376953, 126.21871948242188, 15.451302528381348, -229.59359741210938, 35.65077209472656, -14.7606782913208, 188.6751251220703, -4.95412540435791, -44.58518981933594, 36.73289108276367, -52.21341323852539, -25.763715744018555, -33.95137405395508, 50.00248718261719, -21.93031120300293, -34.22897720336914, -11.938902854919434, -38.504859924316406, 38.72256088256836, 72.74322509765625, 34.476890563964844, -21.276247024536133, -280.4978332519531], [70.12112426757812, -176.40773010253906, -4.892786502838135, -103.85538482666016, -80.09041595458984, -6.277074337005615, 110.26206970214844, 18.246402740478516, -134.5813751220703, -112.368408203125, 86.35173034667969, 37.00041580200195, -41.58592987060547, 104.90444946289062, 4.80309534072876, 1.7300268411636353, -40.4521598815918, -192.12332153320312, 114.71514892578125, 29.912626266479492, -15.309130668640137, -2.463848829269409, -5.865208148956299, -63.43912887573242, -55.79099655151367, 43.07502365112305, -20.651485443115234, 8.349038124084473, 0.89614337682724, -43.2981071472168, -113.98810577392578, 143.73548889160156, -133.0382537841797, -81.15243530273438, -136.9624481201172, 29.22646713256836, 144.98703002929688, 157.5455780029297, -127.0341796875, -48.8838996887207, -82.76026916503906, -4.708817958831787, -78.19859313964844, -123.24787902832031, 84.52898406982422, -151.00555419921875, 94.06841278076172, -76.01470947265625, 17.99266242980957, -11.115870475769043, 112.56401062011719, 44.02520751953125, -62.30039978027344, -121.74016571044922, 146.33648681640625, 60.72016906738281, 54.68044662475586, -98.63866424560547, 33.273826599121094, -33.969024658203125, -54.23207092285156, 66.28072357177734, 104.59022521972656, -132.92166137695312], [73.94420623779297, -36.84318161010742, 26.360340118408203, -11.262361526489258, 117.11530303955078, -42.136451721191406, 223.25108337402344, 112.96468353271484, -3.1430203914642334, -124.38042449951172, 100.97061157226562, 110.98831176757812, -247.19068908691406, 47.06376647949219, 33.71458053588867, 172.4977264404297, 164.38864135742188, -130.2942657470703, 21.63249969482422, 15.502443313598633, 101.65650939941406, -200.61563110351562, -37.383949279785156, -29.76841926574707, -265.75927734375, 52.485389709472656, -3.707895040512085, 43.793540954589844, 67.75729370117188, -45.1983642578125, -88.57209014892578, -18.86821174621582, 19.035619735717773, -43.668914794921875, -74.2311782836914, -153.16281127929688, -64.97320556640625, 155.92276000976562, 162.15908813476562, 53.10773849487305, 23.106534957885742, -90.85186004638672, -102.4946060180664, -84.53614044189453, -119.31796264648438, -49.96059799194336, 11.781081199645996, -103.96135711669922, 37.58617401123047, 9.809598922729492, 21.14407730102539, 118.81680297851562, 55.32371520996094, -150.57669067382812, 7.2910590171813965, -63.404083251953125, -36.70857238769531, -72.98764038085938, -179.8222198486328, 103.54319763183594, -72.41026306152344, 198.62203979492188, 148.83126831054688, -86.85718536376953], [-185.78146362304688, 45.20839309692383, -9.561543464660645, 197.1194610595703, 34.66942596435547, 18.901742935180664, -89.64730834960938, 28.001907348632812, -135.70452880859375, -107.62120056152344, 55.5771598815918, -113.31477355957031, 82.18943786621094, 141.8013153076172, 2.940723180770874, -46.752052307128906, 114.91468811035156, 71.5305404663086, -24.44845962524414, 68.42744445800781, 83.10692596435547, -81.76937103271484, -186.3577117919922, -10.587691307067871, 171.2330780029297, 190.83901977539062, -80.09986877441406, 110.0433120727539, 158.67855834960938, -56.98921203613281, -98.09578704833984, 9.634485244750977, 104.2881088256836, -70.95442962646484, 220.8570556640625, -46.1275749206543, -110.42214965820312, 81.03824615478516, -0.8511106967926025, 119.63981628417969, -116.47633361816406, -49.27129364013672, -5.3637847900390625, -35.652061462402344, -15.83200740814209, -71.2492904663086, 7.763091087341309, 94.9297866821289, 46.50122833251953, -123.18531799316406, 3.0886240005493164, -4.6027421951293945, 13.861454010009766, 49.19093704223633, 63.45594024658203, -50.336124420166016, -97.2773208618164, 51.414894104003906, -41.261863708496094, -1.0071687698364258, -15.402495384216309, -126.8539047241211, -10.336343765258789, -131.44683837890625], [-30.692718505859375, -1.2038902044296265, -68.54205322265625, -109.72700500488281, -156.68215942382812, 24.202571868896484, -95.09410858154297, 200.92788696289062, -186.95596313476562, -23.013717651367188, 65.51317596435547, -131.3153076171875, -59.97835922241211, -135.11846923828125, -51.46631622314453, -44.79201889038086, 160.20529174804688, -107.9776611328125, 48.25901412963867, 184.01576232910156, -56.944671630859375, -97.21371459960938, 201.27906799316406, -40.18401336669922, 239.04901123046875, 14.296530723571777, 23.627412796020508, 92.1375503540039, 89.59159851074219, -130.93016052246094, -57.25783920288086, -90.75753784179688, 48.042869567871094, -132.86485290527344, 138.04483032226562, 56.8796501159668, 20.665380477905273, -61.88362503051758, 57.66984176635742, -87.284912109375, 231.68258666992188, 86.06046295166016, -156.2467498779297, 58.99501419067383, 126.06787872314453, 14.85075569152832, 234.63702392578125, -66.0233154296875, 93.47837829589844, -66.7637710571289, 20.850189208984375, 85.52208709716797, -35.99690628051758, -87.79547119140625, 88.05140686035156, 95.3155746459961, 13.015175819396973, 122.86207580566406, -170.1529083251953, -168.26417541503906, 116.01622772216797, -35.738990783691406, 18.826879501342773, 71.75699615478516], [-86.22449493408203, -169.4925079345703, 105.47222137451172, 56.72406768798828, 39.18532180786133, -14.780939102172852, -40.462100982666016, -59.81718444824219, -15.558948516845703, 169.99620056152344, -31.290437698364258, 2.536893844604492, -45.11671829223633, 159.58604431152344, 134.59869384765625, -98.04449462890625, -17.23540687561035, -129.3009033203125, 16.028675079345703, -83.2320327758789, 33.958396911621094, 119.47193908691406, -83.30290985107422, -93.63382720947266, 5.436087131500244, 35.53534698486328, 49.289276123046875, 59.35673522949219, 57.6138916015625, -37.28380584716797, 15.29923152923584, 29.524303436279297, 50.088592529296875, 67.57673645019531, -92.39662170410156, -106.92967987060547, 2.3718204498291016, -37.80301284790039, -15.540307998657227, 17.359416961669922, -69.88304138183594, 96.74403381347656, 103.91004943847656, -272.1339416503906, -46.797027587890625, 236.93724060058594, -137.61221313476562, 154.10643005371094, -165.07627868652344, -58.689659118652344, -47.37277603149414, 96.18741607666016, 14.211801528930664, -106.83501434326172, 36.5564079284668, 18.901639938354492, -215.02529907226562, -63.11749267578125, -8.577834129333496, -156.54632568359375, -56.15450668334961, -101.76184844970703, -129.1279296875, 70.50189971923828], [151.40084838867188, 18.445650100708008, -90.33466339111328, 55.286766052246094, 111.55841827392578, 25.152124404907227, 128.8933563232422, 177.59815979003906, -97.48033905029297, 84.19481658935547, 130.35699462890625, 140.86920166015625, 38.58498764038086, 83.37264251708984, 153.2479248046875, -59.6685676574707, 55.50758743286133, 55.962371826171875, 22.92703628540039, -131.3447265625, -63.0397834777832, -48.927757263183594, -40.32406234741211, 35.318260192871094, -131.08665466308594, -167.56570434570312, -129.1192169189453, -47.42885971069336, -80.45098114013672, 74.43836975097656, 62.23012161254883, -116.46754455566406, 85.32876586914062, -51.02950668334961, -153.53558349609375, -21.727008819580078, 3.417693853378296, 45.39870071411133, -64.16765594482422, -125.22578430175781, -94.82762145996094, 149.07460021972656, -31.68144416809082, -47.723506927490234, -0.9873259663581848, 80.3902816772461, 52.52727127075195, 64.13890838623047, -102.0165786743164, -43.11306381225586, 136.47181701660156, -112.501220703125, -39.949764251708984, -35.5490837097168, 103.21224975585938, 228.46206665039062, 158.41175842285156, -73.746826171875, 51.26087951660156, 45.067291259765625, -93.81365203857422, -24.94585609436035, 1.2292377948760986, 94.99358367919922], [180.12147521972656, -78.51380920410156, -12.43971061706543, 16.726299285888672, -48.60110092163086, -191.8786163330078, -184.8560333251953, 20.669618606567383, -108.03318786621094, 101.49101257324219, 116.83651733398438, 34.27338790893555, 35.471717834472656, -4.057694435119629, 58.16471481323242, -38.24742889404297, -45.75613784790039, 260.4212646484375, 23.53201675415039, -86.03216552734375, 16.680286407470703, -148.187744140625, -23.30628776550293, -27.903884887695312, -61.51820373535156, 186.41873168945312, -11.682135581970215, -10.758909225463867, 22.365808486938477, -168.2643280029297, -153.02041625976562, 13.041047096252441, 73.31105041503906, 161.01788330078125, 58.91452407836914, -37.702293395996094, 5.624157905578613, -145.09364318847656, -24.735187530517578, -104.07171630859375, 4.754234790802002, 7.243946552276611, -53.48264694213867, 97.21601867675781, 68.50808715820312, 234.4640350341797, -126.33053588867188, -79.74016571044922, 170.35726928710938, 94.97296142578125, -150.01632690429688, -97.0304183959961, -116.29750061035156, -76.44464111328125, 57.07965087890625, 6.907567977905273, 130.03614807128906, -78.25299072265625, 75.10529327392578, -81.6994400024414, 62.33855056762695, 183.6645965576172, -46.69609069824219, 197.03884887695312], [-64.09355926513672, -105.46763610839844, -76.99140167236328, -140.73562622070312, 19.998659133911133, -64.4491195678711, -64.57483673095703, -5.553112030029297, 12.440816879272461, -6.752758026123047, -174.23390197753906, -27.67335319519043, -31.501895904541016, -72.7515869140625, -155.31890869140625, 108.19216918945312, -68.05474090576172, 14.693172454833984, 47.67404556274414, -95.52943420410156, -141.9419708251953, -28.6166934967041, -30.437063217163086, -9.145267486572266, -11.839884757995605, 48.665775299072266, -55.222412109375, -115.52308654785156, -23.349037170410156, 0.1357048749923706, 9.960610389709473, -45.32350158691406, -80.17549896240234, 48.78813934326172, -116.96939849853516, -79.00601959228516, 87.01192474365234, 29.87928581237793, -79.87866973876953, -21.63380241394043, 19.495832443237305, -37.8369140625, -63.04348373413086, -80.90269470214844, -72.53524017333984, -25.007410049438477, 5.754909992218018, 72.0921859741211, 27.345115661621094, -24.693313598632812, 284.2554626464844, 132.6982421875, -10.26979923248291, -108.17053985595703, -137.8759307861328, 126.65505981445312, 146.25392150878906, 177.04693603515625, 0.7435001134872437, 114.82447052001953, -29.364717483520508, 163.41127014160156, 107.7234878540039, -183.10125732421875], [52.18149185180664, 141.4995574951172, -118.02832794189453, -104.72953033447266, 38.976112365722656, -137.8382110595703, 177.66647338867188, 138.7262725830078, 60.15348815917969, -174.9788055419922, 69.83282470703125, -126.58300018310547, -111.7211685180664, -72.35516357421875, 47.991397857666016, 9.135775566101074, 93.81160736083984, 173.8976593017578, -110.16546630859375, 3.580688953399658, 121.93019104003906, 59.67705535888672, 91.55547332763672, -123.7264404296875, 84.21862030029297, -19.00223159790039, 40.2215690612793, 34.83230209350586, 2.740791082382202, 176.06356811523438, -116.6043930053711, -47.39873123168945, -61.5395393371582, -150.89218139648438, 44.890342712402344, 3.2118923664093018, 120.99286651611328, -106.6964111328125, 69.11321258544922, -126.64448547363281, -284.37689208984375, -14.657636642456055, 177.04058837890625, -46.570804595947266, -32.87692642211914, 25.30071449279785, 65.09686279296875, -118.07585906982422, -7.770741939544678, -19.107481002807617, 68.46176147460938, -26.18058967590332, 45.15402603149414, 74.3305435180664, -0.4514777958393097, 23.6879940032959, -32.43301773071289, -69.3678970336914, 41.867713928222656, 54.83327865600586, -65.80705261230469, -40.90617370605469, -108.89899444580078, 112.74359130859375], [-83.43595123291016, 19.026777267456055, 33.24557113647461, 207.72164916992188, 98.58380126953125, -12.721258163452148, 140.8548583984375, -97.02766418457031, -51.89021301269531, 68.51783752441406, 38.23875045776367, -5.376720428466797, -109.26075744628906, 3.5500566959381104, 131.58413696289062, -109.15910339355469, 214.87440490722656, 78.47637176513672, -113.65745544433594, -26.68964385986328, -72.23770141601562, -39.25483703613281, 139.7113037109375, 79.58187103271484, -105.4647216796875, 112.06253051757812, -101.10063171386719, 38.702335357666016, -66.44152069091797, 43.670101165771484, -160.0200958251953, 41.44118881225586, -18.16636085510254, -22.132003784179688, 145.0906219482422, 45.37815475463867, -2.3059206008911133, 6.692704677581787, 3.5795817375183105, -170.52394104003906, -18.563478469848633, -209.1389617919922, 204.45547485351562, -11.961186408996582, 32.50405502319336, 80.53191375732422, -166.94482421875, 13.815741539001465, -63.339603424072266, 171.4771728515625, -10.892489433288574, 101.42192840576172, 34.44753646850586, 117.7474136352539, 75.37446594238281, 50.07704162597656, -16.470521926879883, 86.51107788085938, -3.1868600845336914, 125.91761779785156, 53.71088790893555, 99.17367553710938, -6.872421741485596, 82.53242492675781], [-107.62068939208984, -162.00518798828125, 87.16931915283203, 159.89549255371094, 153.9699249267578, 33.36860656738281, -122.69377899169922, -66.03665924072266, 159.8479766845703, -4.836331844329834, -41.859004974365234, -96.52706909179688, 27.438232421875, -55.76243591308594, -168.1253204345703, 36.662174224853516, -7.108513355255127, -53.18011474609375, 22.31730842590332, -134.93836975097656, 9.025657653808594, -92.3125, 28.733591079711914, -15.295660018920898, -131.5904998779297, -20.73575782775879, 47.825565338134766, -38.39202117919922, 68.04462432861328, -98.9381332397461, -0.929165780544281, 128.7700653076172, 17.63725471496582, 59.25905227661133, 176.26748657226562, 6.578614234924316, -138.28671264648438, -28.46573257446289, -75.3656234741211, -20.482908248901367, -48.30002212524414, -43.20747375488281, -17.771821975708008, 159.69947814941406, 17.500675201416016, -95.35582733154297, 130.7572479248047, 152.4772186279297, 5.238349437713623, -60.55674743652344, 162.32150268554688, 61.20726013183594, 96.6982650756836, 13.973134994506836, -99.93563079833984, 174.31077575683594, -4.514813423156738, 46.695682525634766, -174.83177185058594, -45.42671203613281, -34.801082611083984, 63.47865676879883, 81.98089599609375, -65.32904815673828], [-214.69358825683594, 31.517038345336914, -30.985755920410156, 106.57501983642578, -78.17955017089844, 107.64948272705078, -116.20427703857422, -81.82796478271484, -19.185958862304688, -3.520798683166504, 91.64595031738281, 15.179614067077637, -183.53536987304688, 189.6526641845703, -76.11807250976562, 67.83184051513672, 36.56282043457031, -9.865407943725586, 100.13607025146484, 38.84284591674805, -39.06501770019531, -167.81968688964844, -59.47114944458008, -66.10253143310547, 4.926187038421631, -59.748043060302734, -87.44551086425781, -175.55462646484375, -117.49320983886719, 48.44377899169922, -295.67901611328125, 19.808076858520508, 89.36653900146484, 30.273128509521484, -32.58280944824219, -107.09269714355469, 32.4873046875, 99.79193115234375, -118.45574951171875, 16.04186248779297, -86.8863754272461, -74.17509460449219, 12.613123893737793, 44.46439743041992, 169.88925170898438, 46.80471420288086, -63.68327713012695, 38.07840347290039, 52.199825286865234, 141.5956573486328, -171.03746032714844, 69.37999725341797, 11.158923149108887, -103.18182373046875, 177.0321044921875, -173.6843719482422, 26.987154006958008, -1.6982948780059814, -6.570899963378906, -41.1545295715332, 254.59628295898438, -102.54658508300781, -105.34739685058594, -63.4652214050293], [141.8875274658203, -31.325542449951172, 114.69142150878906, -78.57244873046875, 154.09703063964844, -74.90209197998047, -90.99647521972656, 6.481689929962158, 135.79017639160156, -49.42698669433594, -19.428918838500977, -256.9419860839844, -16.788433074951172, -16.740821838378906, 156.22503662109375, -8.667342185974121, 97.29655456542969, 123.31774139404297, -1.699549913406372, -124.37548065185547, 68.9534912109375, 57.97113037109375, 42.457435607910156, 80.39273834228516, 223.3673553466797, -25.976896286010742, -98.157958984375, -13.00196361541748, -62.23623275756836, 69.13101196289062, -91.82894134521484, 127.80397033691406, 6.457460403442383, 92.68682861328125, -152.54908752441406, 87.4261245727539, 80.019287109375, -129.93911743164062, -232.02621459960938, -40.592681884765625, 35.55319595336914, 45.45320129394531, 4.612764835357666, -80.55020904541016, -143.5334014892578, -54.24641799926758, 227.06436157226562, -33.061378479003906, 34.366783142089844, -43.69509506225586, 101.7575454711914, 52.70349884033203, 3.5082790851593018, -34.25501251220703, 368.8559265136719, -128.51528930664062, -100.20764923095703, 58.561180114746094, 59.29150390625, 172.42831420898438, -14.40634822845459, 129.47979736328125, -150.91586303710938, -34.06818771362305], [-116.96343994140625, -160.42311096191406, 75.51669311523438, 90.06802368164062, 153.76353454589844, -25.347320556640625, 121.27872467041016, -26.03648567199707, 15.640283584594727, -34.468849182128906, 60.695987701416016, 126.51699829101562, 18.26803970336914, 155.69326782226562, 68.0678939819336, -83.8745346069336, -140.291259765625, -34.75994110107422, 18.055286407470703, 98.21167755126953, -139.7245330810547, 118.57340240478516, 36.36660385131836, -33.572837829589844, 39.755977630615234, -20.976436614990234, -37.20842742919922, 81.60885620117188, 112.12852478027344, 71.62944793701172, 142.96563720703125, 34.940765380859375, -77.1869888305664, -53.54519271850586, -19.283096313476562, 2.2965445518493652, 14.40664291381836, 20.372230529785156, 102.25445556640625, 93.43753051757812, 85.94290161132812, -117.61957550048828, -55.16377258300781, -71.94803619384766, 7.390146255493164, 4.4263529777526855, 111.49297332763672, -49.734344482421875, -132.88832092285156, -42.11334228515625, 96.49668884277344, 54.62932205200195, -94.34139251708984, 229.39808654785156, -74.2035903930664, -15.641488075256348, 15.637142181396484, 87.65034484863281, -69.29896545410156, 24.63125991821289, 49.032623291015625, 81.47328186035156, -162.3352508544922, -58.58140182495117], [-79.96562194824219, -116.54582214355469, -79.53095245361328, -32.10639572143555, 73.41778564453125, 46.50878143310547, -76.0403823852539, -47.598751068115234, -46.42674255371094, -42.2791862487793, -26.137500762939453, -23.513526916503906, 47.82887268066406, -107.74219512939453, 81.14495849609375, 66.19845581054688, -99.37690734863281, 39.05794906616211, 32.9749641418457, -39.516868591308594, 257.6627197265625, 40.6934928894043, 52.022254943847656, -6.3233561515808105, 208.98681640625, -174.35633850097656, -32.87456512451172, -16.29174041748047, 149.13526916503906, -85.48138427734375, -35.71155548095703, -224.65371704101562, -123.6676254272461, 78.37859344482422, 26.44729995727539, 20.41386604309082, 101.83132934570312, 110.69945526123047, -136.23739624023438, 170.15478515625, 80.681396484375, -25.38665199279785, 6.983891487121582, -128.36781311035156, -77.60282135009766, 28.829730987548828, -12.369250297546387, -52.298587799072266, -85.80547332763672, 0.5214390158653259, -61.989402770996094, -67.35197448730469, 28.36570167541504, -68.93374633789062, 100.86627960205078, -61.429447174072266, -142.98141479492188, 112.6447982788086, 98.23855590820312, -130.3944854736328, -23.480209350585938, 10.509086608886719, -55.72370910644531, -150.0782470703125], [112.50909423828125, -53.2938232421875, -40.63031768798828, -54.83618927001953, 166.3961639404297, 91.46422576904297, -125.91657257080078, -175.42807006835938, -207.34066772460938, 67.60189056396484, -95.67446899414062, -77.89495849609375, 11.581974983215332, 51.30765151977539, 58.600399017333984, 91.20062255859375, 202.81968688964844, -59.12599563598633, 86.5115966796875, 196.7374725341797, -115.59071350097656, -79.62850189208984, -87.52155303955078, 70.25707244873047, 117.17278289794922, 86.23485565185547, -15.13889217376709, -209.6706085205078, -56.76863098144531, 49.90354919433594, 134.7880096435547, 84.47412872314453, 49.78524398803711, 197.22264099121094, -114.04673767089844, 57.86027145385742, 64.3333969116211, 120.26193237304688, 100.61512756347656, -19.59577178955078, 149.03042602539062, 55.422447204589844, -146.24334716796875, 142.4104766845703, 56.84698486328125, -10.377167701721191, 112.16123962402344, -41.576416015625, 33.07439041137695, -136.27821350097656, 9.56591796875, -83.80365753173828, -95.21440887451172, -112.96514892578125, -35.51412582397461, 76.79647064208984, -13.22357177734375, 15.460948944091797, 8.655752182006836, 64.02527618408203, 91.89488220214844, 33.997039794921875, -19.148874282836914, -50.464664459228516], [94.64749145507812, -18.111061096191406, 119.96292877197266, 150.61558532714844, -75.20954132080078, 79.39744567871094, -10.581855773925781, -74.14846801757812, 125.749755859375, -23.903867721557617, -108.38340759277344, -13.818299293518066, 52.02322769165039, 25.163572311401367, -46.31652069091797, 20.643678665161133, 45.43571090698242, -23.889347076416016, -185.61419677734375, 92.91578674316406, 51.517581939697266, -189.42160034179688, 46.00193786621094, 29.607267379760742, -29.08026695251465, 57.75947570800781, 136.59921264648438, -117.08061981201172, 111.56963348388672, 16.704540252685547, -49.712127685546875, 29.15079689025879, -15.819547653198242, -116.66166687011719, 13.446035385131836, 155.51388549804688, 51.29538345336914, -57.43228530883789, 55.47920608520508, -27.91506576538086, 4.655645370483398, 19.483943939208984, -92.3174057006836, -137.0249481201172, 3.4673092365264893, -104.59510803222656, -360.9542236328125, 117.1600341796875, -99.43611145019531, -14.545928955078125, -127.65668487548828, 123.27731323242188, 27.827587127685547, -2.0818426609039307, 115.77845764160156, -117.95221710205078, 101.98279571533203, -282.6297607421875, 47.956153869628906, -22.961809158325195, -172.9446258544922, 128.6615753173828, 3.728362798690796, 85.39801788330078], [-112.16658782958984, 64.16374969482422, -64.53091430664062, 59.13233947753906, 47.75669479370117, 114.3804931640625, 73.61211395263672, -30.954069137573242, -178.3039093017578, 125.78572082519531, 137.0545196533203, -6.296265125274658, 23.765478134155273, 32.89585876464844, 168.70428466796875, 210.04400634765625, -14.435752868652344, -1.9520899057388306, 50.79502487182617, -32.29701614379883, -169.3643035888672, -1.4177567958831787, 67.53247833251953, -88.82738494873047, -25.287809371948242, 205.49038696289062, 69.50173950195312, -41.77650451660156, 13.578170776367188, 88.03545379638672, 136.38290405273438, 141.3457794189453, -196.3991241455078, -50.13901901245117, 56.633419036865234, 93.8193359375, -180.06788635253906, 47.89217758178711, 18.930721282958984, 96.9582290649414, 97.05416870117188, 132.77574157714844, -147.05686950683594, 2.776071310043335, -28.974241256713867, 27.35082244873047, 12.042817115783691, -4.219891548156738, -30.621131896972656, 12.211575508117676, 135.1019744873047, -2.635056495666504, 36.335880279541016, 162.9143524169922, 187.38613891601562, -16.260456085205078, 202.48719787597656, -3.9547998905181885, -47.19047546386719, -72.89157104492188, -64.19550323486328, -64.04454803466797, -3.567589282989502, -212.7204132080078], [-111.73108673095703, 76.39161682128906, -143.03199768066406, 81.76176452636719, -113.20194244384766, -30.745420455932617, -39.609310150146484, 43.68198776245117, 64.7151870727539, 134.87637329101562, -146.54608154296875, -12.67566967010498, -43.28812789916992, -94.49842834472656, 34.33993911743164, 129.6002197265625, -142.5467529296875, -44.64384460449219, 37.06748962402344, -23.221282958984375, -121.9234619140625, -27.654245376586914, -3.1782572269439697, -50.01279067993164, 94.42147827148438, -0.5189786553382874, -6.1027607917785645, 46.76518249511719, -67.33759307861328, -19.347612380981445, -79.09403228759766, -70.04986572265625, -19.74530029296875, 36.059078216552734, 146.9879608154297, -91.83419799804688, 126.5662612915039, -11.412827491760254, -121.46995544433594, 50.32331848144531, -149.45794677734375, 133.4535369873047, -27.032865524291992, -19.610685348510742, 178.68360900878906, 21.023944854736328, 105.40025329589844, 119.04082489013672, -115.73883056640625, -70.78102111816406, 15.331690788269043, 43.940834045410156, -53.35103988647461, 122.13369750976562, -135.35914611816406, -8.747808456420898, -158.4221649169922, 69.80294799804688, -57.59526062011719, -110.04009246826172, 65.85330963134766, 39.437232971191406, 119.92414093017578, -9.862799644470215], [163.2807159423828, -110.33757019042969, -13.219205856323242, -59.649932861328125, -155.2606964111328, 2.983140468597412, 13.168436050415039, 22.75777816772461, -189.45794677734375, -9.034735679626465, 68.04554748535156, -175.55299377441406, 62.6320686340332, -188.6304931640625, -81.58258056640625, -38.33317565917969, 104.1633529663086, -22.240562438964844, 71.84473419189453, -47.62358093261719, 27.29876136779785, 105.41862487792969, 151.805419921875, 87.6644287109375, -164.6388702392578, 69.08169555664062, 113.91278076171875, 182.0683135986328, -63.49380111694336, 295.20977783203125, 30.279699325561523, 110.42058563232422, -63.9183235168457, -114.31951141357422, -158.1576690673828, 12.982784271240234, 38.82149124145508, 143.7030792236328, 61.19147491455078, -19.599626541137695, 159.02455139160156, 46.377777099609375, 6.72958517074585, -41.47852325439453, -127.1005859375, 150.03807067871094, 24.000720977783203, -148.6819305419922, 26.087432861328125, 112.45345306396484, 138.0240020751953, -83.5772476196289, -165.1253662109375, 23.435518264770508, 115.7717056274414, -243.4346923828125, -29.69529914855957, -3.967395544052124, 10.563170433044434, -87.24687957763672, -57.05265426635742, 62.38225555419922, 134.82249450683594, -38.30968475341797], [0.7605757117271423, -15.703529357910156, -40.152374267578125, -93.11478424072266, 1.392885446548462, -76.32972717285156, 273.8094482421875, 128.1778106689453, 265.9015197753906, 60.856266021728516, 1.083371639251709, 73.66336059570312, 52.37356948852539, 131.35281372070312, -27.479759216308594, 147.82359313964844, 97.02654266357422, -47.100460052490234, -125.83941650390625, -46.96902084350586, -34.59752655029297, -121.88676452636719, -172.49420166015625, 32.13410186767578, 151.81381225585938, 79.43384552001953, 31.521678924560547, -114.09097290039062, -98.54259490966797, -36.21450424194336, 22.534297943115234, -82.37289428710938, -5.2888360023498535, 91.90509033203125, 68.76362609863281, -89.00482940673828, -67.37018585205078, -90.83808135986328, -25.673114776611328, 144.52001953125, 91.734619140625, 60.63010025024414, 27.932392120361328, 59.2750129699707, 5.915546417236328, 87.19003295898438, 3.899857759475708, -41.08081817626953, 203.21278381347656, -14.91362190246582, 23.369043350219727, 36.050865173339844, 97.08890533447266, -78.49729919433594, -20.558998107910156, -55.408905029296875, -25.629980087280273, 24.110517501831055, -26.023958206176758, -60.99706268310547, -25.81829833984375, 90.95894622802734, -11.254558563232422, 134.57000732421875], [-58.35136413574219, -156.35595703125, 29.574146270751953, 106.8802719116211, 116.69168090820312, -47.194190979003906, 160.567138671875, -112.13636779785156, 61.527854919433594, 247.27239990234375, -51.37441635131836, -0.43633583188056946, -68.42504119873047, -84.52641296386719, -39.9751091003418, 192.90475463867188, 16.82771873474121, 41.473411560058594, -22.521648406982422, 135.9895477294922, -208.70254516601562, -6.298012733459473, 53.9334602355957, 50.09738540649414, 187.4929656982422, -35.120689392089844, -17.786367416381836, 144.57679748535156, 120.58155822753906, 80.2411880493164, 106.90866088867188, -49.182838439941406, -123.1985855102539, 48.291778564453125, 0.6087245345115662, -30.55284309387207, -74.62045288085938, -67.2474136352539, -2.5847115516662598, 1.506797194480896, 1.7702176570892334, 3.076998710632324, -61.188621520996094, 26.174571990966797, 62.52580261230469, -43.31963348388672, 52.631134033203125, -113.9182357788086, -141.0845489501953, -5.18841552734375, -59.1237907409668, 18.412137985229492, -41.17134094238281, -204.81051635742188, -194.1292266845703, 57.87409591674805, 25.370834350585938, -95.84423065185547, -25.54804039001465, -39.64059066772461, -37.16581726074219, 58.47029495239258, -23.43370246887207, 114.73197174072266], [10.03849983215332, -125.32318115234375, -63.78400802612305, -163.7646026611328, 242.03759765625, 0.8249865770339966, -155.98895263671875, -184.7942352294922, -74.73159790039062, 0.8652845025062561, 53.51993942260742, 30.19390869140625, -134.57730102539062, 165.17543029785156, -127.3056640625, 197.69363403320312, -43.21474838256836, -207.59605407714844, -32.34000015258789, -24.519025802612305, -20.23933982849121, -129.1726837158203, -14.71235466003418, 54.118106842041016, -21.19802474975586, -129.90757751464844, 56.0021858215332, -60.29060363769531, -8.264933586120605, -79.65023803710938, -49.8298225402832, 93.38557434082031, -69.3606948852539, -241.1289520263672, 108.115478515625, 14.241531372070312, 44.36475372314453, 13.853713035583496, -40.69593811035156, -13.394909858703613, 85.78797149658203, 103.8890609741211, -75.03917694091797, 4.059874534606934, 40.019569396972656, 79.0281753540039, -49.32288360595703, -80.57076263427734, -140.1195526123047, -31.620786666870117, 16.935565948486328, 137.10093688964844, 81.66703033447266, -64.84075927734375, -155.6043243408203, -15.350676536560059, 30.70195960998535, 71.39447784423828, -46.2187385559082, 81.88317108154297, -79.3083724975586, 59.84583282470703, 106.89507293701172, 22.746749877929688], [-86.23918151855469, -82.09760284423828, 196.89157104492188, -34.877071380615234, -90.23885345458984, -16.36526107788086, -59.69236373901367, -107.81720733642578, 102.5929946899414, -52.23994445800781, -84.25605010986328, -78.88888549804688, 30.68950843811035, 91.06723022460938, 145.35275268554688, -53.81367111206055, -65.10643768310547, 1.7165230512619019, 11.000053405761719, -62.47955322265625, -72.74517822265625, 69.73500061035156, -90.24439239501953, -65.40850830078125, -131.34320068359375, -51.939876556396484, -0.3088587820529938, 83.46308898925781, 21.053449630737305, -38.441280364990234, 24.909788131713867, 73.48078918457031, 102.166015625, -115.14703369140625, 33.83388900756836, -64.28890228271484, 62.67498016357422, 63.48037338256836, -16.958913803100586, 211.1710968017578, -102.77483367919922, 72.42532348632812, 7.565898418426514, 41.28335952758789, 81.34871673583984, 39.38309097290039, 8.448688507080078, -49.857120513916016, -227.04576110839844, 192.88926696777344, 6.381972789764404, -68.97676849365234, 11.468859672546387, 29.659263610839844, -41.38825607299805, 23.896438598632812, 88.94441986083984, -114.34465789794922, 36.63021469116211, 91.3111343383789, 47.8322639465332, -196.310302734375, -18.661542892456055, 83.13786315917969], [-0.14219173789024353, -54.64299011230469, 227.71482849121094, 81.9083023071289, 67.93024444580078, 84.34024810791016, 103.73304748535156, 47.00120162963867, -9.97411823272705, -54.28611755371094, -91.22489929199219, -66.66205596923828, 36.65265655517578, -14.078486442565918, 137.56768798828125, -34.74884033203125, 80.86119079589844, -36.49441909790039, -22.95787239074707, -59.341644287109375, -55.10433578491211, 95.43521881103516, 226.3446502685547, -122.25409698486328, 187.61842346191406, -97.48536682128906, -36.146488189697266, 164.8197021484375, -30.033769607543945, -0.11057091504335403, 9.196850776672363, -43.734439849853516, 65.46094512939453, -63.470149993896484, 25.982454299926758, -78.79994201660156, -92.42456817626953, -160.67347717285156, 87.68708038330078, 245.77293395996094, 75.86527252197266, 119.52803039550781, -28.936023712158203, 7.309642791748047, 44.515235900878906, 17.428590774536133, 19.284141540527344, -3.647207260131836, -10.77840805053711, -35.06791687011719, -71.52496337890625, -64.4643783569336, 219.97198486328125, -295.801025390625, 135.6751251220703, -119.7011489868164, -7.017759323120117, -58.68607711791992, -62.94758224487305, -136.25254821777344, -48.298912048339844, -16.0955753326416, -52.424869537353516, -49.76352310180664], [-0.7599564790725708, 126.9683609008789, 91.15157318115234, -82.10858154296875, -65.63468933105469, 146.2625732421875, 52.76860427856445, -168.24781799316406, 8.634403228759766, 105.24763488769531, 57.24745178222656, -37.820350646972656, -34.52630615234375, 0.9315621256828308, 87.45565032958984, -1.157644271850586, 244.8083038330078, 65.33548736572266, 27.843433380126953, 130.73377990722656, -144.18075561523438, 90.06364440917969, -34.467079162597656, -5.6769304275512695, -27.51709747314453, -28.374006271362305, 4.63058614730835, -159.26998901367188, 175.30523681640625, 49.81053924560547, 35.483150482177734, 233.26942443847656, -107.80763244628906, -148.79234313964844, 110.78897857666016, -128.65460205078125, 62.643924713134766, -4.579456329345703, 48.347434997558594, -75.66500091552734, 32.944889068603516, 124.68535614013672, 201.1575927734375, -125.27269744873047, -158.63319396972656, 198.39791870117188, -46.44077682495117, 41.488189697265625, 19.549816131591797, -124.03872680664062, -74.02388000488281, 53.57167053222656, -27.090429306030273, 77.43460083007812, -132.89723205566406, -49.929630279541016, 13.779850959777832, 2.0557706356048584, -46.79112243652344, -63.062591552734375, 23.025808334350586, -205.77806091308594, 57.289398193359375, 22.609111785888672], [-61.14673614501953, 214.75003051757812, 261.22601318359375, -72.5451431274414, -260.08203125, -4.5026984214782715, 114.33060455322266, 95.18080139160156, 90.78599548339844, 141.4395294189453, 3.5263822078704834, 112.83009338378906, -58.132057189941406, 42.08688735961914, 47.646183013916016, -5.2992963790893555, 26.737041473388672, -24.150606155395508, -114.95166778564453, 37.655086517333984, -10.84977912902832, -56.3406982421875, 106.30264282226562, 62.875282287597656, -37.27798080444336, -73.61994171142578, 10.644179344177246, -128.5973358154297, 227.2989959716797, -28.653179168701172, 188.33306884765625, -193.4192352294922, 71.30809020996094, -101.67694854736328, 59.98318862915039, -115.69137573242188, 59.30775451660156, -152.4335479736328, -15.259552955627441, -99.25074005126953, 45.97136688232422, -87.80284881591797, -1.8411086797714233, -26.483823776245117, -83.70233154296875, -1.287030577659607, 8.302343368530273, -89.97897338867188, -118.41213989257812, -32.64945602416992, 35.62276840209961, -28.836048126220703, -48.28752899169922, 61.11699676513672, -1.1069481372833252, 8.507110595703125, 55.6343994140625, -101.1266860961914, -90.62872314453125, -27.81773567199707, -77.2532958984375, 94.8866958618164, -78.12081909179688, -32.375885009765625], [122.76294708251953, -47.501792907714844, -146.7840118408203, -100.52542877197266, 70.4600601196289, -173.00152587890625, 30.8067569732666, -47.98027038574219, 72.30658721923828, 18.049531936645508, 77.74304962158203, 73.05075073242188, 50.611209869384766, 9.790810585021973, 122.93413543701172, 104.39964294433594, -54.92090606689453, -59.94401168823242, -67.24522399902344, -68.50894927978516, -50.15583801269531, -52.2177734375, 101.3537368774414, -116.34542083740234, 114.62239074707031, 138.63323974609375, 119.981689453125, 3.5976271629333496, -174.39968872070312, 81.88824462890625, 135.0476531982422, -104.07964324951172, -75.03173828125, -59.84625244140625, -34.770896911621094, 4.751509666442871, -228.0985107421875, 73.12535858154297, 1.151503562927246, -213.41171264648438, 99.17178344726562, -57.46923065185547, -94.43179321289062, -12.584179878234863, -7.90210485458374, 100.69085693359375, -19.597164154052734, 12.290563583374023, 51.112876892089844, 85.73110961914062, -116.15518951416016, 98.19519805908203, -180.19158935546875, -10.885168075561523, -90.3584213256836, 9.148150444030762, 70.5857925415039, 55.4242057800293, -47.60141372680664, -305.81695556640625, -130.0471954345703, -132.9432830810547, -85.02252197265625, -55.24860763549805], [6.2513885498046875, -25.01774024963379, -42.06546401977539, 24.87049674987793, 240.94862365722656, 40.02149963378906, 5.207455635070801, 136.42115783691406, -114.71420288085938, -8.98292350769043, 7.596253871917725, 77.5738754272461, -87.35140991210938, -29.28379249572754, 28.72673225402832, 12.627872467041016, -47.66629409790039, 6.521852016448975, 62.457584381103516, 51.48422622680664, -26.131258010864258, 72.7319564819336, 6.74241304397583, 81.93013000488281, -72.9039535522461, 116.23021697998047, 41.073570251464844, -8.97045612335205, 85.60620880126953, -22.14726448059082, 56.94487762451172, 247.7828826904297, -52.676368713378906, 14.206082344055176, -82.23136138916016, -59.93185806274414, 26.97258758544922, -33.05682373046875, 44.451385498046875, 194.107421875, -111.8548583984375, 186.19642639160156, 60.57670974731445, 44.895530700683594, 27.287189483642578, 84.94434356689453, -61.041568756103516, -80.27420043945312, -86.89701080322266, -24.98088836669922, 9.982748031616211, 115.65747833251953, -87.32058715820312, -149.44058227539062, 38.90169906616211, -174.06307983398438, 93.43975067138672, -89.46170806884766, -57.53899383544922, 58.70318603515625, 170.1329803466797, -98.74263000488281, -0.5910502672195435, 44.79026794433594], [109.40332794189453, 167.23092651367188, 141.30149841308594, 39.5070686340332, -16.506887435913086, -44.56485366821289, 48.104496002197266, 178.6095428466797, 87.84156036376953, -107.2501449584961, -17.15673065185547, -11.76884937286377, -87.27819061279297, -37.62016296386719, -23.254304885864258, -245.58631896972656, 49.75977325439453, -53.25859451293945, -249.4921112060547, -25.237714767456055, -32.195621490478516, 26.347002029418945, 33.60979461669922, -135.9673309326172, 1.2615127563476562, -150.79714965820312, 84.16783905029297, 13.095959663391113, -2.8355653285980225, -85.33934020996094, -209.86585998535156, 44.693214416503906, 104.01104736328125, 44.27625274658203, -147.34788513183594, -84.02809143066406, 188.6233673095703, -17.645872116088867, -120.516845703125, 146.41552734375, 97.81251525878906, -94.5225601196289, -49.88838195800781, 91.85881805419922, 3.5734548568725586, -24.581836700439453, -128.58047485351562, -22.020776748657227, -158.7085723876953, -90.02709197998047, 148.7022705078125, -248.8353271484375, 32.96095657348633, -113.73926544189453, -40.685794830322266, -6.686246871948242, 23.234968185424805, 81.41358947753906, -61.16524887084961, -48.795387268066406, 3.3683793544769287, 146.75454711914062, -61.0360221862793, -44.833255767822266], [66.4727783203125, 127.23700714111328, 92.95478820800781, 5.006482124328613, 6.625851631164551, 26.19080352783203, 17.114810943603516, 0.7646907567977905, 86.47096252441406, 185.7067413330078, -8.43163013458252, -27.52072525024414, 21.99890899658203, 34.1701545715332, 77.02946472167969, 35.244571685791016, 86.26617431640625, -132.20606994628906, 10.08064079284668, 91.70742797851562, 107.07012176513672, 1.7823901176452637, 67.90694427490234, 107.22576904296875, 183.77842712402344, -60.29769515991211, 70.20845031738281, -63.92845153808594, 12.642532348632812, 49.490020751953125, -66.97319793701172, 35.946903228759766, 40.30373001098633, -93.31346130371094, 1.5082522630691528, -35.51384735107422, -72.92304229736328, -111.40107727050781, 109.29054260253906, 150.75399780273438, 67.87820434570312, -211.95193481445312, 143.32211303710938, -128.5626220703125, 4.405669212341309, 11.974451065063477, 74.72542572021484, -191.2737274169922, 34.33462905883789, 57.430206298828125, -49.240753173828125, -48.54011154174805, 114.65419006347656, -0.555203378200531, -189.28640747070312, -55.73475646972656, 55.29774856567383, 184.59048461914062, 29.07493019104004, 175.99429321289062, -96.57549285888672, -112.51038360595703, -83.14129638671875, -2.031463146209717], [105.77063751220703, -18.655088424682617, 141.8740997314453, -27.85266876220703, 12.95371150970459, 41.61105728149414, 125.30937194824219, -204.0309600830078, 56.91370391845703, 77.34886169433594, -53.806602478027344, 212.53997802734375, -72.83798217773438, -37.352596282958984, -57.27436828613281, -35.826297760009766, 207.9048614501953, -23.254066467285156, 61.09062957763672, 184.39984130859375, -17.253873825073242, 89.15579986572266, 74.39618682861328, -8.410208702087402, -53.5692138671875, 12.336167335510254, 169.22164916992188, -5.006871223449707, 99.83606719970703, -155.90846252441406, -31.628583908081055, -105.1512451171875, -25.43572425842285, 104.74508666992188, -202.3542938232422, 66.70187377929688, 50.92405700683594, -26.270843505859375, -27.082103729248047, 61.29195785522461, 72.29729461669922, -116.14915466308594, -132.1549835205078, -19.644121170043945, 52.268497467041016, 32.407779693603516, -7.447518348693848, 29.3299617767334, 81.42282104492188, 85.70986938476562, 171.2969207763672, 17.587284088134766, 5.518893241882324, 28.051902770996094, -143.52197265625, 110.35096740722656, -81.43012237548828, -138.6414337158203, 93.0625991821289, 64.69114685058594, -10.141425132751465, -130.89306640625, 72.58250427246094, 237.75314331054688], [-234.10794067382812, -105.82144165039062, 2.2998573780059814, 7.980946063995361, -25.813722610473633, 117.84700012207031, -52.12320327758789, 92.87662506103516, 115.18790435791016, 18.129772186279297, 34.88454055786133, -248.03231811523438, -0.28586551547050476, -114.70137023925781, 25.223560333251953, 124.69187927246094, 24.50899887084961, -4.484024524688721, 167.79446411132812, 17.89889907836914, -8.3140230178833, 96.73131561279297, -30.004053115844727, 208.31532287597656, -35.08348083496094, 103.88038635253906, -80.6202163696289, -101.35177612304688, 117.63787078857422, -7.021200180053711, -26.94949722290039, 17.897876739501953, -159.3051300048828, 26.084924697875977, -102.16524505615234, -6.263579845428467, 75.40315246582031, -34.09611129760742, 177.32940673828125, 60.35441970825195, -51.52363204956055, -115.02417755126953, -79.8314437866211, 89.46879577636719, 30.95624542236328, 55.34552001953125, 12.115787506103516, -108.74220275878906, 147.1204071044922, -36.54277420043945, 45.40166091918945, -19.29334831237793, 21.507463455200195, -32.087921142578125, 9.836846351623535, 57.74425506591797, 44.41082763671875, -67.01685333251953, -1.3867405652999878, -158.67103576660156, 144.62786865234375, 8.628653526306152, 130.352294921875, 42.68889617919922], [-70.16645812988281, -128.5674591064453, -34.103763580322266, -130.18173217773438, -6.031588554382324, -10.075078010559082, -23.202402114868164, 72.8084487915039, 87.79193878173828, -21.642154693603516, -150.82965087890625, 160.36956787109375, 36.34748840332031, 33.105220794677734, -91.165771484375, -42.5068244934082, -5.721689224243164, 38.96189880371094, 79.8482894897461, -27.683006286621094, -88.20611572265625, 122.51141357421875, 60.320091247558594, -97.31100463867188, -9.716306686401367, 7.867739677429199, 50.34019470214844, 51.84426498413086, -60.50115203857422, -143.7746124267578, -103.49417114257812, -166.4431915283203, -26.617904663085938, 128.2194061279297, 42.97642517089844, 10.03404426574707, 304.9751892089844, -55.279300689697266, 25.58296012878418, -173.45608520507812, -48.24090576171875, 10.66713809967041, -121.56471252441406, -183.00636291503906, 34.046852111816406, 41.69803237915039, -2.6313278675079346, 113.06533813476562, -155.95892333984375, -6.396857261657715, -183.42874145507812, -29.39939308166504, -55.840179443359375, -162.2928924560547, 115.96598052978516, 108.18669891357422, -66.07384490966797, 24.4179744720459, 44.98419189453125, 64.1181640625, 51.38573455810547, -26.186969757080078, -43.0451545715332, 70.74462890625], [73.5341796875, 134.47401428222656, 122.72208404541016, -72.0719223022461, 123.89254760742188, -59.03285217285156, -64.05683135986328, 40.897674560546875, -69.83983612060547, 103.15864562988281, 42.0908088684082, -48.230464935302734, 3.9891908168792725, 86.9723892211914, -26.775585174560547, 48.18055725097656, -115.62859344482422, -28.897737503051758, 17.872472763061523, 24.954322814941406, 150.12567138671875, -2.9985456466674805, -10.156890869140625, -170.326171875, -85.0798110961914, 35.7177848815918, 205.65953063964844, 2.334146738052368, -8.447315216064453, -11.59007740020752, -61.38645553588867, 11.824946403503418, 32.237728118896484, 188.07003784179688, 0.7841511368751526, 49.52865982055664, 94.46832275390625, -68.96837615966797, -115.27357482910156, -16.486125946044922, 81.53600311279297, 141.93853759765625, 106.39044952392578, 205.8383026123047, -66.02592468261719, -10.267091751098633, -127.5661392211914, 92.3746109008789, 162.68666076660156, 50.253692626953125, 8.958281517028809, 20.648408889770508, -77.95915222167969, -94.2166519165039, -29.40319061279297, -10.21518325805664, -35.58295440673828, -118.20326232910156, 48.06227493286133, -27.543336868286133, 12.621718406677246, 94.7816390991211, -135.5025177001953, 95.31072235107422], [70.7669906616211, 15.953339576721191, -34.37059783935547, 165.69776916503906, -60.683353424072266, -125.8995132446289, 52.92756271362305, 87.84718322753906, 145.2408447265625, 53.49800109863281, 48.73061752319336, 156.10299682617188, -168.55972290039062, 64.97002410888672, -2.11940860748291, -72.4891128540039, -40.762428283691406, 121.1712417602539, 194.88848876953125, -24.339601516723633, 14.11299991607666, 9.475406646728516, 93.11630249023438, 297.5914611816406, -272.01202392578125, 147.14393615722656, 34.67047119140625, 64.40355682373047, -1.5293043851852417, -156.00131225585938, 86.85887145996094, 80.19820404052734, -42.78742599487305, -103.92649841308594, 17.263442993164062, 92.36605072021484, 9.914349555969238, -58.926605224609375, -120.9947738647461, 43.488887786865234, 46.28995132446289, 116.07085418701172, 37.8865966796875, 44.786712646484375, 99.59034729003906, -207.72093200683594, 51.75004577636719, -82.04428100585938, -89.23440551757812, 51.50932312011719, 50.70293426513672, 18.823467254638672, 127.47120666503906, -72.48997497558594, 35.17829132080078, -18.722795486450195, 166.10398864746094, -78.43148803710938, -12.700640678405762, 45.411834716796875, 229.73985290527344, 88.27830505371094, 47.499778747558594, 11.445260047912598], [106.63622283935547, 71.7475357055664, -34.08283615112305, 12.748884201049805, 78.90908813476562, -67.03095245361328, -70.70884704589844, 98.49028015136719, 105.98478698730469, -198.49105834960938, 26.897066116333008, -33.526084899902344, -130.94290161132812, 89.27545166015625, -193.30633544921875, 51.56298065185547, 57.12132263183594, -298.12109375, -8.786341667175293, 46.330081939697266, 10.536283493041992, 190.83685302734375, -88.74018859863281, 24.722248077392578, -128.3018798828125, -39.464027404785156, 136.09593200683594, -26.973834991455078, 204.1958465576172, -41.88829040527344, 42.56782531738281, 30.762897491455078, 68.71800994873047, 122.86555480957031, -68.55811309814453, -39.11549758911133, -8.882094383239746, 1.2287553548812866, 162.10641479492188, -134.69479370117188, -5.0428338050842285, -9.21001148223877, 91.86570739746094, -54.54998779296875, 28.72950553894043, -9.396539688110352, -38.2498664855957, 47.72954177856445, 139.8048553466797, 53.08123779296875, -27.06646156311035, 6.1668195724487305, 23.187341690063477, 75.64373779296875, -131.33595275878906, -8.755819320678711, -18.774276733398438, 125.08978271484375, -144.25283813476562, -56.4660758972168, -82.36917114257812, 174.07342529296875, -30.271764755249023, 51.940189361572266], [106.71565246582031, 35.13097381591797, -48.02212142944336, 138.2649383544922, -78.78507995605469, 140.1584930419922, 33.05467224121094, 98.50167846679688, -203.23316955566406, 75.75303649902344, 181.105224609375, 3.737064838409424, -153.12600708007812, -25.359424591064453, 33.25426483154297, -41.93193435668945, -83.60225677490234, 15.668155670166016, -25.168092727661133, 60.256980895996094, -38.896240234375, -11.095927238464355, 35.981754302978516, 70.2180404663086, 85.15094757080078, -37.94878005981445, -10.624834060668945, -28.382234573364258, 150.18905639648438, -39.78124237060547, 0.08218178898096085, -71.29361724853516, -29.565053939819336, -63.682125091552734, -54.56028747558594, 5.294421672821045, 49.78699493408203, 61.53434753417969, 14.197381973266602, 51.096961975097656, -16.807506561279297, -6.370219707489014, -28.230438232421875, 41.91252517700195, 2.875657320022583, 187.3028106689453, -127.28791809082031, 8.880494117736816, -101.60607147216797, -165.94410705566406, -72.52955627441406, -0.3064873218536377, -96.81990051269531, 110.70548248291016, 20.563440322875977, 30.335037231445312, -93.29586791992188, -2.6308774948120117, 37.24370193481445, 47.46162414550781, 155.3256072998047, 4.2802534103393555, 47.07663345336914, -14.172471046447754], [66.00384521484375, -3.452164888381958, 8.919628143310547, 211.7570343017578, 148.1269989013672, -96.78824615478516, -88.43932342529297, 80.10604095458984, -88.22985076904297, -19.898006439208984, 95.43302154541016, -100.6685791015625, 80.0007553100586, -10.206035614013672, 18.18587875366211, -49.83442687988281, 31.15970802307129, 73.69727325439453, -40.04595184326172, 13.142088890075684, -77.04179382324219, 80.3749771118164, -120.51197814941406, -40.54715347290039, -138.72486877441406, -138.74688720703125, 4.099586009979248, 81.35356140136719, 11.999053001403809, 34.59223175048828, -106.63754272460938, 40.163692474365234, 43.945167541503906, -63.43623352050781, 16.45077896118164, 15.12047290802002, 72.57599639892578, -159.9851837158203, -94.82209777832031, -46.022735595703125, 15.425049781799316, 30.503570556640625, -59.06685256958008, -80.4495620727539, 66.42523956298828, 33.287933349609375, -172.8856658935547, 7.806329727172852, 50.03018569946289, 250.07289123535156, 57.9989013671875, 89.4993896484375, -36.444217681884766, 10.52430534362793, -8.742725372314453, 29.102760314941406, 29.257734298706055, -44.553550720214844, 28.60841941833496, -79.69267272949219, 11.24280071258545, -20.719804763793945, -113.96871185302734, 30.774433135986328], [60.90501403808594, -59.69297790527344, 21.501497268676758, -10.659517288208008, -190.61712646484375, -24.75238037109375, -85.59314727783203, -7.353944778442383, -42.983707427978516, -175.15582275390625, -23.71307373046875, -122.21846771240234, -30.28478240966797, -16.1524715423584, -34.993408203125, 78.73873901367188, -29.146181106567383, 32.233985900878906, 92.36734008789062, 89.75607299804688, 160.92857360839844, 58.01300048828125, -167.6209716796875, -30.400850296020508, -166.38113403320312, 63.9252815246582, 29.74442481994629, -134.37889099121094, -65.67591094970703, 77.32627868652344, 34.60997009277344, -41.154754638671875, -164.9257049560547, 60.8856201171875, 20.005212783813477, -76.96537780761719, -147.080078125, -102.05046844482422, 278.8366394042969, 49.05841827392578, -121.69017791748047, -61.15469741821289, -70.703369140625, 132.70394897460938, 23.45041847229004, -70.76351165771484, -8.485299110412598, -114.92301177978516, 184.21151733398438, 40.8175048828125, 9.626755714416504, 0.6815794706344604, -33.04356002807617, -126.32940673828125, -52.6023063659668, 149.9716339111328, -34.7005729675293, 124.67909240722656, -42.569583892822266, -0.27496495842933655, -144.96502685546875, -122.86673736572266, 5.311367988586426, 94.0286636352539], [35.60803985595703, -155.91981506347656, -105.39717864990234, -78.82444763183594, 129.7969970703125, 106.76435089111328, -150.66502380371094, 29.119091033935547, -135.6190185546875, 162.1295623779297, 117.76757049560547, -125.21527099609375, 138.76393127441406, -113.53807830810547, 76.99540710449219, 1.2603155374526978, -60.93988800048828, -57.716278076171875, -7.538894176483154, 157.11550903320312, 2.4679830074310303, -6.524341583251953, -84.43301391601562, 126.65727996826172, 44.84675598144531, 53.8720703125, -44.90473556518555, -8.707261085510254, -15.886520385742188, -79.15689086914062, 9.815634727478027, -35.60136032104492, -32.41252517700195, 160.90945434570312, -69.89830780029297, 20.051773071289062, 127.05191040039062, -39.0202522277832, -75.39868927001953, 125.5838394165039, 117.29206085205078, -27.722503662109375, -44.35885238647461, -79.2341079711914, 86.345947265625, 3.351820707321167, -58.74604797363281, -17.72865104675293, -153.1836395263672, -23.555004119873047, 41.36960983276367, -68.3386459350586, -34.57191848754883, -76.14436340332031, -107.6960220336914, 165.13037109375, 24.467178344726562, -104.52372741699219, -112.90280151367188, 226.12901306152344, 49.8380012512207, 113.19317626953125, -20.0073299407959, 55.38725280761719], [-279.57025146484375, 87.78309631347656, 25.559627532958984, -85.03614807128906, 112.68254089355469, 56.055419921875, -83.3313980102539, -31.37247085571289, -30.0623836517334, 169.09474182128906, -38.06891632080078, -163.8204803466797, 21.876188278198242, 13.384493827819824, 148.31521606445312, 88.51949310302734, -6.640200138092041, 41.27175521850586, -107.70294952392578, -6.893050670623779, -26.128786087036133, 94.59527587890625, -140.5972900390625, 23.060564041137695, 112.3770751953125, -78.31205749511719, 16.937808990478516, 109.29676055908203, 157.56797790527344, 46.68000793457031, 191.44479370117188, -47.67116928100586, -10.034993171691895, -22.54786491394043, 114.84258270263672, -98.93891906738281, -19.908578872680664, -135.12820434570312, -24.517353057861328, 121.0418701171875, -27.63418197631836, 203.51136779785156, 159.27853393554688, -17.48094367980957, -206.4987335205078, -54.581138610839844, 22.81492805480957, 129.33401489257812, 78.35770416259766, -75.58844757080078, 40.732295989990234, -153.18112182617188, -99.29861450195312, -122.48978424072266, 95.12742614746094, -12.583159446716309, 30.18279457092285, 102.97724914550781, -52.025901794433594, -82.77699279785156, -65.3353271484375, 69.89344024658203, 3.5083441734313965, -17.77271270751953], [36.85332107543945, 32.10445022583008, -52.491397857666016, 6.290146350860596, 53.062286376953125, 15.165017127990723, -104.83741760253906, -108.57968139648438, 24.73048973083496, 12.560208320617676, -8.811222076416016, -113.66754913330078, 172.71356201171875, 60.9253044128418, 27.540008544921875, 58.4326286315918, -81.0396728515625, -79.06966400146484, -98.0817642211914, -50.03614044189453, 16.570110321044922, -57.9970703125, -30.831735610961914, -50.14203643798828, 70.28156280517578, 0.0481160506606102, -33.078853607177734, -110.36201477050781, -203.10984802246094, -74.55465698242188, 21.01670265197754, 58.77064514160156, -88.81652069091797, -16.612606048583984, 97.00179290771484, 29.07033920288086, 49.64484405517578, -149.7311248779297, 12.153066635131836, -45.819759368896484, 65.22196960449219, 51.46784210205078, -105.50039672851562, -27.842491149902344, -23.360836029052734, -28.679290771484375, -12.309030532836914, -75.7066879272461, 88.2070083618164, -158.3080596923828, -187.8563690185547, 166.83416748046875, -72.5739517211914, -34.066505432128906, 212.94827270507812, -61.03343200683594, -24.68433952331543, -67.96956634521484, 47.07960891723633, -15.747892379760742, 141.56591796875, 61.80022048950195, 22.710121154785156, 19.579620361328125], [68.35932159423828, -79.92767333984375, -155.17510986328125, -27.762828826904297, 122.68981170654297, 4.77676248550415, -87.83476257324219, -8.927413940429688, -22.862606048583984, 145.4810791015625, -16.32813262939453, -45.79753494262695, -279.9940490722656, -80.2097396850586, 186.7591094970703, 72.55990600585938, -74.64118194580078, -75.75171661376953, 49.68617630004883, 106.75220489501953, -4.867123603820801, 62.32072067260742, -21.385007858276367, 36.8917121887207, -52.714378356933594, 62.370147705078125, -129.86178588867188, -248.74432373046875, 47.00286102294922, -159.56988525390625, 64.17527770996094, -141.5892333984375, 106.04422760009766, -34.65768814086914, -33.46940612792969, 28.676401138305664, 78.37950897216797, -74.38690948486328, -101.90611267089844, -80.91404724121094, 168.5836944580078, -31.03338050842285, -127.37783813476562, -147.0107421875, -8.810988426208496, 137.55307006835938, 127.92838287353516, -70.19499206542969, -65.95275115966797, 22.20665740966797, -150.42697143554688, 39.59346008300781, -115.57592010498047, -7.75443696975708, -35.521888732910156, -86.24613189697266, -245.80368041992188, 49.971866607666016, 83.27069854736328, 63.916831970214844, -60.58415603637695, 46.940940856933594, -86.72088623046875, 18.63256072998047], [42.15970230102539, 74.02725219726562, -200.8778839111328, -54.667964935302734, 112.48701477050781, 12.942647933959961, -67.78828430175781, 61.53113555908203, -179.1383514404297, 4.548093318939209, -157.6092987060547, -35.71244812011719, -198.41207885742188, -34.90666580200195, -3.5409417152404785, -7.937626361846924, -146.9522705078125, -13.400351524353027, 104.51761627197266, 107.2337417602539, -95.7273178100586, 48.059837341308594, -56.552764892578125, -107.29007720947266, 44.191917419433594, -120.56564331054688, 181.24554443359375, 139.582763671875, -61.869075775146484, -148.68235778808594, 21.383197784423828, -91.87567138671875, -85.16804504394531, -167.65206909179688, 44.27623748779297, -104.94376373291016, -0.434874951839447, 126.0781021118164, 113.6810531616211, -74.728271484375, 75.59066009521484, -37.27824020385742, -0.10080047696828842, -55.10154724121094, 180.1540069580078, -140.4673309326172, -18.75360870361328, 33.03364181518555, 12.536099433898926, -14.95475959777832, 199.04037475585938, -93.15811920166016, -108.34306335449219, -8.755846977233887, 36.514102935791016, -55.908477783203125, -51.192405700683594, 9.418107032775879, -6.025325298309326, -69.14091491699219, 14.407207489013672, 0.08480363339185715, -12.954643249511719, 60.11166763305664], [53.42948532104492, -54.6954231262207, 17.929737091064453, -105.48711395263672, -92.64275360107422, -113.46239471435547, 15.569534301757812, -65.51399993896484, 68.24646759033203, -75.142822265625, -57.98139953613281, 83.18407440185547, 3.8067100048065186, -55.27268600463867, 81.09642791748047, 73.94840240478516, -60.038612365722656, 185.21029663085938, 161.29397583007812, -87.38336944580078, 150.0747528076172, -78.83357238769531, -22.099836349487305, 20.901058197021484, -166.5378875732422, 109.0975341796875, -99.57299041748047, -102.8333740234375, 42.336997985839844, -40.1606330871582, -37.29533386230469, -77.60718536376953, -28.24471092224121, -144.29908752441406, 24.018470764160156, 142.90443420410156, -69.04473876953125, 45.29710388183594, -139.49290466308594, -164.25352478027344, 45.57578659057617, 11.972932815551758, -44.10318374633789, 98.20964050292969, 78.33364868164062, 36.93829345703125, -190.98500061035156, -4.047457695007324, -123.29405975341797, 18.075599670410156, 29.12325668334961, 98.12805938720703, 3.1897809505462646, -16.588214874267578, 238.93031311035156, -14.390055656433105, -60.72499465942383, -52.48969268798828, 152.73867797851562, 17.152488708496094, -36.12068176269531, 258.9538269042969, 112.2601318359375, -49.20642852783203], [-9.378499031066895, 116.3386001586914, 65.07538604736328, -55.915863037109375, 128.6613311767578, 122.78890228271484, 17.544784545898438, -97.2940673828125, -48.02055740356445, 107.71560668945312, 97.85253143310547, 114.96427154541016, -68.55979919433594, -18.47429084777832, -245.28807067871094, 93.56100463867188, 44.94818115234375, -125.51393127441406, -53.464561462402344, -76.61890411376953, 91.2178726196289, -156.01905822753906, -43.13465118408203, -7.006714820861816, -62.43928909301758, 73.54988098144531, 153.3824462890625, 14.474564552307129, -135.10157775878906, -149.35250854492188, -68.43901062011719, -54.464088439941406, -53.183502197265625, -14.30720043182373, -158.88604736328125, -5.387812614440918, -28.78965187072754, 74.9433364868164, -162.74461364746094, -51.92165756225586, 157.41400146484375, 130.12759399414062, -19.2810001373291, 49.487876892089844, -42.69845962524414, 61.03693771362305, -2.8519115447998047, -125.42716217041016, 11.01389217376709, -33.77605056762695, 41.70029067993164, -107.75845336914062, 10.308257102966309, 46.502288818359375, 34.92643356323242, -157.49798583984375, -42.951629638671875, -109.42912292480469, 190.11090087890625, 47.671268463134766, -17.63521957397461, -77.77733612060547, 119.04707336425781, -100.8454818725586], [88.64997863769531, -144.771484375, -92.57418060302734, 163.5582733154297, 10.676243782043457, -163.59637451171875, 115.14459991455078, -104.7385025024414, 118.91145324707031, -323.7767028808594, -207.12069702148438, -154.12081909179688, 8.48454761505127, 8.545496940612793, -54.28235626220703, -26.454021453857422, -63.65336990356445, -27.249792098999023, 43.55380630493164, -5.505056858062744, -88.10364532470703, -206.95518493652344, -189.00396728515625, -16.88947105407715, -19.8583984375, -69.80027770996094, 148.76158142089844, -16.260229110717773, 19.40016746520996, -69.20899200439453, -34.110591888427734, 1.9742882251739502, 5.24495792388916, 4.30042028427124, 10.372416496276855, 69.93186950683594, -6.8989763259887695, 88.48560333251953, -41.82355499267578, -59.375328063964844, 199.42503356933594, 105.74311828613281, 48.15127182006836, 6.1593780517578125, -132.7830810546875, 66.81288146972656, -238.4022674560547, 7.686369895935059, -104.6252212524414, -113.63802337646484, -61.91651916503906, -81.8705062866211, 33.84480285644531, -117.11466217041016, 27.250858306884766, -90.79279327392578, -72.34688568115234, -76.45162963867188, 53.17587661743164, 309.9343566894531, 74.8552017211914, -49.771034240722656, -75.64017486572266, -85.2352066040039], [109.0186767578125, -20.346891403198242, 73.00035095214844, -125.19602966308594, 154.84642028808594, -54.70146560668945, -108.00225067138672, 18.64404296875, -114.79904174804688, -93.76054382324219, 133.57022094726562, 87.04685974121094, -166.7784881591797, 123.75277709960938, 56.98663330078125, 46.078643798828125, 176.84193420410156, -219.87637329101562, -14.000696182250977, -135.31126403808594, 199.1078643798828, 49.0928840637207, -172.7666473388672, 73.87683868408203, -103.21940612792969, -5.490719318389893, -112.66449737548828, 171.4290771484375, -24.79062843322754, -96.49279022216797, 51.578128814697266, -2.8873817920684814, -9.43729019165039, -14.454717636108398, -121.32581329345703, -59.93077850341797, 135.6748504638672, -233.3955535888672, -33.685096740722656, 15.142892837524414, 182.9546356201172, 76.87389373779297, 3.919170618057251, 161.8385772705078, 100.73953247070312, -115.7837142944336, 77.12239837646484, -23.43571662902832, 45.317684173583984, 70.65847778320312, -51.80790328979492, 144.59371948242188, 111.6729965209961, -110.57217407226562, -136.96890258789062, -142.9143524169922, -113.7329330444336, 81.1648941040039, -222.5485382080078, 13.118722915649414, 23.605253219604492, 64.72755432128906, -135.04808044433594, 199.89468383789062], [-45.109046936035156, -88.02947235107422, 85.6445541381836, -64.57801055908203, 152.19277954101562, -21.35232162475586, 36.220943450927734, -59.70315933227539, 91.11558532714844, 4.996575832366943, 44.930606842041016, -141.40472412109375, 246.390625, -31.609222412109375, -138.20628356933594, 87.53998565673828, 101.98241424560547, -184.15272521972656, -61.085628509521484, -113.12139129638672, 59.34357452392578, -136.98138427734375, -81.84651184082031, -61.724517822265625, -27.48334503173828, 16.459426879882812, -31.441749572753906, -46.502017974853516, 67.70134735107422, -77.75970458984375, 52.18513107299805, -22.152618408203125, 7.812197208404541, -114.73371887207031, -57.254600524902344, -23.020341873168945, 11.513693809509277, -264.6107177734375, 170.32858276367188, 123.7757797241211, 18.1726016998291, -6.800657749176025, 24.647729873657227, -69.33737182617188, -13.909878730773926, 170.49786376953125, -159.1605682373047, 32.26063537597656, -3.8072497844696045, -90.30250549316406, -4.078497886657715, 144.2289276123047, -174.82469177246094, -266.63580322265625, -11.294002532958984, 0.08182913064956665, -79.66923522949219, -73.31436157226562, 61.00746154785156, -293.1211242675781, -101.40641021728516, 48.49917221069336, -91.33708953857422, -63.60654067993164], [-69.84519958496094, -84.05902862548828, -46.009952545166016, -28.601200103759766, 41.9476432800293, 114.44175720214844, 40.502559661865234, -59.472999572753906, -8.367819786071777, 127.99038696289062, -46.38677215576172, -25.67980194091797, -107.67112731933594, 92.68479919433594, 57.01405334472656, -64.9228515625, 54.37461853027344, 118.4192123413086, -9.51458740234375, 48.89712905883789, -48.44241714477539, -27.950950622558594, -44.241416931152344, 122.2723388671875, 80.65911102294922, 64.68701934814453, -11.945952415466309, 171.9307098388672, 34.32246398925781, -90.70404052734375, -42.061580657958984, 33.21128463745117, -90.58020782470703, 185.64393615722656, -31.19542694091797, -70.8136215209961, -7.916948318481445, -42.85576248168945, -38.060394287109375, 78.15931701660156, -65.28663635253906, 128.93519592285156, -0.8732197880744934, 140.28213500976562, -103.37145233154297, 6.586687088012695, -87.50643157958984, -133.6424560546875, -193.7783966064453, -99.69759368896484, 46.98419189453125, 97.61264038085938, 27.289472579956055, -72.62671661376953, -18.839675903320312, 20.265335083007812, -54.066287994384766, 77.99783325195312, 91.17342376708984, -35.29967498779297, 2.760939836502075, -73.19861602783203, 114.49436950683594, -16.86282730102539], [252.31871032714844, -139.86061096191406, 212.01931762695312, 7.642437934875488, -150.2602996826172, -107.81547546386719, 107.96717834472656, -15.889350891113281, 115.6956787109375, -128.6464385986328, 20.060333251953125, -55.78980255126953, 2.4902145862579346, -21.73192596435547, 60.831016540527344, 28.099470138549805, -57.68177032470703, 163.83627319335938, -158.10885620117188, 36.116554260253906, -37.117305755615234, -148.2384796142578, -7.5941572189331055, -160.63194274902344, -102.6073226928711, -57.129112243652344, 39.54594421386719, 96.60204315185547, 107.49907684326172, 112.68769073486328, -153.16708374023438, -67.50971221923828, 132.615234375, 121.73542785644531, -93.04794311523438, 18.86290740966797, -152.409912109375, -30.048662185668945, 130.90525817871094, -2.3666739463806152, -118.2016830444336, 181.54171752929688, 16.999332427978516, 18.49287986755371, 61.72850036621094, 69.92366790771484, 136.8367156982422, -37.89124298095703, -56.65182876586914, -54.674049377441406, 208.019287109375, 148.6451416015625, 115.8671875, -137.6922607421875, -89.28756713867188, -10.369858741760254, -55.9517707824707, -173.70870971679688, 62.941688537597656, -72.8954086303711, -48.521690368652344, 228.0736541748047, -47.212745666503906, -110.21404266357422], [-187.3555908203125, -18.835695266723633, -80.65804290771484, 60.29814529418945, -62.141387939453125, -14.529223442077637, 113.50599670410156, 127.3424301147461, 71.71232604980469, 26.25827980041504, -65.54159545898438, 37.881649017333984, 4.274279594421387, 31.369651794433594, -116.29682159423828, -148.69459533691406, 137.33462524414062, 86.47752380371094, 38.73487091064453, -6.036819934844971, 150.3499755859375, -163.20318603515625, -68.3486557006836, -148.28712463378906, 24.741519927978516, -114.11372375488281, 154.83949279785156, 89.19617462158203, 33.54503631591797, 108.86138153076172, -11.940180778503418, 186.2377166748047, -130.34304809570312, -89.30081176757812, -39.79048156738281, 139.7104034423828, 58.034969329833984, 47.608116149902344, 92.21664428710938, -212.98341369628906, 78.81287384033203, 20.342060089111328, 2.0802085399627686, -109.5418701171875, -56.07576370239258, -374.75738525390625, -24.122743606567383, -114.36563110351562, -23.30735206604004, -101.96491241455078, -54.3633918762207, -37.46389389038086, -79.18692779541016, 70.7391586303711, -211.54579162597656, -6.827976226806641, 88.18268585205078, 7.122122287750244, 165.1964111328125, 134.41407775878906, -28.875808715820312, 63.839637756347656, 32.17559051513672, 0.5438871383666992]], "dtype": "float32"}], "e": {"__nd__": [[-0.006887972354888916, 0.015161437913775444, -0.003100855275988579, -0.011182555928826332, -0.005380283575505018, -0.0071984692476689816, 0.00019867335504386574, 0.03184257075190544, -0.003018777584657073, 0.03853059187531471, 0.013273637741804123, 0.009907923638820648, -0.024701224640011787, -0.0169049222022295, -0.01054401695728302, 0.019825348630547523, -0.025340348482131958, -0.012798668816685677, -0.022733572870492935, 0.02217090129852295, -0.031935080885887146, 0.010998151265084743, 0.008956407196819782, 0.00516220647841692, 0.026180021464824677, 0.011721639893949032, -0.027938971295952797, 0.015785235911607742, -0.010313010774552822, 0.013920916244387627, -0.012897633016109467, -0.00665470864623785, -0.03255137428641319, -0.007807628717273474, -0.026743825525045395, 0.003368749050423503, 0.004289313219487667, 0.03458065539598465, -0.022176243364810944, 0.011222206987440586, -0.0004085726977791637, -0.0163764376193285, 0.04595327004790306, -0.03296589478850365, -0.013147292658686638, 0.017472192645072937, 0.003969910554587841, 0.005209128372371197, 0.01329898927360773, 0.025222020223736763, -0.04066610336303711, 0.007324906066060066, 0.0004967048298567533, 0.014941466972231865, 0.029152536764740944, 0.0076924399472773075, -0.0021925221662968397, 0.011381655000150204, -0.03115866705775261, 0.017432821914553642, 0.013128158636391163, -0.016593990847468376, -0.015449874103069305, -0.00600031204521656], [0.013337445445358753, 0.039839234203100204, 0.05411054939031601, -0.016625436022877693, -0.009399541653692722, -0.00955549068748951, 0.020158560946583748, 0.0025565007235854864, -0.029855363070964813, 0.0032011589501053095, -0.03204866126179695, 0.015013489872217178, -0.0001969986769836396, 0.02797163650393486, 0.009016549214720726, -0.016633978113532066, -0.02170390449464321, -0.02679203264415264, -0.006800553761422634, 0.04101705923676491, 0.03408325836062431, -0.018915627151727676, -0.024163678288459778, 0.03820575773715973, -0.013618849217891693, 0.001676838961429894, 0.0017506874864920974, -0.010883873328566551, 0.02091437578201294, 0.017713332548737526, 0.0072660893201828, -0.03039291314780712, -0.031061574816703796, 0.005833102855831385, -0.009920376352965832, 0.0201888307929039, -0.011090251617133617, -0.019219135865569115, -0.016711734235286713, 0.055132266134023666, -0.011129461228847504, -0.010785353370010853, -0.020931674167513847, -0.004222371149808168, 0.004079077858477831, 0.03250490128993988, 0.021040719002485275, -0.010787472128868103, 9.655870962888002e-05, 0.0036031105555593967, 0.003487833309918642, 0.023208634927868843, 0.030770808458328247, 0.02751092053949833, 0.01580420322716236, 0.00450903968885541, 0.004190480336546898, -0.0039430055767297745, -0.0002683226193767041, -0.002605847781524062, 0.004324472043663263, -0.005444976035505533, 0.01505074743181467, -0.03903285413980484], [0.014353306032717228, 0.02236843667924404, 0.007412605918943882, -0.012532094493508339, -0.02811157889664173, -0.004256035201251507, 0.020459134131669998, 0.0019877227023243904, -0.010978177189826965, 0.014923621900379658, 0.006637015845626593, -0.012622526846826077, -0.016786111518740654, 0.02119571529328823, 0.025486309081315994, 0.005044401157647371, 0.02654133550822735, 0.003360177157446742, 0.01643018238246441, -0.025578700006008148, -0.0111208725720644, -0.031137777492403984, -0.01216510497033596, -0.025406206026673317, 0.027687210589647293, 0.002304556081071496, 0.026376161724328995, -0.0056606559082865715, 0.018589375540614128, -0.02743220329284668, 0.016255004331469536, -0.005669052712619305, 0.01681770198047161, 0.006630214862525463, -0.04533553868532181, 0.051149316132068634, -0.024058593437075615, -0.0264313705265522, -0.037120573222637177, -0.007679338566958904, -0.027926059439778328, -0.01925794780254364, -3.639063652371988e-05, -0.013003612868487835, -0.015648534521460533, -0.0074630677700042725, -0.04063684120774269, 0.004850583616644144, 0.028152218088507652, -0.03633233532309532, 0.005602411460131407, 0.005989680998027325, 0.02144470624625683, -0.029171844944357872, 0.01862192153930664, 0.010414481163024902, -0.045740045607089996, -0.023054255172610283, -0.005026008002460003, -0.0004684029845520854, -0.011102215386927128, -0.0276703629642725, 0.009713313542306423, 0.02931530587375164], [-0.024352772161364555, -0.03754438832402229, 0.010074013844132423, 0.014656360261142254, -0.008190354332327843, 0.020340627059340477, 0.0065330457873642445, 0.033174656331539154, 0.018791627138853073, -0.00293852505274117, 0.008712125942111015, -0.019266996532678604, 0.014986315742135048, -0.01118297129869461, 0.023638306185603142, -0.008108207024633884, -0.027119670063257217, 0.02025124430656433, 0.007847921922802925, 0.0162359569221735, 0.027958696708083153, -0.016494378447532654, -0.01454951986670494, 0.0015663449885323644, -0.016248485073447227, -0.04062897339463234, 0.006101395469158888, 0.019993457943201065, -0.012692396529018879, 0.01953854039311409, 0.007726132404059172, 0.012548455968499184, -0.007001932244747877, -0.0001936193584697321, 0.02157524973154068, 0.04101153835654259, 0.032336022704839706, -0.008705724030733109, 0.0009319919045083225, 0.011660016141831875, 0.02214241586625576, -0.0018323768163099885, 0.00939162541180849, -0.033802229911088943, -0.006505538243800402, 0.015616508200764656, 0.010864830575883389, 0.04103939235210419, -0.020083624869585037, -0.020121674984693527, -0.026683738455176353, 0.024958472698926926, -0.011497038416564465, -0.013461586087942123, -0.008536005392670631, 0.0010750292567536235, 0.002750853542238474, 0.013048173859715462, -0.029132366180419922, -0.0004220147675368935, -0.0010990571463480592, -0.039352238178253174, -0.02067992091178894, 0.01027679443359375], [0.00763196824118495, -0.012916762381792068, -0.00808153860270977, 0.006896840408444405, -0.03532114624977112, 0.019585255533456802, 0.02932758443057537, -0.005688750185072422, 0.004850979428738356, 0.022176967933773994, -0.009928977116942406, 0.012573115527629852, 0.0036475874949246645, 0.015689658001065254, -0.0040838164277374744, 0.017487511038780212, 0.008726388216018677, 0.0032758417073637247, -0.009353325702250004, 0.02542855776846409, 0.00045075471280142665, 0.01975342072546482, -0.029493795707821846, -0.0016667203744873405, 0.004022820852696896, 0.03669579327106476, 0.019484560936689377, 0.008068750612437725, -0.009756057523190975, -0.018841572105884552, -0.007381356321275234, 0.009777721017599106, -0.011692969128489494, -0.03455149009823799, -0.021357106044888496, 0.003545989515259862, -0.014270448125898838, 0.011899608187377453, -0.02339235693216324, -0.01159929484128952, 0.029071444645524025, -0.008361873216927052, 0.037494100630283356, -0.014124534092843533, 0.04736045002937317, 0.029388923197984695, -0.014845719560980797, 0.00888790376484394, -0.03084598109126091, -0.01938602514564991, 0.00878194347023964, 0.010113291442394257, -0.0040865615010261536, 0.019208533689379692, 0.005776026286184788, 0.03378262370824814, -0.010153660550713539, 0.0066928742453455925, -0.014334098435938358, 0.0501861535012722, 0.019278137013316154, 0.003092288039624691, 0.020163338631391525, 0.02405647374689579], [0.011636822484433651, -0.033207379281520844, -0.018236076459288597, 0.01867944560945034, 0.002044931286945939, -0.008533233776688576, -0.010084436275064945, -0.0036844685673713684, 0.013396884314715862, -0.021133361384272575, -0.029497023671865463, -0.009045387618243694, -0.002933851443231106, -0.011012870818376541, 0.019687902182340622, -0.019766265526413918, -0.007064183708280325, 0.004460611846297979, -0.018197299912571907, 0.025642110034823418, -0.022971080616116524, -0.008630935102701187, 0.0033634821884334087, 0.021182909607887268, -0.03200041130185127, -0.007669670972973108, -0.01830671727657318, -0.012105201371014118, -0.006382267456501722, 0.01174953393638134, -0.010297787375748158, 0.004725048318505287, 0.03724752366542816, 0.02278193086385727, -0.0064417715184390545, 0.019918780773878098, 0.0025349496863782406, 0.02642580308020115, -0.0007553539471700788, 0.0007862939964979887, -0.013156791217625141, -0.0165163055062294, 0.026588264852762222, 0.04653812199831009, -0.000369735702406615, 0.0028961312491446733, 0.016086531803011894, 0.035120997577905655, 0.0029577377717942, 0.007337456103414297, 0.006886903662234545, -0.0010989473666995764, -0.03296848386526108, -0.009567798115313053, -0.0014345234958454967, 0.023215055465698242, -0.0059015098959207535, 0.02624620869755745, 0.011056233197450638, -0.011232908815145493, 0.01765197142958641, 0.0024027032777667046, -0.00606991071254015, 0.011826971545815468], [0.0063476539216935635, -0.011433016508817673, 0.006557452026754618, 0.009130094200372696, -0.013386222533881664, 6.651716103078797e-05, -0.012350145727396011, -0.01708161272108555, -0.009662767872214317, -0.005913896951824427, -0.019412096589803696, -0.01160457357764244, 0.0017100400291383266, 0.004606144968420267, 0.0029788522515445948, -0.015797004103660583, 0.007459364831447601, 0.02052544802427292, -0.034612368792295456, 0.024975955486297607, -0.004452292807400227, 0.015443105250597, -0.0194183811545372, 0.054980985820293427, 0.0023181019350886345, 0.017242101952433586, 0.017901873216032982, -0.0035865057725459337, 0.04248295724391937, 0.019674858078360558, 0.006302749738097191, 0.04251335933804512, 0.006209108047187328, -0.022375980392098427, 0.023870743811130524, 0.004692957270890474, 0.004191173240542412, 0.011369277723133564, 0.01760062389075756, 0.0076655312441289425, -0.0008637676364742219, -0.0024696779437363148, -0.023478977382183075, 0.00165294308681041, 0.010682187043130398, -0.01390515174716711, 0.01589423231780529, 0.00506176333874464, 0.019732849672436714, 0.006243920419365168, 0.007646005135029554, -0.008330469951033592, 0.051361728459596634, 0.0017927290173247457, -0.01191307045519352, -0.03345853090286255, 0.014830791391432285, 0.01980924978852272, -0.028527025133371353, -0.022466301918029785, -0.013744043186306953, -0.010842411778867245, -0.01562001183629036, 0.002305068541318178], [-0.005346156191080809, 0.02311297506093979, -0.0058385515585541725, -0.021422578021883965, 0.0010597510263323784, -0.033130139112472534, 0.01810339093208313, -0.0016657246742397547, 0.005474681034684181, -0.021700456738471985, -0.015642806887626648, 0.025897588580846786, 0.04211647808551788, 0.018823865801095963, -0.025421710684895515, 0.02107340097427368, -0.014449321664869785, 0.022303743287920952, -0.03568567708134651, 0.0196754839271307, -0.02967117354273796, -0.0007838660385459661, -0.011184829287230968, 0.019325999543070793, -0.008689428679645061, -0.00982070155441761, -0.03495163843035698, 0.0058828662149608135, -0.011335221119225025, -0.0020860019139945507, 0.02898580953478813, 0.010193352587521076, 0.033981628715991974, -0.029126428067684174, 0.009226245805621147, 0.019943436607718468, -0.018820080906152725, -0.010490914806723595, -0.003077472560107708, 0.01352638378739357, 0.03149064630270004, -0.013760185800492764, 0.02772696688771248, 0.00773787684738636, 0.03899766504764557, -0.0071005928330123425, 0.01249891147017479, 0.02647823467850685, 0.012296438217163086, 0.00020135859085712582, -0.01648501679301262, 0.033112529665231705, 0.020092269405722618, -0.016133900731801987, 0.004643960855901241, -0.03749927133321762, 0.02816310152411461, 0.030354315415024757, -0.041479580104351044, 0.006629531737416983, 0.005264081992208958, 0.0075088548474013805, 0.04794446378946304, -0.01072036661207676], [-0.009068933315575123, -0.011846322566270828, 0.01279773935675621, -0.018254943192005157, -0.03561949357390404, 0.003980771638453007, 0.03762480616569519, -0.019054219126701355, -0.007691121194511652, -0.004355734679847956, -0.016924774274230003, -0.032914720475673676, 0.025527721270918846, 0.009792794473469257, 0.007872785441577435, -0.021849988028407097, -0.008573545143008232, 0.022338667884469032, 0.0308028906583786, 0.00741752702742815, -0.03920288756489754, -0.0008763338555581868, 0.006193241570144892, -0.0014359384076669812, -0.009198155254125595, -0.013730049133300781, 0.017938708886504173, -0.00012078385771019384, 0.0010962738888338208, -0.0021156014408916235, -0.027669711038470268, -0.038190510123968124, 0.01976597309112549, 0.027262190356850624, 0.05650743842124939, -0.01181053090840578, -0.00604766421020031, 0.022378262132406235, -0.0008431507740169764, -0.010829465463757515, 0.011571671813726425, 0.009771887212991714, -0.02150469645857811, -0.021494310349225998, -0.030544064939022064, -0.04877255856990814, -0.02060362882912159, 0.013057639822363853, -0.010606705211102962, -0.03664495423436165, -0.015034499578177929, 0.02027197927236557, -0.0065882508642971516, -0.01881713606417179, -0.0013883732026442885, -0.021737001836299896, -0.008556068874895573, 0.0003833285591099411, -0.06018226593732834, -0.008097114972770214, -0.01843435876071453, -0.024049796164035797, -0.035035088658332825, -0.0005319913034327328], [0.015367373824119568, -0.005635254085063934, 0.008152369409799576, 0.04581555724143982, 0.0099417082965374, 0.0017970630433410406, -0.03266952931880951, -0.0007106110570020974, 0.004554872866719961, 0.011636831797659397, -0.029131487011909485, -0.01108772587031126, 0.006539373192936182, -0.0021767436992377043, -0.007455718703567982, 0.0003348235331941396, 0.019291477277874947, -0.01998637430369854, -0.01590876281261444, -0.013144751079380512, -0.00900128111243248, -0.010684370063245296, 0.015779471024870872, -0.0057000769302248955, -0.0014648144133388996, -0.018182098865509033, -0.025642942637205124, -0.020003294572234154, -0.010561298578977585, 0.008584432303905487, 0.0023609460331499577, -0.0004047744150739163, -0.02300276793539524, 0.007260001264512539, -0.024426734074950218, -0.010587596334517002, 0.02095000445842743, 0.011520175263285637, -0.006856159307062626, -0.01879892870783806, 0.013549257069826126, -0.029850555583834648, -0.00034450163366273046, 0.020460516214370728, 0.013691321946680546, 0.00924715492874384, 0.008353685028851032, 0.00595766631886363, 0.023120379075407982, -0.017807550728321075, -0.030526550486683846, -0.024203598499298096, -0.02128106728196144, -0.0004245471500325948, 0.012791663408279419, -0.0002785447577480227, -0.010064597241580486, -0.008278249762952328, -0.04748432710766792, -0.02436787821352482, -0.01205626130104065, -0.006587535608559847, -0.01312729436904192, -0.011467067524790764], [0.013443363830447197, -0.009433350525796413, -0.012077837251126766, -0.0268823504447937, 0.00813620537519455, -0.015687715262174606, 0.004139438737183809, 0.017675161361694336, -0.03622164577245712, 0.0056487261317670345, -0.012188076041638851, -0.03039766661822796, 0.009517548605799675, 0.023715823888778687, -0.00292942114174366, -0.014322236180305481, -0.02036203444004059, -0.014219718053936958, -0.01136812288314104, 0.007420800160616636, -0.013683445751667023, -0.005347183905541897, -0.0035746677312999964, -0.015386106446385384, -0.004468626342713833, 0.017412984743714333, -0.04316331446170807, -0.0020176819525659084, 0.0252591073513031, 0.037748947739601135, 0.0019863059278577566, 0.01282756868749857, -0.027549942955374718, 0.00894839409738779, -0.001398835564032197, -0.016062255948781967, 0.03305421024560928, -0.015786483883857727, 0.010000825859606266, 0.0022737118415534496, -0.025478048250079155, -0.03297465667128563, 0.026868686079978943, 0.0030278414487838745, -0.05138101428747177, 0.00700372364372015, -0.002897045109421015, 0.040560297667980194, -0.0009004003368318081, -0.010202259756624699, 0.007216351106762886, -0.010938734747469425, -0.005932465195655823, -0.00610359525308013, 0.008862623013556004, -0.004915202036499977, -0.007952036336064339, -0.0024486160837113857, -0.008029391057789326, 0.007035952992737293, 0.017068978399038315, 0.006110140588134527, -0.004710618406534195, -0.05617943033576012], [0.012410877272486687, -0.030966266989707947, -0.0009691906161606312, -0.01916794665157795, -0.013128632679581642, -0.001618802547454834, 0.01891515962779522, 0.00374607159756124, -0.023648576810956, -0.020817667245864868, 0.01839056983590126, 0.006816444918513298, -0.008743888698518276, 0.02475094422698021, 0.0009586568339727819, 0.00038067856803536415, -0.007579378318041563, -0.036924805492162704, 0.025961190462112427, 0.006213079672306776, -0.002965094754472375, -0.0004902175278402865, -0.0012586233206093311, -0.013597901910543442, -0.009210766293108463, 0.008939559571444988, -0.004472597036510706, 0.0017600484425202012, 0.00018523696053307503, -0.008681061677634716, -0.024624694138765335, 0.028852662071585655, -0.028439104557037354, -0.01448755245655775, -0.02472234144806862, 0.0075347647070884705, 0.03055519610643387, 0.02931230701506138, -0.025466950610280037, -0.008795814588665962, -0.015174773521721363, -0.0009437124826945364, -0.016646448522806168, -0.02415168657898903, 0.018916837871074677, -0.029665589332580566, 0.018462596461176872, -0.016341207548975945, 0.0032701229210942984, -0.0025436023715883493, 0.022113734856247902, 0.009223301894962788, -0.014345560222864151, -0.021885791793465614, 0.025937212631106377, 0.013609103858470917, 0.012703297659754753, -0.020230352878570557, 0.006938566919416189, -0.006172227673232555, -0.01272539235651493, 0.011746550910174847, 0.023156559094786644, -0.02662217803299427], [0.013087532483041286, -0.006467379629611969, 0.005221604369580746, -0.002078624442219734, 0.019197849556803703, -0.010866622440516949, 0.038298118859529495, 0.02319217659533024, -0.0005522901192307472, -0.023043043911457062, 0.02150399424135685, 0.020446952432394028, -0.051974497735500336, 0.011104130186140537, 0.006729142274707556, 0.03795674443244934, 0.030800919979810715, -0.025041677057743073, 0.004895652178674936, 0.003219975158572197, 0.019688980653882027, -0.039915312081575394, -0.008022274821996689, -0.006380731705576181, -0.04387529939413071, 0.010892536491155624, -0.0008030376629903913, 0.0092320516705513, 0.014005743898451328, -0.009062053635716438, -0.01913410611450672, -0.0037874996196478605, 0.0040691751055419445, -0.007795892655849457, -0.013399062678217888, -0.039486322551965714, -0.013692735694348812, 0.02901037223637104, 0.03250855579972267, 0.009555821307003498, 0.004236772656440735, -0.01820797473192215, -0.021818438544869423, -0.016565725207328796, -0.02670230343937874, -0.009814941324293613, 0.0023122462444007397, -0.02234901860356331, 0.00683119660243392, 0.0022446932271122932, 0.004153854679316282, 0.024892177432775497, 0.01273907907307148, -0.02706986851990223, 0.0012922938913106918, -0.014210645109415054, -0.008528092876076698, -0.01496944110840559, -0.037498198449611664, 0.0188139695674181, -0.016990849748253822, 0.03520064055919647, 0.03295164555311203, -0.017396166920661926], [-0.032881829887628555, 0.007935792207717896, -0.001894004293717444, 0.03638112172484398, 0.005683103576302528, 0.004874594043940306, -0.01537875272333622, 0.0057489220052957535, -0.023845937103033066, -0.019938185811042786, 0.011836422607302666, -0.02087554708123207, 0.01728121191263199, 0.03345631808042526, 0.0005869432352483273, -0.010287415236234665, 0.021531159058213234, 0.01374768652021885, -0.005532932002097368, 0.014212899841368198, 0.016096269711852074, -0.016269171610474586, -0.03999076411128044, -0.0022694256622344255, 0.028269577771425247, 0.03960570693016052, -0.017347635701298714, 0.023198068141937256, 0.03279958665370941, -0.011426061391830444, -0.021191498264670372, 0.0019339729333296418, 0.022293290123343468, -0.012666977010667324, 0.03986569494009018, -0.011891975067555904, -0.02327084355056286, 0.015077655203640461, -0.00017062490223906934, 0.0215271208435297, -0.021356889978051186, -0.009874651208519936, -0.0011418104404583573, -0.00698638753965497, -0.0035430630668997765, -0.013997182250022888, 0.0015236444305628538, 0.020407462492585182, 0.008451486006379128, -0.028188029304146767, 0.0006067748763598502, -0.0009642766672186553, 0.003191798459738493, 0.008843282237648964, 0.011247162707149982, -0.011281746439635754, -0.02259935438632965, 0.010544966906309128, -0.008604306727647781, -0.00018300421652384102, -0.0036141490563750267, -0.022481586784124374, -0.0022884944919496775, -0.026326792314648628], [-0.005432365462183952, -0.00021132851543370634, -0.013577194884419441, -0.02025163546204567, -0.025683753192424774, 0.00624163169413805, -0.016313137486577034, 0.04125143215060234, -0.03285181522369385, -0.004263581708073616, 0.013952524401247501, -0.02419171668589115, -0.012611093930900097, -0.03187957778573036, -0.010272237472236156, -0.009856125339865685, 0.030017100274562836, -0.020752577111124992, 0.010921499691903591, 0.03822147101163864, -0.011029127053916454, -0.01934204064309597, 0.04319275915622711, -0.008613268844783306, 0.03946559131145477, 0.002967025386169553, 0.005117108579725027, 0.019423382356762886, 0.018518995493650436, -0.026250865310430527, -0.012369331903755665, -0.018218163400888443, 0.0102699501439929, -0.023719392716884613, 0.024917716160416603, 0.014663927257061005, 0.004355112090706825, -0.011513822712004185, 0.01156125869601965, -0.0157054141163826, 0.04248090088367462, 0.01724771223962307, -0.03326087445020676, 0.011560678482055664, 0.02821287512779236, 0.0029174848459661007, 0.04605168104171753, -0.01419331505894661, 0.016989469528198242, -0.015277300029993057, 0.004096118733286858, 0.01791691780090332, -0.00828880351036787, -0.01578339748084545, 0.015606553293764591, 0.02136291190981865, 0.003023670520633459, 0.025198467075824738, -0.03548186272382736, -0.030573878437280655, 0.027222858741879463, -0.006333815399557352, 0.004168321844190359, 0.01437183003872633], [-0.015261044725775719, -0.029752381145954132, 0.02089253067970276, 0.010469210334122181, 0.006423361599445343, -0.003811874892562628, -0.0069411639124155045, -0.012280747294425964, -0.002734011271968484, 0.03149394318461418, -0.0066640120930969715, 0.0004673622315749526, -0.009486274793744087, 0.03765241056680679, 0.026864750310778618, -0.021573906764388084, -0.0032293375115841627, -0.024850759655237198, 0.003627450205385685, -0.017287924885749817, 0.006577111780643463, 0.02377062849700451, -0.017876090481877327, -0.02007000520825386, 0.0008974661468528211, 0.00737481564283371, 0.010674828663468361, 0.012512906454503536, 0.011909056454896927, -0.007475222926586866, 0.0033050719648599625, 0.005926544312387705, 0.010707256384193897, 0.012063981965184212, -0.016678007319569588, -0.02756713517010212, 0.000499847752507776, -0.007033478934317827, -0.003115415573120117, 0.0031235276255756617, -0.01281362771987915, 0.019388847053050995, 0.02211974933743477, -0.05332744121551514, -0.010472760535776615, 0.04654718190431595, -0.027008838951587677, 0.03312891721725464, -0.03000221587717533, -0.01342973206192255, -0.009306606836616993, 0.020151309669017792, 0.0032724710181355476, -0.019206224009394646, 0.006479391362518072, 0.00423639127984643, -0.049954429268836975, -0.012945118360221386, -0.0017887295689433813, -0.028444724157452583, -0.013176485896110535, -0.018034666776657104, -0.028589272871613503, 0.014120453037321568], [0.0267967376857996, 0.003237913129851222, -0.017893997952342033, 0.010203937068581581, 0.018286950886249542, 0.006486512720584869, 0.022111305966973305, 0.0364617295563221, -0.01712920144200325, 0.01559815276414156, 0.02776249311864376, 0.025951793417334557, 0.008112908340990543, 0.019670773297548294, 0.030586978420615196, -0.013129591010510921, 0.010400260798633099, 0.010755589231848717, 0.005188618786633015, -0.027281297370791435, -0.012209637090563774, -0.009734868071973324, -0.008653197437524796, 0.007570316083729267, -0.02164163812994957, -0.034775689244270325, -0.02796400524675846, -0.009998408146202564, -0.01662958785891533, 0.014924533665180206, 0.01344348769634962, -0.023379046469926834, 0.018240420147776604, -0.009109925478696823, -0.027713866904377937, -0.005601357668638229, 0.0007202596752904356, 0.008446702733635902, -0.012863896787166595, -0.02253221906721592, -0.017387421801686287, 0.029876619577407837, -0.006744156125932932, -0.009351911023259163, -0.00022095479653216898, 0.015792962163686752, 0.010309409350156784, 0.013788215816020966, -0.01854126714169979, -0.00986539851874113, 0.02681053802371025, -0.023569058626890182, -0.009199006482958794, -0.006390823051333427, 0.01829371601343155, 0.051204800605773926, 0.036802034825086594, -0.015125147067010403, 0.010689394548535347, 0.008188800886273384, -0.022013090550899506, -0.00442101014778018, 0.00027215658337809145, 0.019025763496756554], [0.031880058348178864, -0.013782159425318241, -0.002464127726852894, 0.003087069606408477, -0.00796682108193636, -0.04948381707072258, -0.031711552292108536, 0.004243568982928991, -0.018983542919158936, 0.018802491948008537, 0.024883000180125237, 0.006314055062830448, 0.007458309642970562, -0.0009573642746545374, 0.011609180830419064, -0.008416040800511837, -0.008573166094720364, 0.05005120858550072, 0.0053255315870046616, -0.017869533970952034, 0.0032306620851159096, -0.029484042897820473, -0.005001329351216555, -0.00598107697442174, -0.010156295262277126, 0.03868834301829338, -0.0025300593115389347, -0.002268069889396429, 0.00462311552837491, -0.03373618423938751, -0.03305678814649582, 0.002617787104099989, 0.015671437606215477, 0.028745347633957863, 0.010634337551891804, -0.009719885885715485, 0.0011852594325318933, -0.026995548978447914, -0.004958742763847113, -0.018725909292697906, 0.0008717278833501041, 0.0014517874224111438, -0.011385065503418446, 0.01905047707259655, 0.015331503003835678, 0.04606131091713905, -0.02479461021721363, -0.01714208535850048, 0.03096202202141285, 0.021732302382588387, -0.029471421614289284, -0.020327918231487274, -0.026779167354106903, -0.013742806389927864, 0.010117005556821823, 0.0015481810551136732, 0.030209846794605255, -0.01604934222996235, 0.015661653131246567, -0.014844922348856926, 0.014627552591264248, 0.032549817115068436, -0.01033864077180624, 0.039463870227336884], [-0.011344046331942081, -0.01851358264684677, -0.015250889584422112, -0.025974703952670097, 0.00327823287807405, -0.01662086509168148, -0.011077637784183025, -0.0011400798102840781, 0.002186094643548131, -0.001251033740118146, -0.03710708022117615, -0.005098155699670315, -0.006623611785471439, -0.017164863646030426, -0.031000329181551933, 0.023806821554899216, -0.012751176953315735, 0.0028239283710718155, 0.0107891159132123, -0.01984218880534172, -0.027491526678204536, -0.005693695042282343, -0.006531532388180494, -0.00196024845354259, -0.001954695675522089, 0.010099833831191063, -0.011959798634052277, -0.024353252723813057, -0.004826353397220373, 2.7208172468817793e-05, 0.002151777036488056, -0.009097986854612827, -0.017138825729489326, 0.008709791116416454, -0.021113505586981773, -0.020368242636322975, 0.018337270244956017, 0.005559221841394901, -0.016013534739613533, -0.0038926294073462486, 0.003574720583856106, -0.007583042606711388, -0.013420319184660912, -0.01585371419787407, -0.016232743859291077, -0.004912796430289745, 0.0011295032454654574, 0.0154979657381773, 0.004969908855855465, -0.005650477018207312, 0.05584333837032318, 0.027800343930721283, -0.002364768646657467, -0.019446317106485367, -0.024437634274363518, 0.028386974707245827, 0.03397754207253456, 0.03631154075264931, 0.00015504154725931585, 0.020863795652985573, -0.006890342570841312, 0.028960438445210457, 0.023850271478295326, -0.036672383546829224], [0.009235706180334091, 0.024838553741574287, -0.023379715159535408, -0.019329281523823738, 0.00638906704261899, -0.03554726764559746, 0.030478201806545258, 0.028481150045990944, 0.010570143349468708, -0.03241703286767006, 0.014872492291033268, -0.023319901898503304, -0.023490576073527336, -0.017071330919861794, 0.009578673169016838, 0.0020102544222027063, 0.017577150836586952, 0.03342195600271225, -0.02493155188858509, 0.0007437362801283598, 0.02361561730504036, 0.011873592622578144, 0.01964701898396015, -0.026520226150751114, 0.013904000632464886, -0.003943621180951595, 0.008710989728569984, 0.007342946249991655, 0.0005665341159328818, 0.03529989346861839, -0.02518988586962223, -0.00951455719769001, -0.013155084103345871, -0.026937680318951607, 0.008102909661829472, 0.0008280458278022707, 0.025498561561107635, -0.01985151134431362, 0.013855348341166973, -0.022787488996982574, -0.052142832428216934, -0.0029375937301665545, 0.03768734261393547, -0.00912602711468935, -0.007357564754784107, 0.004970417357981205, 0.01277641486376524, -0.025383273139595985, -0.0014123135479167104, -0.004372292198240757, 0.013449638150632381, -0.005484845954924822, 0.01039736159145832, 0.013362745754420757, -8.002157119335607e-05, 0.005309148225933313, -0.0075348010286688805, -0.014227048493921757, 0.00873064436018467, 0.00996329728513956, -0.015441426075994968, -0.007249565329402685, -0.02411053143441677, 0.022580819204449654], [-0.014767494983971119, 0.003339923219755292, 0.0065854694694280624, 0.03833790123462677, 0.016160117462277412, -0.0032807011157274246, 0.024163268506526947, -0.01992023177444935, -0.009118124842643738, 0.012693794444203377, 0.008143813349306583, -0.0009905325714498758, -0.02297324687242508, 0.0008375932229682803, 0.026263071224093437, -0.02401958778500557, 0.04026025906205177, 0.015082628466188908, -0.025721823796629906, -0.005543641746044159, -0.013991102576255798, -0.007810303941369057, 0.029980847612023354, 0.017058029770851135, -0.01741160824894905, 0.023256856948137283, -0.0218958780169487, 0.008158782497048378, -0.01373376790434122, 0.008755643852055073, -0.034568920731544495, 0.008318673819303513, -0.0038833569269627333, -0.003951065242290497, 0.026189513504505157, 0.011698770336806774, -0.0004859597247559577, 0.0012452182127162814, 0.0007176102953962982, -0.030682841315865517, -0.0034037658479064703, -0.04191435128450394, 0.043523259460926056, -0.002343917265534401, 0.007274119649082422, 0.015820786356925964, -0.03276588395237923, 0.0029700291343033314, -0.011511820368468761, 0.039238471537828445, -0.0021398812532424927, 0.021247942000627518, 0.007932038977742195, 0.021167997270822525, 0.013359645381569862, 0.011223678477108479, -0.0038264126051217318, 0.01774303801357746, -0.0006645536632277071, 0.02287943847477436, 0.012603097595274448, 0.017575979232788086, -0.0015215727034956217, 0.016529982909560204], [-0.019047999754548073, -0.02843807265162468, 0.01726698875427246, 0.029510922729969025, 0.025239156559109688, 0.008605471812188625, -0.021047785878181458, -0.013557634316384792, 0.028088413178920746, -0.0008959915721789002, -0.008914829231798649, -0.01778281293809414, 0.005769182927906513, -0.013156476430594921, -0.03355637937784195, 0.008067217655479908, -0.001331897103227675, -0.01022085826843977, 0.0050506312400102615, -0.028027724474668503, 0.0017481024842709303, -0.018366875126957893, 0.006165982224047184, -0.003278558375313878, -0.021724820137023926, -0.004303388297557831, 0.010357825085520744, -0.008093365468084812, 0.014065137133002281, -0.01983661763370037, -0.00020072641200385988, 0.02584858611226082, 0.003770251991227269, 0.01057908684015274, 0.031817078590393066, 0.0016960075590759516, -0.029143141582608223, -0.0052962214685976505, -0.015108789317309856, -0.003685545874759555, -0.008856204338371754, -0.008659377694129944, -0.003783159190788865, 0.03129475191235542, 0.003916495945304632, -0.01873299852013588, 0.025663429871201515, 0.03277868032455444, 0.0009520573657937348, -0.013856970705091953, 0.03188883140683174, 0.012822950258851051, 0.022266162559390068, 0.0025120151694864035, -0.01771295629441738, 0.039067965000867844, -0.001048876205459237, 0.009577077813446522, -0.03645754233002663, -0.008254108019173145, -0.008165969513356686, 0.011249956674873829, 0.01815079152584076, -0.0130844097584486], [-0.037999045103788376, 0.005532438866794109, -0.006137832533568144, 0.019669892266392708, -0.012815398164093494, 0.027761859819293022, -0.019934529438614845, -0.01679966226220131, -0.0033713479060679674, -0.0006522724288515747, 0.019518094137310982, 0.0027964820619672537, -0.03859028220176697, 0.044746268540620804, -0.015192518010735512, 0.014925853349268436, 0.006850646808743477, -0.0018960647284984589, 0.02266179956495762, 0.008067954331636429, -0.007566169369965792, -0.0333900973200798, -0.012761997990310192, -0.014168791472911835, 0.000813284597825259, -0.012399788945913315, -0.018938519060611725, -0.037008412182331085, -0.024286387488245964, 0.009712743572890759, -0.06387513130903244, 0.003976162988692522, 0.019103560596704483, 0.005404440686106682, -0.005881344433873892, -0.027609162032604218, 0.006846515461802483, 0.01856689155101776, -0.02374720573425293, 0.002886456437408924, -0.015931328758597374, -0.01486571878194809, 0.0026850062422454357, 0.008713255636394024, 0.038019709289073944, 0.009194956161081791, -0.012498973868787289, 0.008185877464711666, 0.009487192146480083, 0.03240079805254936, -0.033601123839616776, 0.014535142108798027, 0.0025695019867271185, -0.018549473956227303, 0.03137781843543053, -0.03892756998538971, 0.0062696244567632675, -0.0003483127511572093, -0.001370224985294044, -0.0074778455309569836, 0.05974024906754494, -0.018173741176724434, -0.02332419902086258, -0.0127111142501235], [0.025112956762313843, -0.005498824175447226, 0.022718722000718117, -0.014501631259918213, 0.025259992107748985, -0.019316595047712326, -0.015610198490321636, 0.001330721192061901, 0.023860987275838852, -0.009156973101198673, -0.004137831274420023, -0.047335438430309296, -0.003529948415234685, -0.003949795849621296, 0.031181184574961662, -0.0019071792485192418, 0.01823011226952076, 0.023700837045907974, -0.00038462522206828, -0.0258337315171957, 0.01335501205176115, 0.011534174904227257, 0.009111001156270504, 0.017231835052371025, 0.03687664121389389, -0.005391106009483337, -0.02125856839120388, -0.002740924945101142, -0.01286451518535614, 0.013860433362424374, -0.019837679341435432, 0.025654656812548637, 0.0013803878100588918, 0.01654670387506485, -0.027535799890756607, 0.02253899723291397, 0.016863612458109856, -0.024175958707928658, -0.04651504382491112, -0.00730395270511508, 0.006518969777971506, 0.009109452366828918, 0.000981937744654715, -0.01578463986515999, -0.03212150186300278, -0.01065690629184246, 0.044565409421920776, -0.007107345853000879, 0.006246079690754414, -0.009998582303524017, 0.01999075338244438, 0.011041408404707909, 0.0008078315877355635, -0.006158181931823492, 0.0653773695230484, -0.028803905472159386, -0.023280125111341476, 0.012010638602077961, 0.012364014983177185, 0.03133050724864006, -0.003380406182259321, 0.022946959361433983, -0.033413179218769073, -0.00682333717122674], [-0.020701592788100243, -0.02816035784780979, 0.01495877094566822, 0.016623297706246376, 0.025205325335264206, -0.006536852102726698, 0.020805036649107933, -0.005345412064343691, 0.0027483035810291767, -0.006385789718478918, 0.012926594354212284, 0.023307742550969124, 0.003841051599010825, 0.0367339551448822, 0.0135857705026865, -0.01845592074096203, -0.02628587931394577, -0.006680625956505537, 0.004086092580109835, 0.020399311557412148, -0.027062050998210907, 0.02359185181558132, 0.007803961168974638, -0.007196192163974047, 0.006563479080796242, -0.0043533374555408955, -0.008058418519794941, 0.017203843221068382, 0.023177482187747955, 0.014361358247697353, 0.030884670093655586, 0.007013814523816109, -0.01649998314678669, -0.009559032507240772, -0.003480686107650399, 0.0005920634721405804, 0.0030361185781657696, 0.0037903767079114914, 0.020499279722571373, 0.01681247167289257, 0.01575833559036255, -0.023572595790028572, -0.01174293365329504, -0.014098956249654293, 0.0016538492636755109, 0.0008695731521584094, 0.021882474422454834, -0.01069160457700491, -0.024152131751179695, -0.009636635892093182, 0.01895723305642605, 0.01144486851990223, -0.021723458543419838, 0.04123995453119278, -0.013152115046977997, -0.0035056991036981344, 0.0036328027490526438, 0.01797669753432274, -0.014450863003730774, 0.004475540481507778, 0.01150535698980093, 0.014439040794968605, -0.035941462963819504, -0.011732959188520908], [-0.014153274707496166, -0.020458225160837173, -0.015753937885165215, -0.005925679113715887, 0.012034837156534195, 0.011994208209216595, -0.01304452121257782, -0.009772245772182941, -0.008158085867762566, -0.007832753472030163, -0.00556657649576664, -0.004331807140260935, 0.010056533850729465, -0.025420475751161575, 0.016195841133594513, 0.014566440135240555, -0.018619900569319725, 0.007506673689931631, 0.007462565787136555, -0.008207953535020351, 0.049904488027095795, 0.00809654500335455, 0.011163529939949512, -0.0013553840108215809, 0.03450249880552292, -0.03618498146533966, -0.0071198116056621075, -0.003434437792748213, 0.03082694672048092, -0.017138604074716568, -0.007714718114584684, -0.04509573429822922, -0.026435978710651398, 0.013992358930408955, 0.004773857071995735, 0.005262821447104216, 0.02146037667989731, 0.020596303045749664, -0.02731194905936718, 0.03061641752719879, 0.014793595299124718, -0.00508783757686615, 0.0014866890851408243, -0.025154991075396538, -0.01736682467162609, 0.00566370552405715, -0.0024276848416775465, -0.011242851614952087, -0.015594937838613987, 0.00011931890912819654, -0.012178112752735615, -0.014110270887613297, 0.006531609687954187, -0.012392538599669933, 0.017877908423542976, -0.013768073171377182, -0.03321727737784386, 0.02310294844210148, 0.020485615357756615, -0.023692892864346504, -0.005509560462087393, 0.001862464938312769, -0.012337380088865757, -0.030058380216360092], [0.0199132077395916, -0.009355093352496624, -0.008048282004892826, -0.010120776481926441, 0.02727609872817993, 0.023587822914123535, -0.021600646898150444, -0.03601619973778725, -0.036433808505535126, 0.012524103745818138, -0.020376058295369148, -0.014350289478898048, 0.002435234608128667, 0.012105423957109451, 0.011696139350533485, 0.020067967474460602, 0.03800160810351372, -0.011363617144525051, 0.019578443840146065, 0.04086386784911156, -0.022387776523828506, -0.015843214467167854, -0.018781375139951706, 0.01505929883569479, 0.019344540312886238, 0.01789671927690506, -0.0032787066884338856, -0.0442003533244133, -0.01173433754593134, 0.010005420073866844, 0.029118068516254425, 0.016956865787506104, 0.010642411187291145, 0.035208720713853836, -0.02058595046401024, 0.0149167375639081, 0.013557899743318558, 0.022375458851456642, 0.02017063833773136, -0.0035259209107607603, 0.0273259487003088, 0.01110742799937725, -0.031131409108638763, 0.027906794100999832, 0.012721851468086243, -0.0020386322867125273, 0.02201363444328308, -0.008937859907746315, 0.006011190824210644, -0.03118402697145939, 0.0018792700720950961, -0.01755690574645996, -0.02192448265850544, -0.020308267325162888, -0.006294653285294771, 0.01721225678920746, -0.0030720848590135574, 0.003170971991494298, 0.0018049777718260884, 0.011633497662842274, 0.021562857553362846, 0.006025099661201239, -0.004239612724632025, -0.01010730117559433], [0.01675184816122055, -0.0031791802030056715, 0.023762930184602737, 0.027798186987638474, -0.01232854649424553, 0.020475907251238823, -0.0018152886768803, -0.015223026275634766, 0.02209668979048729, -0.004428493324667215, -0.023082716390490532, -0.0025456922594457865, 0.010938442312180996, 0.005937042646110058, -0.009244382381439209, 0.004542476497590542, 0.008513129316270351, -0.0045913709327578545, -0.04200635850429535, 0.01929931342601776, 0.009978000074625015, -0.03768810257315636, 0.009871620684862137, 0.006346189416944981, -0.004800981841981411, 0.011987091973423958, 0.02958398498594761, -0.024681594222784042, 0.023061957210302353, 0.003349179634824395, -0.010739242658019066, 0.005851568654179573, -0.003381687216460705, -0.020826756954193115, 0.0024270701687783003, 0.04009244590997696, 0.01081021223217249, -0.010685623623430729, 0.01112209539860487, -0.005022834520787001, 0.0008536507375538349, 0.003904852783307433, -0.019651977345347404, -0.02685144543647766, 0.0007759530562907457, -0.02054808847606182, -0.07084367424249649, 0.025186393409967422, -0.018072273582220078, -0.0033284896053373814, -0.02507876232266426, 0.025826655328273773, 0.0064077009446918964, -0.0003742624830920249, 0.020520996302366257, -0.026436422020196915, 0.023692524060606956, -0.05796610936522484, 0.010000262409448624, -0.004172198474407196, -0.04058093577623367, 0.022801950573921204, 0.0008254696149379015, 0.01710391789674759], [-0.019852587953209877, 0.011263178661465645, -0.012782646343111992, 0.01091369055211544, 0.007828403264284134, 0.029497729614377022, 0.012627958320081234, -0.006355014629662037, -0.03133147954940796, 0.02330339327454567, 0.029188884422183037, -0.0011599367717280984, 0.0049969470128417015, 0.00776138249784708, 0.03367193415760994, 0.04621850326657295, -0.0027047761250287294, -0.00037517849705182016, 0.011495424434542656, -0.006708335597068071, -0.03280272334814072, -0.00028208273579366505, 0.01449189055711031, -0.019039765000343323, -0.004174869041889906, 0.042646374553442, 0.015052344650030136, -0.008806843310594559, 0.0028066702652722597, 0.017650682479143143, 0.02946261130273342, 0.02837296389043331, -0.041983526200056076, -0.008950953371822834, 0.01022258773446083, 0.024187207221984863, -0.03794828802347183, 0.008910628966987133, 0.0037951027043163776, 0.017445961013436317, 0.017795676365494728, 0.026610102504491806, -0.03130458667874336, 0.000543999660294503, -0.006484178360551596, 0.005373168271034956, 0.0023636166006326675, -0.0009071681415662169, -0.005565316881984472, 0.0027943283785134554, 0.02654142491519451, -0.0005520455888472497, 0.008366857655346394, 0.02928786538541317, 0.03321300446987152, -0.0036444272845983505, 0.04704159125685692, -0.0008111119386740029, -0.009840595535933971, -0.01324451807886362, -0.015063281171023846, -0.011350246146321297, -0.0007898739422671497, -0.04260464757680893], [-0.0197755079716444, 0.013409634120762348, -0.028332581743597984, 0.015090263448655605, -0.01855636201798916, -0.00792897492647171, -0.006794869899749756, 0.008968115784227848, 0.011371723376214504, 0.02498755231499672, -0.03121032938361168, -0.002335190074518323, -0.009101794101297855, -0.022295769304037094, 0.0068539585918188095, 0.02851749211549759, -0.026708481833338737, -0.008580245077610016, 0.008388745598495007, -0.0048232367262244225, -0.023614313453435898, -0.005502202548086643, -0.0006820267299190164, -0.010720024816691875, 0.015588432550430298, -0.00010770605149446055, -0.0013217058731243014, 0.00985849928110838, -0.013918990269303322, -0.0038791028782725334, -0.0170865748077631, -0.01406142022460699, -0.004220881033688784, 0.006437364034354687, 0.026531990617513657, -0.023675426840782166, 0.026673123240470886, -0.002123425481840968, -0.02435147389769554, 0.009054812602698803, -0.027404338121414185, 0.02674594148993492, -0.005754594691097736, -0.0038429151754826307, 0.03998780995607376, 0.004130230285227299, 0.020686671137809753, 0.025590715929865837, -0.021035252138972282, -0.01619655266404152, 0.003011983586475253, 0.009205625392496586, -0.012284842319786549, 0.021956538781523705, -0.023991549387574196, -0.00196063076145947, -0.036804452538490295, 0.014316274784505367, -0.012010298669338226, -0.0199944656342268, 0.015452280640602112, 0.006989233661442995, 0.02655152976512909, -0.001975368009880185], [0.028899379074573517, -0.01936843991279602, -0.002618534490466118, -0.011009219102561474, -0.02545074373483658, 0.0007693258230574429, 0.0022590095177292824, 0.004672278184443712, -0.03329146280884743, -0.0016737988917157054, 0.014491850510239601, -0.032341454178094864, 0.013169065117835999, -0.04450509697198868, -0.0162831861525774, -0.008434908464550972, 0.01951671950519085, -0.004274485632777214, 0.01625918596982956, -0.009891779161989689, 0.0052872635424137115, 0.020974524319171906, 0.03257614001631737, 0.018790490925312042, -0.027180911973118782, 0.0143368449062109, 0.024670669808983803, 0.03838155418634415, -0.013124460354447365, 0.05918813496828079, 0.0065412819385528564, 0.022165212780237198, -0.013663588091731071, -0.0204086285084486, -0.02854817360639572, 0.0033470424823462963, 0.008181409910321236, 0.026736825704574585, 0.012267252430319786, -0.003526614513248205, 0.02915845438838005, 0.009294750168919563, 0.0014325538650155067, -0.008128142915666103, -0.02844398468732834, 0.02947552502155304, 0.004710567649453878, -0.031962789595127106, 0.004741327837109566, 0.02573229745030403, 0.02711547166109085, -0.017509473487734795, -0.0380224846303463, 0.004213111475110054, 0.02051980048418045, -0.054560586810112, -0.006898777559399605, -0.0008136952528730035, 0.0022027303930372, -0.015852902084589005, -0.013387233018875122, 0.011055647395551205, 0.0298500657081604, -0.0076728444546461105], [0.00013461580965667963, -0.0027565667405724525, -0.007953607477247715, -0.017185620963573456, 0.00022832545801065862, -0.019684769213199615, 0.04697126895189285, 0.02631550282239914, 0.046724092215299606, 0.011274391785264015, 0.00023072869225870818, 0.013570718467235565, 0.011012105271220207, 0.030991118401288986, -0.005484725348651409, 0.03252739831805229, 0.018179520964622498, -0.009052390232682228, -0.028478724882006645, -0.009755821898579597, -0.006700898986309767, -0.024251092225313187, -0.037015777081251144, 0.00688780564814806, 0.025063570588827133, 0.016485273838043213, 0.006826810073107481, -0.024051349610090256, -0.020369207486510277, -0.0072608329355716705, 0.004868053365498781, -0.0165350753813982, -0.0011305753141641617, 0.01640714518725872, 0.012412144802510738, -0.02294599823653698, -0.014197885990142822, -0.01690097339451313, -0.005146772135049105, 0.02600388415157795, 0.01682029478251934, 0.012151113711297512, 0.005946080666035414, 0.011615547351539135, 0.0013238468673080206, 0.017128799110651016, 0.0007654163055121899, -0.008831319399178028, 0.03693343326449394, -0.0034126273822039366, 0.004590959753841162, 0.007552673574537039, 0.02235611341893673, -0.014111822471022606, -0.0036439518444240093, -0.012418700382113457, -0.0059543270617723465, 0.004944960121065378, -0.005426757037639618, -0.011083266697824001, -0.006058185826987028, 0.016120130196213722, -0.002491789637133479, 0.026952315121889114], [-0.010327723808586597, -0.02744641713798046, 0.005858213175088167, 0.019726231694221497, 0.019128408282995224, -0.012170969508588314, 0.027544857934117317, -0.0230221189558506, 0.010811646468937397, 0.045810334384441376, -0.01094135269522667, -8.038448140723631e-05, -0.014387100003659725, -0.0199429914355278, -0.007978690788149834, 0.04244714602828026, 0.0031529502011835575, 0.007970909588038921, -0.0050968751311302185, 0.028246061876416206, -0.04042181372642517, -0.001253078575246036, 0.011573657393455505, 0.010738157667219639, 0.030953990295529366, -0.00728875957429409, -0.0038520838133990765, 0.03047802299261093, 0.024924762547016144, 0.0160879697650671, 0.023095330223441124, -0.00987268891185522, -0.026335714384913445, 0.008621178567409515, 0.00010987753194058314, -0.007876712828874588, -0.015725838020443916, -0.012511787936091423, -0.0005181654705666006, 0.00027112214593216777, 0.0003245839034207165, 0.000616673263721168, -0.013025467284023762, 0.005129176191985607, 0.013992720283567905, -0.008510299026966095, 0.010329794138669968, -0.024489490315318108, -0.025641776621341705, -0.0011872454779222608, -0.011615149676799774, 0.0038573516067117453, -0.009480291977524757, -0.036819733679294586, -0.03440817445516586, 0.012971218675374985, 0.0058941226452589035, -0.019657226279377937, -0.005327514372766018, -0.00720276078209281, -0.008720846846699715, 0.010362353175878525, -0.005188284907490015, 0.02297906018793583], [0.0017767340177670121, -0.02199898473918438, -0.012634694576263428, -0.03022501990199089, 0.03967544063925743, 0.00021275682956911623, -0.02675948292016983, -0.03793911635875702, -0.013131801970303059, 0.00016030488768592477, 0.01139829121530056, 0.005562508013099432, -0.028296323493123055, 0.03897115960717201, -0.025409124791622162, 0.043500903993844986, -0.008096994832158089, -0.03989855945110321, -0.007318866904824972, -0.0050927880220115185, -0.003919984679669142, -0.025700729340314865, -0.0031571451108902693, 0.011599981226027012, -0.003499669721350074, -0.026960322633385658, 0.012128678150475025, -0.01270977407693863, -0.001708399853669107, -0.01596948690712452, -0.010764667764306068, 0.018745699897408485, -0.014826984144747257, -0.043046992272138596, 0.019515331834554672, 0.0036715553142130375, 0.009349620901048183, 0.002577567007392645, -0.008158446289598942, -0.002410182962194085, 0.01572992652654648, 0.02082080952823162, -0.0159738902002573, 0.0007955741020850837, 0.00895602535456419, 0.015525372698903084, -0.00968049094080925, -0.01732064224779606, -0.02546638995409012, -0.0072356644086539745, 0.0033270723652094603, 0.02872271090745926, 0.01880500465631485, -0.011656722985208035, -0.027579879388213158, -0.003440520027652383, 0.0071326433680951595, 0.014642691239714622, -0.009637960232794285, 0.014878306537866592, -0.01860946975648403, 0.010606132447719574, 0.023666858673095703, 0.004555826541036367], [-0.01526364404708147, -0.014411251991987228, 0.03900139033794403, -0.006437045522034168, -0.014792190864682198, -0.004220457747578621, -0.010240063071250916, -0.022135375067591667, 0.018027594313025475, -0.009678109548985958, -0.017944246530532837, -0.014533396810293198, 0.006452798843383789, 0.02148621901869774, 0.02901116944849491, -0.011841267347335815, -0.012198763899505138, 0.00032990414183586836, 0.002489422680810094, -0.0129774771630764, -0.014089391566812992, 0.013874762691557407, -0.019365672022104263, -0.01402003038674593, -0.021683992817997932, -0.010779323987662792, -6.689111614832655e-05, 0.017594730481505394, 0.004351845011115074, -0.007707290817052126, 0.005381227470934391, 0.014750123955309391, 0.021839657798409462, -0.020556360483169556, 0.006107169669121504, -0.016574079170823097, 0.013208397664129734, 0.011810907162725925, -0.003399808192625642, 0.03799659386277199, -0.0188446082174778, 0.014515040442347527, 0.001610583160072565, 0.008089898154139519, 0.018205121159553528, 0.00773695157840848, 0.0016582050593569875, -0.010717998258769512, -0.04126501828432083, 0.044138118624687195, 0.0012537689181044698, -0.014450665563344955, 0.0026408694684505463, 0.005331982858479023, -0.0073358057998120785, 0.005355866625905037, 0.020663464441895485, -0.023451581597328186, 0.007638472132384777, 0.016591383144259453, 0.011223696172237396, -0.034790944308042526, -0.004131716210395098, 0.016651242971420288], [-2.516679887776263e-05, -0.009591923095285892, 0.04510703310370445, 0.015117309056222439, 0.011135305278003216, 0.021750612184405327, 0.017795123159885406, 0.009649565443396568, -0.0017526474548503757, -0.01005718857049942, -0.019428420811891556, -0.012280895374715328, 0.007706614676862955, -0.0033216497395187616, 0.027457335963845253, -0.007646204438060522, 0.015150676481425762, -0.007013980764895678, -0.0051955971866846085, -0.01232570968568325, -0.01067268755286932, 0.01898818463087082, 0.048571620136499405, -0.026204636320471764, 0.030974702909588814, -0.020231591537594795, -0.007828428409993649, 0.03474539890885353, -0.006208118516951799, -2.2168933355715126e-05, 0.0019867829978466034, -0.008779007941484451, 0.013993348926305771, -0.011330862529575825, 0.004689950495958328, -0.02031511440873146, -0.019477954134345055, -0.0298942718654871, 0.017578911036252975, 0.044222597032785416, 0.01391051895916462, 0.023955076932907104, -0.0061597274616360664, 0.0014323994982987642, 0.009962115436792374, 0.0034239101223647594, 0.00378485512919724, -0.0007840557373128831, -0.001958949724212289, -0.008024457842111588, -0.014051418751478195, -0.013505318202078342, 0.05065170302987099, -0.05317751690745354, 0.024047555401921272, -0.026828406378626823, -0.0016303576994687319, -0.01203625462949276, -0.013126414269208908, -0.02475731261074543, -0.011333193629980087, -0.0028525260277092457, -0.011607008054852486, -0.009966873563826084], [-0.00013450620463117957, 0.022287776693701744, 0.018055815249681473, -0.015154273249208927, -0.010759011842310429, 0.037719838321208954, 0.009052311070263386, -0.034542061388492584, 0.0015172334387898445, 0.01949845440685749, 0.01219214964658022, -0.006967498455196619, -0.007259526755660772, 0.00021979089069645852, 0.01745540089905262, -0.0002547303447499871, 0.04586886987090111, 0.01255703903734684, 0.006301248911768198, 0.02715439908206463, -0.02792513743042946, 0.01791943423449993, -0.007396339438855648, -0.0012168254470452666, -0.004542911425232887, -0.005888589192181826, 0.0010028695687651634, -0.03357547149062157, 0.03623639792203903, 0.009986772201955318, 0.007665376178920269, 0.04682520404458046, -0.023045646026730537, -0.02656281180679798, 0.019997911527752876, -0.03316795453429222, 0.013201852329075336, -0.0008520355331711471, 0.009692366234958172, -0.013614610768854618, 0.006040715146809816, 0.024988675490021706, 0.04282122477889061, -0.024548470973968506, -0.035500701516866684, 0.03897599130868912, -0.009114827029407024, 0.00891889352351427, 0.003553131828084588, -0.02838331088423729, -0.014542343094944954, 0.011223290115594864, -0.0062379599548876286, 0.013920776546001434, -0.023555191233754158, -0.011190639808773994, 0.0032013189047574997, 0.00042162943282164633, -0.00975731946527958, -0.011458576656877995, 0.005402936600148678, -0.036468859761953354, 0.012684027664363384, 0.004528259392827749], [-0.010822482407093048, 0.03769679740071297, 0.05174511671066284, -0.013389208354055882, -0.042633332312107086, -0.0011612065136432648, 0.019613105803728104, 0.01954106241464615, 0.01595287397503853, 0.02620345912873745, 0.0007510234718210995, 0.020786255598068237, -0.012222888879477978, 0.009929895401000977, 0.009509771130979061, -0.001166067784652114, 0.005009625107049942, -0.004641583189368248, -0.026014717295765877, 0.007821246981620789, -0.0021014008671045303, -0.011209777556359768, 0.022811634466052055, 0.013477044180035591, -0.006154376082122326, -0.015278688631951809, 0.0023052638862282038, -0.0271094162017107, 0.0469837486743927, -0.005744824185967445, 0.04068533331155777, -0.03882589936256409, 0.015243271365761757, -0.018151644617319107, 0.01082723680883646, -0.029825955629348755, 0.012498773634433746, -0.028361182659864426, -0.0030591317918151617, -0.017858458682894707, 0.008429226465523243, -0.017596909776329994, -0.0003919242008123547, -0.005189777351915836, -0.018731839954853058, -0.00025284182629548013, 0.0016294822562485933, -0.019343165680766106, -0.02152112126350403, -0.007471051067113876, 0.006998262368142605, -0.00604116590693593, -0.011118896305561066, 0.01098728459328413, -0.00019619952945504338, 0.0019066836684942245, 0.01292491890490055, -0.020740635693073273, -0.018898742273449898, -0.00505452835932374, -0.018127253279089928, 0.01681622304022312, -0.017296161502599716, -0.006484394893050194], [0.02172805927693844, -0.00833837129175663, -0.029075801372528076, -0.01855335757136345, 0.011549999006092548, -0.04461558163166046, 0.005284815561026335, -0.009850572794675827, 0.01270567998290062, 0.0033439036924391985, 0.016557155176997185, 0.013457859866321087, 0.0106415506452322, 0.002310024108737707, 0.024536604061722755, 0.022972306236624718, -0.010290336795151234, -0.01152083370834589, -0.015218270011246204, -0.014229828491806984, -0.00971425510942936, -0.010389463044703007, 0.021749641746282578, -0.024938136339187622, 0.018923485651612282, 0.02877119928598404, 0.02598504349589348, 0.0007584104896523058, -0.036049216985702515, 0.016418198123574257, 0.029174158349633217, -0.020892368629574776, -0.01603926159441471, -0.01068391464650631, -0.00627630390226841, 0.0012249686988070607, -0.04807047173380852, 0.013605414889752865, 0.00023084563144948334, -0.03839975595474243, 0.018183959648013115, -0.011517631821334362, -0.02010207436978817, -0.0024659992195665836, -0.0017684210324659944, 0.0197810847312212, -0.003846291219815612, 0.002642155159264803, 0.009289641864597797, 0.01961752399802208, -0.022819239646196365, 0.020571941509842873, -0.04149169847369194, -0.001956877065822482, -0.01601545698940754, 0.0020503588020801544, 0.016398409381508827, 0.011367258615791798, -0.00992628838866949, -0.05556744337081909, -0.03051518276333809, -0.023560771718621254, -0.018824215978384018, -0.011065451428294182], [0.0011064456775784492, -0.00439156498759985, -0.008332563564181328, 0.0045901937410235405, 0.03949693590402603, 0.010321194306015968, 0.0008933249046094716, 0.028007900342345238, -0.020157527178525925, -0.0016641999827697873, 0.0016177954385057092, 0.014291138388216496, -0.01836657151579857, -0.0069091590121388435, 0.005733610596507788, 0.0027786623686552048, -0.00893106497824192, 0.0012534558773040771, 0.014134778641164303, 0.010693664662539959, -0.005061139818280935, 0.014471049420535564, 0.001446863985620439, 0.01756136864423752, -0.012036015279591084, 0.02412179671227932, 0.008895511738955975, -0.0018910487415269017, 0.017695197835564613, -0.004440419375896454, 0.012301723472774029, 0.049738556146621704, -0.011260435916483402, 0.0025361082516610622, -0.014843132346868515, -0.015450805425643921, 0.0056843203492462635, -0.006150421686470509, 0.0089113125577569, 0.03492628037929535, -0.0205095037817955, 0.037316348403692245, 0.012895207852125168, 0.008797740563750267, 0.006106631364673376, 0.016687624156475067, -0.011980491690337658, -0.01725688762962818, -0.015793321654200554, -0.005716281943023205, 0.0019611583556979895, 0.024230295792222023, -0.020106816664338112, -0.026865623891353607, 0.006895079277455807, -0.039012450724840164, 0.021707814186811447, -0.018348200246691704, -0.011998564936220646, 0.010666465386748314, 0.03992119058966637, -0.017499588429927826, -0.00013086012040730566, 0.008970806375145912], [0.019363513216376305, 0.02935538813471794, 0.027989795431494713, 0.00729157542809844, -0.0027058525010943413, -0.011492885649204254, 0.00825219601392746, 0.036669373512268066, 0.015435478650033474, -0.019869443029165268, -0.0036539167631417513, -0.0021681301295757294, -0.018351176753640175, -0.008876025676727295, -0.004641360603272915, -0.054039303213357925, 0.009323312900960445, -0.010235941968858242, -0.056462571024894714, -0.0052420650608837605, -0.006235694512724876, 0.005242107901722193, 0.007212373428046703, -0.029144007712602615, 0.00020826836407650262, -0.03129563108086586, 0.01822865568101406, 0.0027607399970293045, -0.0005861244280822575, -0.017110126093029976, -0.04533703252673149, 0.008971466682851315, 0.02223406359553337, 0.007904316298663616, -0.026596959680318832, -0.021662963554263115, 0.03975130617618561, -0.003283121157437563, -0.02416040003299713, 0.026344947516918182, 0.017934726551175117, -0.018943633884191513, -0.010619940236210823, 0.018000679090619087, 0.0007997075445018709, -0.0048291911371052265, -0.02523620054125786, -0.004733900539577007, -0.02884490042924881, -0.020600557327270508, 0.02921326830983162, -0.05213112011551857, 0.007589732762426138, -0.02044743299484253, -0.007211298681795597, -0.00149857671931386, 0.005397920962423086, 0.016697566956281662, -0.012754744850099087, -0.00886620208621025, 0.0007903800578787923, 0.02600846253335476, -0.013513539917767048, -0.008979415521025658], [0.011765149421989918, 0.022334935143589973, 0.018413007259368896, 0.0009240154759027064, 0.0010861271293833852, 0.006754379253834486, 0.0029359995387494564, 0.0001569945743540302, 0.015194637700915337, 0.03440451994538307, -0.0017957078525796533, -0.005070037674158812, 0.004625506699085236, 0.008062037639319897, 0.015374423936009407, 0.007755286060273647, 0.016163388267159462, -0.025409113615751266, 0.002281350316479802, 0.019048329442739487, 0.02073749713599682, 0.00035463168751448393, 0.014572247862815857, 0.022983379662036896, 0.03034074231982231, -0.012513860128819942, 0.015205400995910168, -0.013476663269102573, 0.002613269491121173, 0.009922510012984276, -0.014468127861618996, 0.007215781137347221, 0.008615582250058651, -0.016658572480082512, 0.00027224633959122, -0.009155690670013428, -0.01536811888217926, -0.020726844668388367, 0.02190982736647129, 0.027125580236315727, 0.0124460244551301, -0.042478110641241074, 0.030509553849697113, -0.025193165987730026, 0.0009859497658908367, 0.002352424431592226, 0.014666191302239895, -0.04111893102526665, 0.006240236107259989, 0.01314153615385294, -0.009673580527305603, -0.010169176384806633, 0.026400770992040634, -9.981147741200402e-05, -0.03354981541633606, -0.012491732835769653, 0.012846708297729492, 0.037858691066503525, 0.006062974222004414, 0.03197845444083214, -0.022661147639155388, -0.019939567893743515, -0.018407708033919334, -0.00040687102591618896], [0.018720556050539017, -0.0032746775541454554, 0.028103219345211983, -0.00514059467241168, 0.002123406622558832, 0.010731128044426441, 0.02149648405611515, -0.041888508945703506, 0.010000850073993206, 0.014329853467643261, -0.011459342204034328, 0.03915542736649513, -0.015314968302845955, -0.008812895976006985, -0.011431475169956684, -0.007883289828896523, 0.03895439952611923, -0.004469274543225765, 0.013825423084199429, 0.0383012481033802, -0.0033417553640902042, 0.01773880422115326, 0.01596478372812271, -0.0018026918405666947, -0.008843963034451008, 0.0025601820088922977, 0.0366491936147213, -0.0010554911568760872, 0.020636575296521187, -0.03125889226794243, -0.006832679267972708, -0.021107474341988564, -0.005437302403151989, 0.01869937591254711, -0.03652586415410042, 0.017196157947182655, 0.010731956921517849, -0.0048878490924835205, -0.005429236683994532, 0.011028430424630642, 0.013256301172077656, -0.023277902975678444, -0.02813236229121685, -0.0038494672626256943, 0.011697226203978062, 0.006366625893861055, -0.0014617076376453042, 0.006305187474936247, 0.014798400923609734, 0.019612664356827736, 0.03365209326148033, 0.0036845444701611996, 0.0012708043213933706, 0.0050430200062692165, -0.025438359007239342, 0.024732766672968864, -0.0189177505671978, -0.028434744104743004, 0.019406277686357498, 0.01175448764115572, -0.0023796549066901207, -0.023197423666715622, 0.016069963574409485, 0.0476183220744133], [-0.041435230523347855, -0.01857568882405758, 0.00045556866098195314, 0.0014729938702657819, -0.0042314534075558186, 0.030391709879040718, -0.008941594511270523, 0.019068004563450813, 0.020240765064954758, 0.003358769230544567, 0.007429458200931549, -0.04569404572248459, -6.0106292949058115e-05, -0.02706241048872471, 0.005034407135099173, 0.027437450364232063, 0.004592164885252714, -0.0008617992280051112, 0.03797357156872749, 0.0037177372723817825, -0.0016102720983326435, 0.019246060401201248, -0.006438612472265959, 0.04465148597955704, -0.005792077165096998, 0.02155877836048603, -0.017460329458117485, -0.0213658194988966, 0.02431628853082657, -0.0014077166561037302, -0.0058218627236783504, 0.0035927200224250555, -0.03405407816171646, 0.004656751174479723, -0.01844128780066967, -0.0016147898277267814, 0.015890788286924362, -0.006343787536025047, 0.035549797117710114, 0.010859736241400242, -0.009447280317544937, -0.023052440956234932, -0.016994040459394455, 0.017532330006361008, 0.006927733309566975, 0.010872828774154186, 0.0023779382463544607, -0.02337677590548992, 0.026738779619336128, -0.008361943997442722, 0.008919372223317623, -0.004041965585201979, 0.004952401854097843, -0.005768593866378069, 0.001743518514558673, 0.012942117638885975, 0.01031747180968523, -0.013744859024882317, -0.00028917600866407156, -0.028830789029598236, 0.03393649309873581, 0.001529206521809101, 0.02886035107076168, 0.008549933321774006], [-0.012418901547789574, -0.022568479180336, -0.006755465175956488, -0.024026837199926376, -0.0009887139312922955, -0.0025982744991779327, -0.00398030923679471, 0.014947913587093353, 0.015426759608089924, -0.004009482450783253, -0.03212261199951172, 0.029544271528720856, 0.00764244981110096, 0.007810779381543398, -0.01819591037929058, -0.00935328658670187, -0.0010720527498051524, 0.00748821347951889, 0.018070470541715622, -0.0057499706745147705, -0.01708388887345791, 0.024375375360250473, 0.012944173999130726, -0.020858192816376686, -0.0016041053459048271, 0.0016328284982591867, 0.010902431793510914, 0.010929213836789131, -0.012505866587162018, -0.028826115652918816, -0.022357704117894173, -0.03341088071465492, -0.005690012592822313, 0.02289007604122162, 0.007757438812404871, 0.002586839022114873, 0.06427179276943207, -0.010285047814249992, 0.005128698423504829, -0.031210429966449738, -0.008845365606248379, 0.0021378425881266594, -0.025877969339489937, -0.03586197644472122, 0.007619383744895458, 0.00819172989577055, -0.0005164447939023376, 0.02430613897740841, -0.028345158323645592, -0.0014637685380876064, -0.0360354483127594, -0.006159186828881502, -0.012858002446591854, -0.029176143929362297, 0.020554233342409134, 0.02424769289791584, -0.015350199304521084, 0.005008018109947443, 0.00938052125275135, 0.01165037602186203, 0.012057507410645485, -0.004640965722501278, -0.009530313313007355, 0.014169068075716496], [0.013014961034059525, 0.023605303838849068, 0.02430948056280613, -0.013301868923008442, 0.02030879259109497, -0.01522405631840229, -0.01098877564072609, 0.008396483026444912, -0.012272223830223083, 0.01911144144833088, 0.008964196778833866, -0.008885313756763935, 0.0008387702400796115, 0.02052009105682373, -0.005344178061932325, 0.010601745918393135, -0.021664921194314957, -0.005553949624300003, 0.004044719971716404, 0.005183202214539051, 0.029076557606458664, -0.0005966029711998999, -0.0021795816719532013, -0.036508675664663315, -0.014046178199350834, 0.007412677630782127, 0.044540729373693466, 0.0004920580540783703, -0.0017460989765822887, -0.002323754597455263, -0.01326123159378767, 0.00237367395311594, 0.006891341879963875, 0.033574771136045456, 0.00014154282689560205, 0.012768796645104885, 0.019908664748072624, -0.012831983156502247, -0.02310926467180252, -0.0029663939494639635, 0.01495029404759407, 0.02844645269215107, 0.022647762671113014, 0.04033612832427025, -0.01477601658552885, -0.002017007442191243, -0.025037119165062904, 0.019858164712786674, 0.029567908495664597, 0.01149936206638813, 0.0017598969861865044, 0.004325851332396269, -0.01795121282339096, -0.01693776249885559, -0.005211528856307268, -0.0022895110305398703, -0.008266590535640717, -0.024242965504527092, 0.010022391565144062, -0.005004669539630413, 0.0029616481624543667, 0.01679760403931141, -0.03000062331557274, 0.01908928155899048], [0.012525190599262714, 0.0028004178311675787, -0.006808320991694927, 0.03058181330561638, -0.009947375394403934, -0.03246838226914406, 0.009079580195248127, 0.018035436049103737, 0.0255216546356678, 0.009911180473864079, 0.010378295555710793, 0.028758257627487183, -0.03544149175286293, 0.01532889623194933, -0.0004230158810969442, -0.015950649976730347, -0.007637512404471636, 0.023288294672966003, 0.044105224311351776, -0.0050555202178657055, 0.0027334264013916254, 0.0018852659268304706, 0.019981959834694862, 0.06378743797540665, -0.0449075922369957, 0.03053746372461319, 0.00750875985249877, 0.013576819561421871, -0.00031611425220035017, -0.03127750754356384, 0.01876400224864483, 0.016098540276288986, -0.009146512486040592, -0.018553240224719048, 0.00311612943187356, 0.023812541738152504, 0.0020893930923193693, -0.010963651351630688, -0.024256212636828423, 0.007825074717402458, 0.00848764181137085, 0.023262210190296173, 0.008065071888267994, 0.008776416070759296, 0.022287437692284584, -0.040807534009218216, 0.010156865231692791, -0.017637409269809723, -0.016218136996030807, 0.011786682531237602, 0.009960833005607128, 0.00394352525472641, 0.02935207262635231, -0.013031858019530773, 0.0062351287342607975, -0.00419630715623498, 0.03858908638358116, -0.016085950657725334, -0.0026484555564820766, 0.008251405321061611, 0.053907763212919235, 0.01564505510032177, 0.010516579262912273, 0.0022923105861991644], [0.018873756751418114, 0.01259442139416933, -0.006751319859176874, 0.002352982759475708, 0.01293498557060957, -0.017286695539951324, -0.012129911221563816, 0.02022051438689232, 0.01862359791994095, -0.03677297383546829, 0.00572834350168705, -0.006176382303237915, -0.027532150968909264, 0.02106347121298313, -0.038582295179367065, 0.011346021667122841, 0.010702621191740036, -0.057296860963106155, -0.001988437492400408, 0.009623109363019466, 0.0020406825933605433, 0.0379696860909462, -0.019042883068323135, 0.005299106705933809, -0.02118188887834549, -0.008190153166651726, 0.029474986717104912, -0.0056863147765398026, 0.042208220809698105, -0.00839839968830347, 0.009195868857204914, 0.0061751725152134895, 0.01468959916383028, 0.02193429134786129, -0.01237504929304123, -0.010084218345582485, -0.0018718511564657092, 0.00022861735487822443, 0.03249799460172653, -0.024236002936959267, -0.0009246448753401637, -0.0018458141712471843, 0.019555822014808655, -0.010689630173146725, 0.006429408676922321, -0.0018459843704476953, -0.00750721525400877, 0.010260623879730701, 0.025409195572137833, 0.012146377936005592, -0.005317335017025471, 0.0012919516302645206, 0.005339217837899923, 0.013598824851214886, -0.023278463631868362, -0.0019624263513833284, -0.004361618310213089, 0.025655359029769897, -0.030080940574407578, -0.010259978473186493, -0.01932767778635025, 0.030850030481815338, -0.006702250801026821, 0.010402826592326164], [0.01888781599700451, 0.006166822277009487, -0.009512491524219513, 0.02551870420575142, -0.012914658524096012, 0.0361456498503685, 0.005670439451932907, 0.020222853869199753, -0.03571204096078873, 0.014034206978976727, 0.03857048600912094, 0.0006884651375003159, -0.03219638764858246, -0.005983251612633467, 0.006637267302721739, -0.009226786904036999, -0.01566426083445549, 0.003011313732713461, -0.005695792380720377, 0.012515832670032978, -0.007533480413258076, -0.002207691315561533, 0.007721375674009323, 0.015050932765007019, 0.014057922177016735, -0.007875686511397362, -0.002301074331626296, -0.005983217153698206, 0.031044768169522285, -0.007975947111845016, 1.775361852196511e-05, -0.01431108359247446, -0.006320014130324125, -0.011368704959750175, -0.009848378598690033, 0.0013649348402395844, 0.010492327623069286, 0.011448836885392666, 0.0028461948968470097, 0.009194016456604004, -0.003081793896853924, -0.001276680501177907, -0.006009526550769806, 0.00821318943053484, 0.0006435466348193586, 0.036796316504478455, -0.02498251385986805, 0.0019090778660029173, -0.018466658890247345, -0.03797236084938049, -0.014248776249587536, -6.420924182748422e-05, -0.022294171154499054, 0.01990203559398651, 0.0036447392776608467, 0.006798938382416964, -0.02167438715696335, -0.0005395812913775444, 0.007766402326524258, 0.00862385518848896, 0.03644668683409691, 0.0007585646817460656, 0.010422893799841404, -0.0028385294135659933], [0.011682151816785336, -0.0006059862207621336, 0.0017668500076979399, 0.039082687348127365, 0.024281369522213936, -0.024960841983556747, -0.015171525999903679, 0.01644614338874817, -0.015503709204494953, -0.003686357056722045, 0.02032463811337948, -0.018545785918831825, 0.016821017488837242, -0.002407991560176015, 0.0036297461483627558, -0.010965666733682156, 0.005838284734636545, 0.014164118096232414, -0.009062801487743855, 0.002729711588472128, -0.014921566471457481, 0.015991736203432083, -0.02586083672940731, -0.008691106922924519, -0.02290266379714012, -0.02879478596150875, 0.0008878681692294776, 0.017150023952126503, 0.002480259397998452, 0.006935575511306524, -0.023036761209368706, 0.00806223601102829, 0.009393998421728611, -0.01132480800151825, 0.0029694398399442434, 0.0038981519173830748, 0.015294980257749557, -0.029766211286187172, -0.019009292125701904, -0.008280997164547443, 0.0028283093124628067, 0.006113338749855757, -0.012573798187077045, -0.01576491817831993, 0.01486537978053093, 0.006539535708725452, -0.033931881189346313, 0.0016781601589173079, 0.009092865511775017, 0.057223230600357056, 0.011394159868359566, 0.018750164657831192, -0.008391804061830044, 0.001892002997919917, -0.0015495925908908248, 0.006522750481963158, 0.006797122769057751, -0.009137735702097416, 0.005965692922472954, -0.0144802900031209, 0.0026380892377346754, -0.0036720517091453075, -0.025232981890439987, 0.006163648795336485], [0.010779699310660362, -0.010478388518095016, 0.004259137436747551, -0.001967361196875572, -0.03124646283686161, -0.0063834222964942455, -0.014683272689580917, -0.0015097991563379765, -0.007553077302873135, -0.03244982659816742, -0.005050239618867636, -0.02251584082841873, -0.006367700640112162, -0.003810981521382928, -0.006984385661780834, 0.017325829714536667, -0.00546101713553071, 0.006195154041051865, 0.020903656259179115, 0.01864301785826683, 0.031168879941105843, 0.011542505584657192, -0.03597002103924751, -0.006516290362924337, -0.027468550950288773, 0.013266710564494133, 0.006441900972276926, -0.028328217566013336, -0.013575512915849686, 0.01550354529172182, 0.007476744242012501, -0.008261175826191902, -0.03525556996464729, 0.01086946576833725, 0.0036110312212258577, -0.019842153415083885, -0.0309962946921587, -0.018987106159329414, 0.055899281054735184, 0.008827215991914272, -0.022312890738248825, -0.012256249785423279, -0.015050910413265228, 0.026004701852798462, 0.005247995723038912, -0.013901748694479465, -0.0016653905622661114, -0.024705491960048676, 0.033479999750852585, 0.009340114891529083, 0.0018912219675257802, 0.0001427912211511284, -0.007608753629028797, -0.02271082065999508, -0.0093234246596694, 0.03361287713050842, -0.008061597123742104, 0.025571128353476524, -0.008877004496753216, -4.996158531866968e-05, -0.034015607088804245, -0.021774964407086372, 0.001175951212644577, 0.018832504749298096], [0.006302338093519211, -0.02736985869705677, -0.020877664908766747, -0.014548140577971935, 0.02127666585147381, 0.02753359265625477, -0.025846175849437714, 0.00597828533500433, -0.02383091114461422, 0.030036550015211105, 0.025081288069486618, -0.023067930713295937, 0.029176607728004456, -0.02678794600069523, 0.015367626212537289, 0.00027732233866117895, -0.01141809206455946, -0.011092678643763065, -0.001706127543002367, 0.032634083181619644, 0.00047800253378227353, -0.0012981098843738437, -0.0181185994297266, 0.02714843861758709, 0.007403936702758074, 0.011180320754647255, -0.009725246578454971, -0.001835564966313541, -0.0032838168554008007, -0.015870574861764908, 0.0021204580552875996, -0.007146418560296297, -0.00692870793864131, 0.028725991025567055, -0.0126169603317976, 0.005169471725821495, 0.02677547186613083, -0.0072599537670612335, -0.01511541847139597, 0.02259664423763752, 0.02150646038353443, -0.005555974319577217, -0.009442847222089767, -0.015526737086474895, 0.019323457032442093, 0.0006584773655049503, -0.0115299541503191, -0.0038112038746476173, -0.027840755879878998, -0.005390002392232418, 0.00812725629657507, -0.014316978864371777, -0.007960679940879345, -0.013688824139535427, -0.019088435918092728, 0.03701037913560867, 0.005684186238795519, -0.021437352523207664, -0.02354354038834572, 0.04108801484107971, 0.01169433631002903, 0.020060574635863304, -0.004429677035659552, 0.011093220673501492], [-0.04948169365525246, 0.015409272164106369, 0.005062994547188282, -0.015694595873355865, 0.018471218645572662, 0.014456202276051044, -0.014295275323092937, -0.006440914701670408, -0.005282548256218433, 0.031326938420534134, -0.008107643574476242, -0.03018002025783062, 0.004599703475832939, 0.0031579104252159595, 0.029602451249957085, 0.019478006288409233, -0.001244150917045772, 0.007932152599096298, -0.024374259635806084, -0.0014317389577627182, -0.005060661118477583, 0.01882106624543667, -0.030170971527695656, 0.004942932166159153, 0.01855279691517353, -0.016252465546131134, 0.003668307326734066, 0.023040689527988434, 0.03257002681493759, 0.009359115734696388, 0.041357554495334625, -0.009569245390594006, -0.0021451441571116447, -0.004025306086987257, 0.020729605108499527, -0.025507068261504173, -0.004195620305836201, -0.02514141984283924, -0.0049150725826621056, 0.021779395639896393, -0.005066953599452972, 0.0407865047454834, 0.03390626236796379, -0.003425570437684655, -0.046212587505578995, -0.010722663253545761, 0.004477834794670343, 0.027803486213088036, 0.014241323806345463, -0.017296617850661278, 0.008002053014934063, -0.03209151700139046, -0.022864930331707, -0.022020554170012474, 0.016860732808709145, -0.002820241264998913, 0.007012031972408295, 0.02112017758190632, -0.010848923586308956, -0.015040716156363487, -0.015330737456679344, 0.012386811897158623, 0.0007767569040879607, -0.0035596026573330164], [0.006522742565721273, 0.005635552108287811, -0.010397790931165218, 0.0011609334032982588, 0.008698110468685627, 0.00391092523932457, -0.017984574660658836, -0.022291915491223335, 0.004345630295574665, 0.002326937159523368, -0.001876550610177219, -0.020940536633133888, 0.03631488233804703, 0.014374593272805214, 0.005496750585734844, 0.012857632711529732, -0.01518411748111248, -0.015196654945611954, -0.02219688892364502, -0.010392886586487293, 0.0032093229237943888, -0.011539336293935776, -0.006616225931793451, -0.010747727937996387, 0.01160307414829731, 9.985747055907268e-06, -0.007164055481553078, -0.023265253752470016, -0.04198373854160309, -0.01494784839451313, 0.0045402091927826405, 0.011797292158007622, -0.018985984846949577, -0.0029657273553311825, 0.017509261146187782, 0.007494513876736164, 0.01046236976981163, -0.02785838022828102, 0.002436364535242319, -0.008244475349783897, 0.011958982795476913, 0.010314869694411755, -0.022458292543888092, -0.00545602198690176, -0.005227948073297739, -0.005634150933474302, -0.002415865659713745, -0.016274990513920784, 0.016031410545110703, -0.036225032061338425, -0.03690527752041817, 0.0349518358707428, -0.016711194068193436, -0.006124292965978384, 0.03774372860789299, -0.013679315336048603, -0.005734636913985014, -0.013940257020294666, 0.009817477315664291, -0.0028614180628210306, 0.03321801498532295, 0.010952496901154518, 0.005028082057833672, 0.003921498544514179], [0.012099052779376507, -0.014030346646904945, -0.03073795698583126, -0.0051240138709545135, 0.02011163718998432, 0.0012318852823227644, -0.015067815780639648, -0.0018328396836295724, -0.0040174066089093685, 0.02695220708847046, -0.003477448131889105, -0.008437104523181915, -0.05887175723910332, -0.018924525007605553, 0.0372755229473114, 0.015966227278113365, -0.013985254801809788, -0.014558968134224415, 0.01124448049813509, 0.022173242643475533, -0.0009426715550944209, 0.012399586848914623, -0.004589039366692305, 0.007907578721642494, -0.008702835068106651, 0.012943966314196587, -0.028124826028943062, -0.052437424659729004, 0.00971570797264576, -0.031992990523576736, 0.013863696716725826, -0.028421834111213684, 0.022668689489364624, -0.006187184248119593, -0.006041379179805517, 0.007392954546958208, 0.016518037766218185, -0.013840134255588055, -0.020429447293281555, -0.014559086412191391, 0.030911201611161232, -0.006219520233571529, -0.027115432545542717, -0.028808264061808586, -0.001971821067854762, 0.027022801339626312, 0.02510821633040905, -0.015090118162333965, -0.011986752972006798, 0.0050814650021493435, -0.029552094638347626, 0.0082948487251997, -0.026613013818860054, -0.0013940510107204318, -0.0062960293143987656, -0.019330192357301712, -0.05710482597351074, 0.010249008424580097, 0.01736437901854515, 0.011613793671131134, -0.014215889386832714, 0.008319072425365448, -0.019200239330530167, 0.003731817239895463], [0.007461929693818092, 0.012994598597288132, -0.03979102149605751, -0.01008972804993391, 0.018439168110489845, 0.003337795613333583, -0.011628895998001099, 0.01263262890279293, -0.03147811070084572, 0.0008425917476415634, -0.033566493541002274, -0.006579167675226927, -0.041718270629644394, -0.008235808461904526, -0.0007067417609505355, -0.0017466112039983273, -0.02753392793238163, -0.0025754570960998535, 0.023653386160731316, 0.022273262962698936, -0.018540605902671814, 0.009562183171510696, -0.012135738506913185, -0.022997163236141205, 0.0072958264499902725, -0.025021547451615334, 0.039253268390893936, 0.029425237327814102, -0.0127886226400733, -0.029810093343257904, 0.0046193827874958515, -0.018442610278725624, -0.01820606365799904, -0.02992970123887062, 0.007992060855031013, -0.027055153623223305, -9.164743823930621e-05, 0.023457592353224754, 0.022790007293224335, -0.013446062803268433, 0.013860167004168034, -0.007471076678484678, -2.145780308637768e-05, -0.01079771388322115, 0.0403168722987175, -0.027595316991209984, -0.0036807286087423563, 0.007101383525878191, 0.0022784057073295116, -0.003422040957957506, 0.039102427661418915, -0.01951667107641697, -0.024947544559836388, -0.0015740790404379368, 0.006471892818808556, -0.012530668638646603, -0.011892960406839848, 0.0019316120306029916, -0.00125645671505481, -0.012563018128275871, 0.0033806078135967255, 1.5029259884613566e-05, -0.002868192968890071, 0.012039448134601116], [0.009456590749323368, -0.009601127356290817, 0.003551623085513711, -0.01946910470724106, -0.015186244621872902, -0.02926095947623253, 0.0026709113735705614, -0.01345033012330532, 0.011992238461971283, -0.013921156525611877, -0.012348460964858532, 0.015324683859944344, 0.000800401670858264, -0.013040926307439804, 0.016186155378818512, 0.01627175509929657, -0.011249222792685032, 0.03559616580605507, 0.036502446979284286, -0.018150189891457558, 0.029066696763038635, -0.015685051679611206, -0.0047424351796507835, 0.004480051342397928, -0.02749442867934704, 0.022641517221927643, -0.0215650275349617, -0.021678153425455093, 0.00875125266611576, -0.008052012883126736, -0.008056859485805035, -0.015578433871269226, -0.006037769373506308, -0.02576066367328167, 0.004335442557930946, 0.03684164956212044, -0.014550788328051567, 0.008427799679338932, -0.02796459197998047, -0.029554586857557297, 0.008356694132089615, 0.0023995419032871723, -0.009388421662151814, 0.01924518682062626, 0.017530376091599464, 0.007256662007421255, -0.037484195083379745, -0.0008700992912054062, -0.022408397868275642, 0.0041361707262694836, 0.005721402354538441, 0.02055787481367588, 0.0007344927871599793, -0.002982140053063631, 0.04234888032078743, -0.003225217806175351, -0.01410755980759859, -0.010765403509140015, 0.03185048699378967, 0.00311663537286222, -0.008475609123706818, 0.04589289426803589, 0.024854697287082672, -0.009855295531451702], [-0.0016599191585555673, 0.020421849563717842, 0.012890498153865337, -0.010320045053958893, 0.02109050564467907, 0.03166618570685387, 0.0030097602866590023, -0.019974926486611366, -0.008438150398433208, 0.019955676048994064, 0.020839927718043327, 0.021179428324103355, -0.014415433630347252, -0.004358787089586258, -0.048957403749227524, 0.02058735117316246, 0.008421782404184341, -0.024122929200530052, -0.01209956780076027, -0.015914328396320343, 0.01766720972955227, -0.03104219399392605, -0.009256325662136078, -0.0015018589328974485, -0.010308360680937767, 0.015264147892594337, 0.03321881592273712, 0.0030513617675751448, -0.027926117181777954, -0.02994445711374283, -0.014784784987568855, -0.010932818055152893, -0.011368843726813793, -0.002554160077124834, -0.02867964841425419, -0.0013890115078538656, -0.006067256443202496, 0.013943661004304886, -0.03262593597173691, -0.009342405945062637, 0.028863146901130676, 0.02607937529683113, -0.004104423802345991, 0.009697658009827137, -0.00955553725361824, 0.011990928091108799, -0.0005597382551059127, -0.02696361392736435, 0.0020017484202980995, -0.0077288453467190266, 0.008192220702767372, -0.02257544733583927, 0.002373624127358198, 0.008359931409358978, 0.006190488580614328, -0.035299744457006454, -0.009978472255170345, -0.02244342863559723, 0.03964368999004364, 0.008661948144435883, -0.004138051066547632, -0.013784029521048069, 0.026357343420386314, -0.020197808742523193], [0.015690336003899574, -0.025412902235984802, -0.018337612971663475, 0.030186939984560013, 0.0017500780522823334, -0.0421900749206543, 0.01975274458527565, -0.021503303200006485, 0.020895065739750862, -0.05998372286558151, -0.04411107301712036, -0.028393087908625603, 0.001783967250958085, 0.0020162074360996485, -0.01083429530262947, -0.0058209956623613834, -0.011926507577300072, -0.005237225908786058, 0.009856663644313812, -0.0011434421176090837, -0.017064042389392853, -0.041176654398441315, -0.04055862873792648, -0.0036201849579811096, -0.003278505289927125, -0.014485975727438927, 0.032218050211668015, -0.003427795134484768, 0.004010103642940521, -0.013876068405807018, -0.0073688640259206295, 0.0003963076160289347, 0.0011211956152692437, 0.0007677226676605642, 0.0018722680397331715, 0.018028870224952698, -0.001453920267522335, 0.016463281586766243, -0.008384502492845058, -0.010683565400540829, 0.03656621277332306, 0.021192388609051704, 0.010250154882669449, 0.0012069933582097292, -0.029715677723288536, 0.01312563382089138, -0.046790678054094315, 0.0016523718368262053, -0.01901538111269474, -0.026003357023000717, -0.012163794599473476, -0.017151910811662674, 0.007793251425027847, -0.02105424366891384, 0.0048300414346158504, -0.020349228754639626, -0.016807544976472855, -0.015679890289902687, 0.011088727042078972, 0.0563155822455883, 0.017564548179507256, -0.00882063340395689, -0.016746940091252327, -0.017071308568120003], [0.019295433536171913, -0.0035716532729566097, 0.01446031965315342, -0.023106656968593597, 0.025382833555340767, -0.014107029885053635, -0.018527492880821228, 0.0038277090061455965, -0.02017243579030037, -0.017370324581861496, 0.028446823358535767, 0.01603630930185318, -0.03506696969270706, 0.02919798158109188, 0.01137404516339302, 0.01013923715800047, 0.03313424810767174, -0.04225875437259674, -0.0031684981659054756, -0.028105176985263824, 0.038563501089811325, 0.009767722338438034, -0.037074241787195206, 0.01583518087863922, -0.0170409195125103, -0.0011395144974812865, -0.024400321766734123, 0.036138713359832764, -0.005124337039887905, -0.019346337765455246, 0.011142352595925331, -0.0005795969627797604, -0.0020173753146082163, -0.0025804953183978796, -0.021899856626987457, -0.015450526960194111, 0.028592707589268684, -0.04342465475201607, -0.006752959918230772, 0.002724702237173915, 0.03354623168706894, 0.015406595543026924, 0.0008342895889654756, 0.031713929027318954, 0.02254461497068405, -0.02274613082408905, 0.015136639587581158, -0.00503807608038187, 0.008236379362642765, 0.016168510541319847, -0.010177909396588802, 0.0302924532443285, 0.02571430802345276, -0.019878070801496506, -0.024276869371533394, -0.032031141221523285, -0.026422303169965744, 0.01664656028151512, -0.04640788584947586, 0.0023836935870349407, 0.005538901314139366, 0.01147129200398922, -0.02990001067519188, 0.040035851299762726], [-0.007983939722180367, -0.015452520921826363, 0.016964955255389214, -0.011918764561414719, 0.024947842583060265, -0.005506576970219612, 0.006213604472577572, -0.012257336638867855, 0.016010789200663567, 0.0009256787598133087, 0.009568996727466583, -0.026050450280308723, 0.05180627480149269, -0.007457816507667303, -0.027584794908761978, 0.01926247403025627, 0.019108084961771965, -0.03539291024208069, -0.01382429152727127, -0.023496171459555626, 0.011493749916553497, -0.0272543802857399, -0.017563559114933014, -0.013230382464826107, -0.004537339322268963, 0.0034159012138843536, -0.006809499580413103, -0.009803022257983685, 0.013994179666042328, -0.015590445138514042, 0.011273481883108616, -0.004446793347597122, 0.0016699850093573332, -0.02048257365822792, -0.010334714315831661, -0.005934786982834339, 0.0024264459498226643, -0.049232423305511475, 0.03414632007479668, 0.02227131649851799, 0.003332095453515649, -0.0013629462337121367, 0.00524686137214303, -0.013587369583547115, -0.0031129075214266777, 0.03349492326378822, -0.03123808652162552, 0.0069352067075669765, -0.0006919584702700377, -0.020663579925894737, -0.0008012403268367052, 0.030216028913855553, -0.040255893021821976, -0.04793434962630272, -0.002001790329813957, 1.83402189577464e-05, -0.018508663401007652, -0.015036450698971748, 0.012721842154860497, -0.05326059088110924, -0.023794708773493767, 0.008595229126513004, -0.02022227831184864, -0.01273941807448864], [-0.012362040579319, -0.014755556359887123, -0.009113909676671028, -0.005278746597468853, 0.006876168306916952, 0.0295135285705328, 0.006948104128241539, -0.01221008412539959, -0.0014703894266858697, 0.023711835965514183, -0.009879120625555515, -0.004730891436338425, -0.022639010101556778, 0.021867865696549416, 0.011379518546164036, -0.014285754412412643, 0.010187980718910694, 0.022759372368454933, -0.0021532466635107994, 0.010156304575502872, -0.0093823978677392, -0.0055612362921237946, -0.009493828751146793, 0.02620854601264, 0.01331634633243084, 0.013424796983599663, -0.002587195485830307, 0.036244459450244904, 0.0070946114137768745, -0.018185721710324287, -0.009086505509912968, 0.006666648201644421, -0.019363002851605415, 0.03314165771007538, -0.005630915518850088, -0.018256191164255142, -0.001668452052399516, -0.007973574101924896, -0.0076300897635519505, 0.014063420705497265, -0.011970839463174343, 0.025840403512120247, -0.00018588580132927746, 0.02748972550034523, -0.023133615031838417, 0.0012939786538481712, -0.01717469096183777, -0.028729690238833427, -0.03521875664591789, -0.022813422605395317, 0.009230268187820911, 0.020449895411729813, 0.0062837922014296055, -0.013056440278887749, -0.003339212853461504, 0.004542033653706312, -0.012560617178678513, 0.015997009351849556, 0.01901232823729515, -0.006414009258151054, 0.0006478462019003928, -0.012972569093108177, 0.025349363684654236, -0.00337736657820642], [0.044658392667770386, -0.02455085702240467, 0.04199797660112381, 0.0014105174923315644, -0.024631064385175705, -0.027804668992757797, 0.018521476536989212, -0.0032621577847748995, 0.020329991355538368, -0.0238333772867918, 0.004272305406630039, -0.010277941823005676, 0.0005235943826846778, -0.0051273866556584835, 0.012141351588070393, 0.0061830636113882065, -0.01080762967467308, 0.03148822486400604, -0.0357816219329834, 0.007501682266592979, -0.007188933435827494, -0.029494138434529305, -0.0016296409303322434, -0.03443076089024544, -0.016939867287874222, -0.011856270022690296, 0.008564665913581848, 0.020364534109830856, 0.022220553830266, 0.022593338042497635, -0.033088475465774536, -0.01355152390897274, 0.02834867686033249, 0.02173253893852234, -0.016795573756098747, 0.004862974863499403, -0.03211952745914459, -0.005590735003352165, 0.026242999359965324, -0.00042584212496876717, -0.02167324721813202, 0.036383479833602905, 0.003618716262280941, 0.0036238697357475758, 0.01381429098546505, 0.013736758381128311, 0.02685663476586342, -0.008145642466843128, -0.010296332649886608, -0.012510854750871658, 0.04086637869477272, 0.03114122711122036, 0.026680082082748413, -0.024753574281930923, -0.015825655311346054, -0.0023241781163960695, -0.012998650781810284, -0.035626884549856186, 0.013125184923410416, -0.013245215639472008, -0.011385468766093254, 0.040420178323984146, -0.010453029535710812, -0.02207418903708458], [-0.03316044062376022, -0.0033063809387385845, -0.015977198258042336, 0.011128855869174004, -0.010186380706727505, -0.003746959613636136, 0.01947164535522461, 0.02614399418234825, 0.012601257301867008, 0.004864677786827087, -0.013958577066659927, 0.006978791207075119, 0.0008987131877802312, 0.0074012926779687405, -0.023211853578686714, -0.03271905705332756, 0.025731904432177544, 0.016620395705103874, 0.008766090497374535, -0.0012538933660835028, 0.029120001941919327, -0.03247157856822014, -0.014667035080492496, -0.031784702092409134, 0.004084679763764143, -0.023682551458477974, 0.033534374088048935, 0.018803315237164497, 0.006933913566172123, 0.021826181560754776, -0.002579420804977417, 0.03738432377576828, -0.02786296047270298, -0.015942223370075226, -0.0071823615580797195, 0.036018211394548416, 0.012230541557073593, 0.00885777734220028, 0.01848696731030941, -0.03832269087433815, 0.014450985938310623, 0.004076831042766571, 0.00044282234739512205, -0.021465854719281197, -0.012549258768558502, -0.0736224502325058, -0.004734516609460115, -0.024585669860243797, -0.004236055072396994, -0.023332243785262108, -0.010679947212338448, -0.007848703302443027, -0.018233925104141235, 0.012717105448246002, -0.037495147436857224, -0.0015303422696888447, 0.02048649825155735, 0.0014607156626880169, 0.0344482883810997, 0.024423258379101753, -0.006775621324777603, 0.011313931085169315, 0.007123762741684914, 0.00010893228318309411]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(l1_normalize_columns, json.loads(_TESTS), 'cuda', *[1e-05, 1e-07])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")